In [1]:
import os

MB_DIR = r"../data_raw/musicbrainz/mbdump/mbdump"
TABLES = ["artist", "artist_credit_name", "medium"]

def check_table(path: str, delimiter: str = "\t", n_lines: int = 50):
    name = os.path.basename(path)
    print(f"\n--- Checking {name} ---")
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline().rstrip("\n")
        cols_first = first.split(delimiter)
        expected = len(cols_first)
        print(f"✅ Read first line")
        print(f"📊 Column count (line 1): {expected}")

        mismatches = 0
        last_i = 1
        for i in range(2, n_lines + 1):
            line = f.readline()
            if not line:
                last_i = i - 1
                break
            cols = line.rstrip("\n").split(delimiter)
            if len(cols) != expected:
                mismatches += 1
                if mismatches <= 3:
                    print(f"⚠️ Mismatch at line {i}: got {len(cols)} expected {expected}")
            last_i = i

        if mismatches == 0:
            print(f"✅ Column count consistent for first {last_i} lines")
        else:
            print(f"⚠️ Total mismatches in first {min(n_lines, last_i)} lines: {mismatches}")

        if "\\N" in cols_first:
            print("⚠️ Found '\\N' in line 1 (NULL marker present)")
        else:
            print("ℹ️ No '\\N' in line 1 (still use nullstr='\\\\N' when loading)")

for t in TABLES:
    fp = os.path.join(MB_DIR, t)
    if not os.path.exists(fp):
        print(f"\n❌ File not found: {fp}")
        continue
    check_table(fp)


--- Checking artist ---
✅ Read first line
📊 Column count (line 1): 19
✅ Column count consistent for first 50 lines
⚠️ Found '\N' in line 1 (NULL marker present)

--- Checking artist_credit_name ---
✅ Read first line
📊 Column count (line 1): 5
✅ Column count consistent for first 50 lines
ℹ️ No '\N' in line 1 (still use nullstr='\\N' when loading)

--- Checking medium ---
✅ Read first line
📊 Column count (line 1): 9
✅ Column count consistent for first 50 lines
ℹ️ No '\N' in line 1 (still use nullstr='\\N' when loading)


In [11]:
import duckdb

# Connect to the existing DuckDB database file.
# This creates a "working copy" database in data_processed/ and keeps the raw MusicBrainz dump immutable.
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Importing MusicBrainz 'artist' (19 columns)...")

# Ingest the MusicBrainz "artist" dump file into DuckDB.
# Important details:
# - header=false because MusicBrainz TSV dumps have no header row.
# - delim='\t' because the dumps are tab-separated.
# - nullstr='\\N' because MusicBrainz uses the PostgreSQL NULL marker \N in its TSV exports.
# - columns={...} explicitly defines the schema (names + types) so the 19 fields are aligned correctly.
#   This prevents silent column shifts and enables correct type casting (INTEGER/BOOLEAN/TIMESTAMP).

con.execute("""
    CREATE OR REPLACE TABLE mb_artist AS
    SELECT * FROM read_csv(
        '../data_raw/musicbrainz/mbdump/mbdump/artist',
        header = false,
        delim = '\t',
        nullstr = '\\N',

        columns = {
            'id': 'INTEGER',
            'gid': 'VARCHAR',
            'name': 'VARCHAR',
            'sort_name': 'VARCHAR',
            'begin_date_year': 'INTEGER',
            'begin_date_month': 'INTEGER',
            'begin_date_day': 'INTEGER',
            'end_date_year': 'INTEGER',
            'end_date_month': 'INTEGER',
            'end_date_day': 'INTEGER',
            'type': 'INTEGER',
            'area': 'INTEGER',
            'gender': 'INTEGER',
            'comment': 'VARCHAR',
            'edits_pending': 'INTEGER',
            'last_updated': 'TIMESTAMP',
            'ended': 'BOOLEAN',
            'begin_area': 'INTEGER',
            'end_area': 'INTEGER'
        }
    );
""")

print("✅ mb_artist table created!")

# Quick proof
print("\n--- Preview ---")
print(con.execute("SELECT id, name, sort_name, type, area FROM mb_artist LIMIT 5").fetchdf())

print("\n--- Row count (optional but useful) ---")
print(con.execute("SELECT COUNT(*) AS n FROM mb_artist").fetchdf())
# Sanity check: preview a few rows to confirm parsing worked and key columns look reasonable.
# We also print a row count to confirm the table is populated.

Connected ✅
Importing MusicBrainz 'artist' (19 columns)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mb_artist table created!

--- Preview ---
        id           name      sort_name  type  area
0  2252039  Доктор Сатана  Доктор Сатана  <NA>  <NA>
1   371203    Pete Moutso   Moutso, Pete     1  <NA>
2  3087346           UNlT           UNlT  <NA>  <NA>
3  2851271      Babyglock      Babyglock  <NA>  <NA>
4   145773   Aric Leavitt  Leavitt, Aric  <NA>  <NA>

--- Row count (optional but useful) ---
         n
0  2807098


In [4]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Importing MusicBrainz 'artist_credit_name' (5 columns)...")

con.execute("""
    CREATE OR REPLACE TABLE mb_artist_credit_name AS
    SELECT * FROM read_csv(
        '../data_raw/musicbrainz/mbdump/mbdump/artist_credit_name',
        header = false,
        delim = '\t',
        nullstr = '\\N',
        columns = {
            'artist_credit': 'INTEGER',
            'position': 'INTEGER',
            'artist': 'INTEGER',
            'name': 'VARCHAR',
            'join_phrase': 'VARCHAR'
        }
    );
""")

print("✅ mb_artist_credit_name table created!")

# Proof / sanity check
print("\n--- Preview ---")
print(con.execute("""
    SELECT artist_credit, position, artist, name, join_phrase
    FROM mb_artist_credit_name
    ORDER BY artist_credit, position
    LIMIT 10
""").fetchdf())

print("\n--- Row count ---")
print(con.execute("SELECT COUNT(*) AS n FROM mb_artist_credit_name").fetchdf())

Connected ✅
Importing MusicBrainz 'artist_credit_name' (5 columns)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mb_artist_credit_name table created!

--- Preview ---
   artist_credit  position  artist                  name join_phrase
0              1         0       1       Various Artists            
1              4         0       4        Massive Attack            
2              6         0       6          Apartment 26            
3              7         0       7              Dr. Evil            
4              9         0       9          Robert Miles            
5             10         0      10         Vincent Gallo            
6             11         0      11  Squirrel Nut Zippers            
7             12         0      12            Giant Sand            
8             15         0      15            Éric Serra            
9             16         0      16  William S. Burroughs            

--- Row count ---
         n
0  6642750


In [10]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Importing MusicBrainz 'medium' (9 columns) with corrected schema...")

con.execute("""
    CREATE OR REPLACE TABLE mb_medium AS
    SELECT * FROM read_csv(
        '../data_raw/musicbrainz/mbdump/mbdump/medium',
        header = false,
        delim = '\t',
        nullstr = '\\N',
        columns = {
            'id': 'BIGINT',
            'release': 'BIGINT',
            'position': 'BIGINT',
            'format': 'BIGINT',
            'name': 'VARCHAR',
            'edits_pending': 'BIGINT',
            'last_updated': 'TIMESTAMPTZ',
            'track_count': 'BIGINT',
            'tracklist': 'VARCHAR'
        }
    );
""")

print("✅ mb_medium table created!")

# Proof / sanity check
print("\n--- Preview ---")
print(con.execute("""
    SELECT id, release, position, format, name, track_count, tracklist, last_updated
    FROM mb_medium
    LIMIT 10
""").fetchdf())

print("\n--- Row count ---")
print(con.execute("SELECT COUNT(*) AS n FROM mb_medium").fetchdf())

Connected ✅
Importing MusicBrainz 'medium' (9 columns) with corrected schema...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mb_medium table created!

--- Preview ---
        id  release  position  format name  track_count  \
0  3316155  3046812         1       8                 8   
1  5434234  4997020         1      12                 1   
2  5500452  5059003         1      12                 1   
3  5648581  5196850         1      12                 1   
4  5439958  5002457         1      29                 2   
5  5506285  5064328         1      12                 1   
6  5570073  5123945         1      12                 2   
7  5653374  5201344         1      12                 1   
8  5738436  5281321         1      12                 1   
9  5511562  5069293         1       1                 8   

                              tracklist                     last_updated  
0  4ecb0d23-669b-33cd-911d-dd12c42dfb67 2026-02-02 16:00:52.332103+01:00  
1  9560d0bb-b9e8-4050-8c1b-f4740304c056 2025-05-19 20:02:17.904703+02:00  
2  bb6dabf2-25aa-43b7-a8d1-226ba17d42ab 2025-06-24 02:57:19.742813+02:00  
3  445

In [1]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

path = "../data_raw/musicbrainz/mbdump/mbdump/artist_credit"

# 1) Column count (line 1)
with open(path, "r", encoding="utf-8") as f:
    first = f.readline().rstrip("\n")
col_count = len(first.split("\t"))
print(f"artist_credit column count (line 1): {col_count}")

print("Importing artist_credit safely as VARCHAR columns (so it never fails)...")

# 2) Build a dynamic columns mapping: c1..cN as VARCHAR
cols_map = ",\n            ".join([f"'c{i}': 'VARCHAR'" for i in range(1, col_count + 1)])

con.execute(f"""
    CREATE OR REPLACE TABLE mb_artist_credit_raw AS
    SELECT * FROM read_csv(
        '{path}',
        header = false,
        delim = '\\t',
        nullstr = '\\\\N',
        columns = {{
            {cols_map}
        }}
    );
""")

print("✅ mb_artist_credit_raw created!")

# 3) Create the core table we actually need (id + full credit name)
# Assumption (based on MB schema and your screenshot): column 1 = id, column 2 = name
con.execute("""
    CREATE OR REPLACE TABLE mb_artist_credit_core AS
    SELECT
        CAST(c1 AS BIGINT) AS id,
        c2 AS name
    FROM mb_artist_credit_raw;
""")

print("✅ mb_artist_credit_core (id, name) created!")

print("\n--- Preview core ---")
print(con.execute("SELECT * FROM mb_artist_credit_core LIMIT 10").fetchdf())

print("\n--- Row count core ---")
print(con.execute("SELECT COUNT(*) AS n FROM mb_artist_credit_core").fetchdf())

Connected ✅


OSError: [Errno 22] Invalid argument

In [12]:
import duckdb
con = duckdb.connect("../data_processed/thesis.duckdb")

# This lists all tables currently in your database
print("Tables currently in your database:")
print(con.execute("SHOW TABLES").fetchdf())

con.close()

Tables currently in your database:
                    name
0              mb_artist
1  mb_artist_credit_core
2  mb_artist_credit_name
3   mb_artist_credit_raw
4              mb_medium
5             mb_release


In [11]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Building mb_release_features (title + full artist credit + main artist + total tracks)...")

con.execute("""
    CREATE OR REPLACE TABLE mb_release_features AS
    WITH main_artist AS (
        -- Main artist = first entry in the artist credit list (position = 0)
        SELECT
            acn.artist_credit,
            a.name AS main_artist_name
        FROM mb_artist_credit_name acn
        JOIN mb_artist a
            ON a.id = acn.artist
        WHERE acn.position = 0
    ),
    tracks AS (
        -- Total tracks per release = sum of track_count across all media (CD1+CD2+...)
        SELECT
            release AS release_id,
            SUM(track_count) AS total_tracks
        FROM mb_medium
        GROUP BY release
    )
    SELECT
        r.id AS release_id,
        r.name AS release_title,
        r.barcode,
        r.artist_credit AS artist_credit_id,
        ac.name AS artist_credit_full,
        ma.main_artist_name,
        EXTRACT(YEAR FROM r.last_updated) AS mb_year,
        t.total_tracks
    FROM mb_release r
    LEFT JOIN mb_artist_credit_core ac
        ON ac.id = r.artist_credit
    LEFT JOIN main_artist ma
        ON ma.artist_credit = r.artist_credit
    LEFT JOIN tracks t
        ON t.release_id = r.id;
""")

print("✅ mb_release_features created!")

print("\n--- Preview ---")
print(con.execute("""
    SELECT release_id, release_title, artist_credit_full, main_artist_name, total_tracks, barcode, mb_year
    FROM mb_release_features
    WHERE main_artist_name IS NOT NULL
    LIMIT 10
""").fetchdf())

print("\n--- Row count ---")
print(con.execute("SELECT COUNT(*) AS n FROM mb_release_features").fetchdf())

print("\n--- Missing values check ---")
print(con.execute("""
    SELECT
      SUM(CASE WHEN artist_credit_full IS NULL THEN 1 ELSE 0 END) AS missing_artist_credit_full,
      SUM(CASE WHEN main_artist_name IS NULL THEN 1 ELSE 0 END) AS missing_main_artist,
      SUM(CASE WHEN mb_year IS NULL THEN 1 ELSE 0 END) AS missing_mb_year,
      SUM(CASE WHEN total_tracks IS NULL THEN 1 ELSE 0 END) AS missing_total_tracks
    FROM mb_release_features
""").fetchdf())

Connected ✅
Building mb_release_features (title + full artist credit + main artist + total tracks)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mb_release_features created!

--- Preview ---
   release_id                         release_title           artist_credit_full main_artist_name  total_tracks        barcode  mb_year
0      178087  Blechdance: Ritter Rost zum Abtanzen  Jörg Hilbert & Felix Janosa     Jörg Hilbert          11.0  9783898351300     2010
1      178088                                 Verte                        Judy4            Judy4          11.0           None     <NA>
2      178097                             Egouality                          WaK              WaK          14.0           None     <NA>
3      178098                             Egouality                          WaK              WaK          14.0           None     <NA>
4      178100                             Démo 2005                          WaK              WaK           4.0           None     <NA>
5      178101                             Démo 2005                          WaK              WaK           4.0           None     <NA>


In [15]:
#deletes the old CSVs, then writes the new ones.
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Output folder
out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

tables = [
    "mb_release_features",   
]

for table in tables:
    out_path = os.path.join(out_dir, f"{table}.csv")

    # Delete old CSV if it already exists
    if os.path.exists(out_path):
        os.remove(out_path)
        print(f"🗑️ Deleted old file: {out_path}")

    # Export fresh CSV
    con.execute(f"""
        COPY (
            SELECT *
            FROM {table}
        ) TO '{out_path}'
        (HEADER, DELIMITER ',');
    """)
    print(f"✅ Exported new file: {out_path}")

con.close()
print("Done ✅")

Connected ✅
🗑️ Deleted old file: ../data_processed/exports\mb_release_features.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported new file: ../data_processed/exports\mb_release_features.csv
Done ✅


In [144]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Loading CDS (cds_clean.tsv) into DuckDB (tracks kept as TEXT)...")

# ----------------------------
# Step A: Load CDS clean TSV safely
# ----------------------------
# Comments:
# - The CDS column called "tracks" is actually a tracklist (TEXT), not a numeric track count.
# - Therefore we do NOT cast it to BIGINT.
# - We cast only cds_id and cds_year (safe numeric fields).
# - We keep the tracklist in a column called cds_tracklist for clarity.
# - We use read_csv with explicit column names/types because read_csv_auto
#   failed to detect the TSV dialect reliably.
# - strict_mode=false / ignore_errors=true / null_padding=true make loading
#   more robust for messy TSV rows.
con.execute(r"""
    CREATE OR REPLACE TABLE cds AS
    SELECT
        artist,
        category,
        genre,
        TRY_CAST(id AS BIGINT) AS cds_id,
        title,
        tracks,
        TRY_CAST(year AS INTEGER) AS cds_year
    FROM read_csv(
        '../data_raw/hpi_cd/cds_clean.tsv',
        delim='\t',
        header=true,
        columns={
            'artist': 'VARCHAR',
            'category': 'VARCHAR',
            'genre': 'VARCHAR',
            'id': 'VARCHAR',
            'title': 'VARCHAR',
            'tracks': 'VARCHAR',
            'year': 'VARCHAR'
        },
        quote='"',
        escape='"',
        null_padding=true,
        ignore_errors=true,
        strict_mode=false
    );
""")

print("✅ cds table created!")

print("\n--- CDS preview ---")
print(con.execute("""
    SELECT cds_id, artist, title, cds_year, substr(tracks, 1, 80) AS tracks
    FROM cds
    LIMIT 10
""").fetchdf())

print("\n--- CDS row count ---")
print(con.execute("SELECT COUNT(*) AS n FROM cds").fetchdf())

# ----------------------------
# Step B: Normalized view for matching
# ----------------------------
# Comments:
# - We build normalized keys for artist and title to reduce differences from punctuation/case.
# - This is a high-precision baseline before fuzzy matching.
con.execute(r"""
    CREATE OR REPLACE VIEW cds_norm AS
    SELECT
        *,
        lower(trim(coalesce(artist, ''))) AS artist_clean,
        lower(trim(coalesce(title,  ''))) AS title_clean,

        lower(regexp_replace(coalesce(artist, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key,
        lower(regexp_replace(coalesce(title,  ''), '[^a-zA-Z0-9]+', '', 'g')) AS title_key
    FROM cds;
""")
 

print("\n✅ cds_norm view created!")

print("\n--- Normalization preview ---")
print(con.execute("""
    SELECT artist, title, artist_key, title_key
    FROM cds_norm
    LIMIT 10
""").fetchdf())

Connected ✅
Loading CDS (cds_clean.tsv) into DuckDB (tracks kept as TEXT)...
✅ cds table created!

--- CDS preview ---
   cds_id                       artist  \
0   10000              backstreet boys   
1  100001                      various   
2  100002                    no return   
3  100003      ã â â ã â ã â ºâ ã â å    
4  100004                      emanuel   
5  100005  ã å â ã â ºâ ã â ºâ ºã â â    
6  100006             luv lite massive   
7  100007              various artists   
8  100008                         dmbq   
9  100009                       4 p m    

                                                                        title  \
0                                                                  millennium   
1                                                 frankfurt trance vol 04 cd1   
2                                                             self mutilation   
3                                       ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ    
4            

In [ ]:
#I used a deterministic exact matching method based on normalized artist and title fields. Both datasets were preprocessed by lowercasing
#and removing non-alphanumeric characters,
#and records were linked only when the normalized artist and normalized title matched exactly

In [205]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# STEP 0: Create normalized MB view for matching
# ------------------------------------------------------------
print("Creating mb_norm view...")

con.execute("""
    CREATE OR REPLACE VIEW mb_norm AS
    SELECT
        release_id,
        release_title,
        artist_credit_full,
        main_artist_name,
        total_tracks,
        barcode,
        mb_year,

        lower(trim(coalesce(main_artist_name, ''))) AS artist_clean,
        lower(trim(coalesce(release_title, '')))    AS title_clean,

        lower(regexp_replace(coalesce(main_artist_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key,
        lower(regexp_replace(coalesce(release_title, ''),   '[^a-zA-Z0-9]+', '', 'g')) AS title_key
    FROM mb_release_features;
""")

print("✅ mb_norm created!")

# ------------------------------------------------------------
# STEP 1: Create exact candidate matches between CDS and MB
# ------------------------------------------------------------
print("Creating cds_to_mb_exact_matches...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_exact_matches AS
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.category,
        c.genre,
        c.tracks,
        c.artist_clean AS cds_artist_clean,
        c.title_clean  AS cds_title_clean,
        c.artist_key   AS cds_artist_key,
        c.title_key    AS cds_title_key,

        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        m.artist_clean AS mb_artist_clean,
        m.title_clean  AS mb_title_clean,
        m.artist_key   AS mb_artist_key,
        m.title_key    AS mb_title_key
    FROM cds_norm c
    JOIN mb_norm m
      ON c.artist_key = m.artist_key
     AND c.title_key  = m.title_key
    WHERE
        length(c.artist_key) >= 4
        AND length(c.title_key) >= 4
        AND length(m.artist_key) >= 4
        AND length(m.title_key) >= 4
        AND c.artist IS NOT NULL
        AND c.title IS NOT NULL
        AND c.artist NOT LIKE '%Ã%'
        AND c.artist NOT LIKE '%â%'
        AND c.title  NOT LIKE '%Ã%'
        AND c.title  NOT LIKE '%â%';
""")

print("✅ cds_to_mb_exact_matches created!")

print("\n--- Exact candidate stats ---")
print(con.execute("""
    WITH per_cds AS (
        SELECT cds_id, COUNT(*) AS candidate_count
        FROM cds_to_mb_exact_matches
        GROUP BY cds_id
    )
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(DISTINCT cds_id) FROM cds_to_mb_exact_matches) AS cds_records_with_candidates,
        (SELECT COUNT(*) FROM cds_to_mb_exact_matches) AS total_candidate_pairs,
        (SELECT COUNT(*) FROM per_cds WHERE candidate_count = 1) AS cds_records_matched_uniquely,
        (SELECT COUNT(*) FROM per_cds WHERE candidate_count > 1) AS cds_records_ambiguous
""").fetchdf())

print("\n--- Sample exact candidates ---")
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year, category, genre,
        release_id, main_artist_name, release_title, mb_year
    FROM cds_to_mb_exact_matches
    LIMIT 20
""").fetchdf())

Connected ✅
Creating mb_norm view...
✅ mb_norm created!
Creating cds_to_mb_exact_matches...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ cds_to_mb_exact_matches created!

--- Exact candidate stats ---
   cds_total  cds_records_with_candidates  total_candidate_pairs  cds_records_matched_uniquely  cds_records_ambiguous
0       9763                         3482                  17203                          1584                   1898

--- Sample exact candidates ---
    cds_id          artist                       title  cds_year   category        genre  release_id main_artist_name               release_title  mb_year
0   108204    guns n roses    appetite for destruction      1987      blues    hard rock      743465    Guns N’ Roses    Appetite for Destruction     2024
1     7599       kraftwerk                     the mix      1991       rock   electronic     4212910        Kraftwerk                     The Mix     2024
2    10293       kraftwerk                     the mix      1991     newage     synthpop     4212910        Kraftwerk                     The Mix     2024
3   107724    don williams              borro

In [ ]:
# previous result is good because now we know: exact matching works, but we need a tie-breaker to
#choose one “best” MB release per CDS row

In [200]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Choose ONE "best" exact match per CDS record
# ------------------------------------------------------------
# Comments:
# - Exact matches often return multiple MB releases (different editions).
# - We rank candidates per cds_id using a simple, explainable tie-breaker:
#   1) barcode present first (more specific release)
#   2) higher total_tracks next (often more complete release)
#   3) lowest release_id last (stable deterministic tie-break)
print("Creating cds_to_mb_best_exact (one best match per CDS)...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_exact AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id
                ORDER BY
                     CASE
                        WHEN barcode IS NOT NULL AND trim(barcode) <> '' THEN 1
                        ELSE 0
                    END DESC,
                    total_tracks DESC NULLS LAST,
                    release_id ASC
            ) AS rn
        FROM cds_to_mb_exact_matches
    )
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode
    FROM ranked
    WHERE rn = 1;
""")

print("✅ cds_to_mb_best_exact created!")

# ------------------------------------------------------------
# Step 2: Report coverage after tie-breaking
# ------------------------------------------------------------
# Comments:
# - best_exact_rows: how many CDS items now have a single chosen MB release.
# - unmatched_rows: CDS items with no exact match (we'll handle these with fuzzy matching later).
print("\n--- Coverage after picking best match ---")
print(con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(*) FROM cds_to_mb_best_exact) AS best_exact_rows,
        (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_best_exact) AS unmatched_rows
""").fetchdf())

# ------------------------------------------------------------
# Step 3: How much ambiguity did we resolve?
# ------------------------------------------------------------
# Comments:
# - Shows how many CDS rows had multiple candidates originally.
# - After ranking, each cds_id becomes 1 row in cds_to_mb_best_exact.
print("\n--- Ambiguity diagnostic (top 15 most ambiguous) ---")
print(con.execute("""
    SELECT cds_id, artist, title, COUNT(*) AS candidates
    FROM cds_to_mb_exact_matches
    GROUP BY cds_id, artist, title
    HAVING COUNT(*) > 1
    ORDER BY candidates DESC
    LIMIT 20
""").fetchdf())

# ------------------------------------------------------------
# Step 4: Preview chosen best matches
# ------------------------------------------------------------
print("\n--- Preview best matches ---")
print(con.execute("""
    SELECT cds_id, artist, title, release_id, main_artist_name, release_title, total_tracks, barcode
    FROM cds_to_mb_best_exact
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_to_mb_best_exact (one best match per CDS)...
✅ cds_to_mb_best_exact created!

--- Coverage after picking best match ---
   cds_total  best_exact_rows  unmatched_rows
0       9763             3482            6281

--- Ambiguity diagnostic (top 15 most ambiguous) ---
    cds_id             artist                                 title  candidates
0   105307  bruce springsteen                    born in the u s a          263
1     4266          kate bush                       the kick inside         120
2     3678          kate bush                       the kick inside         120
3     6690          kate bush                       the kick inside         120
4   108908      georges bizet                                carmen          98
5     4905     britney spears                               britney          88
6     4291     britney spears                               britney          88
7     3670            nirvana                             nevermind  

In [147]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")

print(con.execute("""
    SELECT COUNT(*) AS n
    FROM mb_release_features
    WHERE release_title = 'The Kick Inside'
      AND main_artist_name = 'Kate Bush'
""").fetchdf())

     n
0  120


In [152]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")

print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_norm
    WHERE lower(title) = 'the kick inside'
      AND lower(artist) = 'kate bush'
""").fetchdf())

   n
0  3


In [168]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")

print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_exact_matches
    WHERE lower(title) = 'the kick inside'
      AND lower(artist) = 'kate bush'
""").fetchdf())

     n
0  360


In [ ]:
#delete this cell

In [169]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# STEP 1: Build per-CDS match counts from exact candidate matches
# ------------------------------------------------------------
# This tells us whether each CDS record has:
# - 0 exact matches (not present in this table)
# - 1 exact match  -> unique
# - >1 exact match -> ambiguous
print("Building match diagnostics...")

con.execute("""
    CREATE OR REPLACE TABLE cds_match_counts AS
    SELECT
        cds_id,
        MIN(artist) AS artist,
        MIN(title) AS title,
        COUNT(*) AS candidate_count
    FROM cds_to_mb_exact_matches
    GROUP BY cds_id;
""")

print("✅ cds_match_counts created!")

# ------------------------------------------------------------
# STEP 2: Keep only uniquely matched CDS records
# ------------------------------------------------------------
# These are the safest automatic enrichment links:
# one CDS record -> exactly one MB release
print("Creating cds_to_mb_unique_exact...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_unique_exact AS
    SELECT m.*
    FROM cds_to_mb_exact_matches m
    JOIN cds_match_counts c
      ON m.cds_id = c.cds_id
    WHERE c.candidate_count = 1;
""")

print("✅ cds_to_mb_unique_exact created!")

# ------------------------------------------------------------
# STEP 3: Build a best-match table for ambiguous cases
# ------------------------------------------------------------
# This keeps one best MB release per CDS record using explainable rules:
# 1) barcode present first
# 2) higher total_tracks next
# 3) lowest release_id last
print("Creating cds_to_mb_best_exact...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_exact AS
    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cds_id
                ORDER BY
                    CASE
                        WHEN barcode IS NOT NULL AND trim(barcode) <> '' THEN 1
                        ELSE 0
                    END DESC,
                    total_tracks DESC NULLS LAST,
                    release_id ASC
            ) AS rn
        FROM cds_to_mb_exact_matches
    )
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        category,
        genre,
        tracks,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year
    FROM ranked
    WHERE rn = 1;
""")

print("✅ cds_to_mb_best_exact created!")

# ------------------------------------------------------------
# STEP 4: Create enriched MB table using UNIQUE exact matches only
# ------------------------------------------------------------
# This is the conservative enrichment table for the thesis:
# only high-confidence links are used automatically.
print("Creating mb_enriched_from_cds_unique...")

con.execute("""
    CREATE OR REPLACE TABLE mb_enriched_from_cds_unique AS
    SELECT
        u.release_id,
        u.release_title,
        u.main_artist_name,
        u.artist_credit_full,
        u.mb_year,
        u.total_tracks,
        u.barcode,

        u.cds_id,
        u.cds_year,
        u.category AS cds_category,
        u.genre AS cds_genre,
        u.tracks AS cds_tracklist,

        'CDS' AS enrichment_source,
        'exact_unique' AS match_type
    FROM cds_to_mb_unique_exact u;
""")

print("✅ mb_enriched_from_cds_unique created!")

# ------------------------------------------------------------
# STEP 5: Main thesis metrics
# ------------------------------------------------------------
print("\n--- Main integration / enrichment metrics ---")
print(con.execute("""
    WITH
    cds_total AS (
        SELECT COUNT(*) AS n FROM cds
    ),
    mb_total AS (
        SELECT COUNT(*) AS n FROM mb_release_features
    ),
    matched_cds AS (
        SELECT COUNT(DISTINCT cds_id) AS n
        FROM cds_to_mb_exact_matches
    ),
    unique_cds AS (
        SELECT COUNT(DISTINCT cds_id) AS n
        FROM cds_match_counts
        WHERE candidate_count = 1
    ),
    ambiguous_cds AS (
        SELECT COUNT(DISTINCT cds_id) AS n
        FROM cds_match_counts
        WHERE candidate_count > 1
    ),
    enriched_mb AS (
        SELECT COUNT(DISTINCT release_id) AS n
        FROM mb_enriched_from_cds_unique
    ),
    enriched_with_new_metadata AS (
        SELECT COUNT(DISTINCT release_id) AS n
        FROM mb_enriched_from_cds_unique
        WHERE cds_category IS NOT NULL
           OR cds_genre IS NOT NULL
           OR cds_year IS NOT NULL
    )
    SELECT
        (SELECT n FROM cds_total) AS cds_total,
        (SELECT n FROM matched_cds) AS cds_records_with_candidates,
        (SELECT n FROM unique_cds) AS cds_records_matched_uniquely,
        (SELECT n FROM ambiguous_cds) AS cds_records_ambiguous,
        (SELECT n FROM cds_total) - (SELECT n FROM matched_cds) AS cds_records_unmatched,

        (SELECT n FROM enriched_mb) AS mb_releases_enriched,
        (SELECT n FROM enriched_with_new_metadata) AS mb_releases_with_new_metadata,

        ROUND(
            100.0 * (SELECT n FROM enriched_with_new_metadata) / (SELECT n FROM mb_total),
            4
        ) AS pct_of_mb_with_new_metadata
""").fetchdf())

# ------------------------------------------------------------
# STEP 6: Ambiguity diagnostics
# ------------------------------------------------------------
print("\n--- Top 20 most ambiguous CDS records ---")
print(con.execute("""
    SELECT cds_id, artist, title, candidate_count
    FROM cds_match_counts
    WHERE candidate_count > 1
    ORDER BY candidate_count DESC, cds_id
    LIMIT 20
""").fetchdf())

# ------------------------------------------------------------
# STEP 7: Preview high-confidence enrichment rows
# ------------------------------------------------------------
print("\n--- Preview enriched MB releases (unique exact only) ---")
print(con.execute("""
    SELECT
        release_id, release_title, main_artist_name, mb_year,
        cds_id, cds_year, cds_category, cds_genre, enrichment_source, match_type
    FROM mb_enriched_from_cds_unique
    LIMIT 20
""").fetchdf())

# ------------------------------------------------------------
# STEP 8: Prepare evaluation sample for match accuracy
# ------------------------------------------------------------
# IMPORTANT:
# True accuracy cannot be computed automatically from these tables alone.
# We therefore create a sample for manual verification.
print("\nCreating evaluation sample for manual accuracy checking...")

con.execute("""
    CREATE OR REPLACE TABLE match_eval_sample AS
    (
        SELECT
            'unique_exact' AS sample_type,
            cds_id, artist, title, cds_year,
            release_id, main_artist_name, release_title, mb_year,
            barcode, total_tracks
        FROM cds_to_mb_unique_exact
        USING SAMPLE 50 ROWS
    )
    UNION ALL
    (
        SELECT
            'best_exact_ambiguous' AS sample_type,
            b.cds_id, b.artist, b.title, b.cds_year,
            b.release_id, b.main_artist_name, b.release_title, b.mb_year,
            b.barcode, b.total_tracks
        FROM cds_to_mb_best_exact b
        JOIN cds_match_counts c
          ON b.cds_id = c.cds_id
        WHERE c.candidate_count > 1
        USING SAMPLE 50 ROWS
    );
""")

print("✅ match_eval_sample created!")

print("\n--- Evaluation sample preview ---")
print(con.execute("""
    SELECT *
    FROM match_eval_sample
    LIMIT 20
""").fetchdf())

print("\nNOTE:")
print("- 'How accurate are the matches?' requires manual labeling of match_eval_sample.")
print("- Add a column like manual_label with values: correct / incorrect / uncertain.")
print("- Then compute precision from the labeled sample.")

Connected ✅
Building match diagnostics...
✅ cds_match_counts created!
Creating cds_to_mb_unique_exact...
✅ cds_to_mb_unique_exact created!
Creating cds_to_mb_best_exact...
✅ cds_to_mb_best_exact created!
Creating mb_enriched_from_cds_unique...
✅ mb_enriched_from_cds_unique created!

--- Main integration / enrichment metrics ---
   cds_total  cds_records_with_candidates  cds_records_matched_uniquely  \
0       9763                         3482                          1584   

   cds_records_ambiguous  cds_records_unmatched  mb_releases_enriched  \
0                   1898                   6281                  1572   

   mb_releases_with_new_metadata  pct_of_mb_with_new_metadata  
0                           1572                       0.0296  

--- Top 20 most ambiguous CDS records ---
    cds_id             artist                                 title  \
0   105307  bruce springsteen                    born in the u s a    
1     3678          kate bush                       the kic

In [ ]:
##############################
#remove cds_features
#remove cds_trackcount_est
#remove best_exact_v2
#remove best_exact_v3

In [201]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Create a CDS helper view with an estimated track count
# ------------------------------------------------------------
# Comments:
# - CDS "tracks" is actually a tracklist text field.
# - We create a heuristic estimate of track count by counting common separators.
# - This is not perfect, but it helps avoid selecting huge box sets in MB.
print("Creating cds_features with heuristic track count...")

con.execute("""
    CREATE OR REPLACE VIEW cds_features AS
    SELECT
        *,
        -- heuristic separator counts (add 1 because N separators -> N+1 items)
        greatest(
            case when tracks is null or length(trim(tracks)) = 0 then NULL else (length(tracks) - length(replace(tracks, ' / ', ''))) / 3 + 1 end,
            case when tracks is null or length(trim(tracks)) = 0 then NULL else (length(tracks) - length(replace(tracks, ' - ', ''))) / 3 + 1 end,
            case when tracks is null or length(trim(tracks)) = 0 then NULL else (length(tracks) - length(replace(tracks, ';', ''))) + 1 end,
            case when tracks is null or length(trim(tracks)) = 0 then NULL else (length(tracks) - length(replace(tracks, '|', ''))) + 1 end,
            case when tracks is null or length(trim(tracks)) = 0 then NULL else (length(tracks) - length(replace(tracks, ',', ''))) + 1 end
        )::BIGINT AS cds_trackcount_est
    FROM cds;
""")

print("✅ cds_features view created!")

print("\n--- Heuristic trackcount preview ---")
print(con.execute("""
    SELECT cds_id, artist, title, cds_year, cds_trackcount_est, substr(tracks, 1, 60) AS tracklist_preview
    FROM cds_features
    LIMIT 15
""").fetchdf())

# ------------------------------------------------------------
# Step 2: Re-rank candidates using closeness to estimated track count
# ------------------------------------------------------------
# Comments:
# - We keep barcode preference.
# - Then we prefer candidates whose total_tracks is close to cds_trackcount_est.
# - This helps select "normal" album releases over large compilations/box sets.
print("\nRebuilding cds_to_mb_best_exact using trackcount heuristic...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_exact_v2 AS
    WITH joined AS (
        SELECT
            e.*,
            f.cds_trackcount_est
        FROM cds_to_mb_exact_matches e
        LEFT JOIN cds_features f USING (cds_id)
    ),
    ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id
                ORDER BY
                     CASE
                        WHEN barcode IS NOT NULL AND trim(barcode) <> '' THEN 1
                        ELSE 0
                    END DESC,
                    -- prefer closer track counts (if we have an estimate)
                    CASE
                        WHEN cds_trackcount_est IS NULL OR total_tracks IS NULL THEN 999999
                        ELSE abs(total_tracks - cds_trackcount_est)
                    END ASC,
                    total_tracks ASC NULLS LAST,
                    release_id ASC
            ) AS rn
        FROM joined
    )
    SELECT
        cds_id, artist, title, cds_year, cds_trackcount_est,
        release_id, release_title, main_artist_name, artist_credit_full,
        total_tracks, barcode
    FROM ranked
    WHERE rn = 1;
""")

print("✅ cds_to_mb_best_exact_v2 created!")

print("\n--- Compare old vs new for a known ambiguous title (example: Let It Be) ---")
print(con.execute("""
    SELECT
      old.cds_id,
      old.artist,
      old.title,
      old.release_id AS old_release_id,
      old.release_title AS old_release_title,
      old.total_tracks AS old_total_tracks,
      v2.release_id  AS new_release_id,
      v2.release_title AS new_release_title,
      v2.total_tracks AS new_total_tracks,
      v2.cds_trackcount_est
    FROM cds_to_mb_best_exact old
    JOIN cds_to_mb_best_exact_v2 v2 USING (cds_id)
    WHERE lower(old.title) LIKE '%let it be%'
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_features with heuristic track count...
✅ cds_features view created!

--- Heuristic trackcount preview ---
    cds_id                       artist                                                                       title  cds_year  cds_trackcount_est  \
0    10000              backstreet boys                                                                  millennium      <NA>                   1   
1   100001                      various                                                 frankfurt trance vol 04 cd1      <NA>                   1   
2   100002                    no return                                                             self mutilation      <NA>                   1   
3   100003      ã â â ã â ã â ºâ ã â å                                        ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ       1989                   1   
4   100004                      emanuel                                                                  felicidade      199

In [22]:
import duckdb

# Always connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Goal: Pick one "best" MB release per CDS record from the EXACT candidates
# ------------------------------------------------------------
# Comments:
# - Exact matches often return many MB candidates (editions, reissues, etc.).
# - Our previous attempt used a CDS trackcount heuristic, but it produced mostly "1",
#   which biased selection toward singles (2 tracks) and other tiny releases.
# - Here we use a safer, professional rule:
#   1) Prefer barcode (more specific release)
#   2) Prefer "album-like" track counts (6..40) by giving them a bonus
#   3) Prefer medium track counts (closer to ~12) as a weak tie-break
#   4) Deterministic fallback: smallest release_id
print("Creating cds_to_mb_best_exact_v3 (barcode + album-like track count preference)...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_exact_v3 AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id
                ORDER BY
                    -- 1) barcode present first
                    CASE
                        WHEN barcode IS NOT NULL AND trim(barcode) <> '' THEN 1
                        ELSE 0
                    END DESC,
                    -- 2) album-like total_tracks gets priority (EP/Album range)
                    CASE
                        WHEN total_tracks BETWEEN 6 AND 40 THEN 0
                        WHEN total_tracks IS NULL THEN 2
                        ELSE 1
                    END ASC,

                    -- 3) weak preference toward ~12 tracks (common album length)
                    CASE
                        WHEN total_tracks IS NULL THEN 999999
                        ELSE abs(total_tracks - 12)
                    END ASC,

                    -- 4) deterministic final tie-break
                    release_id ASC
            ) AS rn
        FROM cds_to_mb_exact_matches
    )
    SELECT
        cds_id, artist, title, cds_year,
        release_id, release_title, main_artist_name, artist_credit_full,
        total_tracks, barcode, mb_year
    FROM ranked
    WHERE rn = 1;
""")

print("✅ cds_to_mb_best_exact_v3 created!")

# ------------------------------------------------------------
# Quick quality check on the specific example: Let It Be
# ------------------------------------------------------------
# Comments:
# - We compare v1/v2/v3 for "let it be" to ensure v3 avoids extremes.
print("\n--- Compare v1 vs v2 vs v3 for 'let it be' ---")
print(con.execute("""
    SELECT
      old.cds_id,
      old.artist,
      old.title,
      old.release_id AS v1_release_id,
      old.total_tracks AS v1_tracks,
      v2.release_id  AS v2_release_id,
      v2.total_tracks AS v2_tracks,
      v3.release_id  AS v3_release_id,
      v3.total_tracks AS v3_tracks
    FROM cds_to_mb_best_exact old
    JOIN cds_to_mb_best_exact_v2 v2 USING (cds_id)
    JOIN cds_to_mb_best_exact_v3 v3 USING (cds_id)
    WHERE lower(old.title) LIKE '%let it be%'
    LIMIT 20
""").fetchdf())

# ------------------------------------------------------------
# Preview best matches
# ------------------------------------------------------------
print("\n--- Preview best matches (v3) ---")
print(con.execute("""
    SELECT cds_id, artist, title, release_id, release_title, total_tracks, barcode, mb_year
    FROM cds_to_mb_best_exact_v3
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_to_mb_best_exact_v3 (barcode + album-like track count preference)...
✅ cds_to_mb_best_exact_v3 created!

--- Compare v1 vs v2 vs v3 for 'let it be' ---
   cds_id       artist      title  v1_release_id  v1_tracks  v2_release_id  v2_tracks  v3_release_id  v3_tracks
0    6272  the beatles  let it be        3213597       93.0        4273987        2.0         275837       12.0
1    7460  the beatles  let it be        3213597       93.0        4273987        2.0         275837       12.0

--- Preview best matches (v3) ---
    cds_id         artist                      title  release_id              release_title  total_tracks        barcode  mb_year
0     3475     rob zombie           hellbilly deluxe      455415           Hellbilly Deluxe          13.0   720642521212     2019
1     4441        nirvana                  nevermind        7518                  Nevermind          12.0  0720642442524     2022
2     4586       ministry           power department       162

In [206]:
#deletes the old CSVs, then writes the new ones.
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Output folder
out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

tables = [

    "cds_to_mb_exact_matches"
]

for table in tables:
    out_path = os.path.join(out_dir, f"{table}.csv")

    # Delete old CSV if it already exists
    if os.path.exists(out_path):
        os.remove(out_path)
        print(f"🗑️ Deleted old file: {out_path}")

    # Export fresh CSV
    con.execute(f"""
        COPY (
            SELECT *
            FROM {table}
        ) TO '{out_path}'
        (HEADER, DELIMITER ',');
    """)
    print(f"✅ Exported new file: {out_path}")

con.close()
print("Done ✅")

Connected ✅
🗑️ Deleted old file: ../data_processed/exports\cds_to_mb_exact_matches.csv
✅ Exported new file: ../data_processed/exports\cds_to_mb_exact_matches.csv
Done ✅


In [211]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating cds_to_mb_exact_scored without year scoring...")

con.execute(r"""
CREATE OR REPLACE TABLE cds_to_mb_exact_scored AS
WITH params AS (
    SELECT
        26.0 AS w_barcode,
        24.0 AS w_title_raw_exact,
        24.0 AS w_artist_raw_exact,
         8.0 AS w_both_raw_exact,

        10.0 AS w_edition_penalty,
         8.0 AS w_various_artist_penalty
),
match_counts AS (
    SELECT
        cds_id,
        COUNT(*) AS n_candidates
    FROM cds_to_mb_exact_matches
    GROUP BY cds_id
),
features AS (
    SELECT
        e.*,
        mc.n_candidates,

        (NULLIF(trim(e.barcode), '') IS NOT NULL) AS has_real_barcode,

        (lower(trim(coalesce(e.artist, ''))) = lower(trim(coalesce(e.main_artist_name, '')))) AS artist_raw_exact,
        (lower(trim(coalesce(e.title,  ''))) = lower(trim(coalesce(e.release_title,   '')))) AS title_raw_exact,

        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])remaster(ed)?([^a-z0-9]|$)') AS mb_has_remaster,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])deluxe([^a-z0-9]|$)')        AS mb_has_deluxe,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])anniversary([^a-z0-9]|$)')   AS mb_has_anniversary,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])live([^a-z0-9]|$)')          AS mb_has_live,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])bonus([^a-z0-9]|$)')         AS mb_has_bonus,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])(disc|disk|cd)[ ]*[0-9]+([^a-z0-9]|$)') AS mb_has_discnum,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])expanded([^a-z0-9]|$)')      AS mb_has_expanded,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])limited([^a-z0-9]|$)')       AS mb_has_limited,
        regexp_matches(lower(coalesce(e.release_title, '')), '(^|[^a-z0-9])special([^a-z0-9]|$)')       AS mb_has_special,

        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])remaster(ed)?([^a-z0-9]|$)') AS cds_has_remaster,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])deluxe([^a-z0-9]|$)')        AS cds_has_deluxe,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])anniversary([^a-z0-9]|$)')   AS cds_has_anniversary,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])live([^a-z0-9]|$)')          AS cds_has_live,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])bonus([^a-z0-9]|$)')         AS cds_has_bonus,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])(disc|disk|cd)[ ]*[0-9]+([^a-z0-9]|$)') AS cds_has_discnum,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])expanded([^a-z0-9]|$)')      AS cds_has_expanded,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])limited([^a-z0-9]|$)')       AS cds_has_limited,
        regexp_matches(lower(coalesce(e.title, '')), '(^|[^a-z0-9])special([^a-z0-9]|$)')       AS cds_has_special
    FROM cds_to_mb_exact_matches e
    JOIN match_counts mc
      ON e.cds_id = mc.cds_id
),
scored AS (
    SELECT
        f.*,

        (
            (CASE WHEN f.mb_has_remaster    AND NOT f.cds_has_remaster    THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_deluxe      AND NOT f.cds_has_deluxe      THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_anniversary AND NOT f.cds_has_anniversary THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_live        AND NOT f.cds_has_live        THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_bonus       AND NOT f.cds_has_bonus       THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_discnum     AND NOT f.cds_has_discnum     THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_expanded    AND NOT f.cds_has_expanded    THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_limited     AND NOT f.cds_has_limited     THEN 1 ELSE 0 END) +
            (CASE WHEN f.mb_has_special     AND NOT f.cds_has_special     THEN 1 ELSE 0 END)
        ) AS edition_penalty,

        (
            p.w_barcode * CAST(f.has_real_barcode AS INTEGER)
            + p.w_title_raw_exact  * CAST(f.title_raw_exact AS INTEGER)
            + p.w_artist_raw_exact * CAST(f.artist_raw_exact AS INTEGER)
            + p.w_both_raw_exact   * CAST((f.title_raw_exact AND f.artist_raw_exact) AS INTEGER)

            - p.w_edition_penalty * (
                (CASE WHEN f.mb_has_remaster    AND NOT f.cds_has_remaster    THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_deluxe      AND NOT f.cds_has_deluxe      THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_anniversary AND NOT f.cds_has_anniversary THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_live        AND NOT f.cds_has_live        THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_bonus       AND NOT f.cds_has_bonus       THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_discnum     AND NOT f.cds_has_discnum     THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_expanded    AND NOT f.cds_has_expanded    THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_limited     AND NOT f.cds_has_limited     THEN 1 ELSE 0 END) +
                (CASE WHEN f.mb_has_special     AND NOT f.cds_has_special     THEN 1 ELSE 0 END)
              )

            - p.w_various_artist_penalty * CAST(
                regexp_matches(lower(coalesce(f.main_artist_name, '')), '(^|[^a-z])various')
                AS INTEGER
              )
        ) AS candidate_score
    FROM features f
    CROSS JOIN params p
),
ranked AS (
    SELECT
        s.*,

        ROW_NUMBER() OVER (
            PARTITION BY cds_id
            ORDER BY
                candidate_score DESC,
                has_real_barcode DESC,
                release_id ASC
        ) AS candidate_rank,

        LEAD(candidate_score) OVER (
            PARTITION BY cds_id
            ORDER BY
                candidate_score DESC,
                has_real_barcode DESC,
                release_id ASC
        ) AS second_best_score
    FROM scored s
)
SELECT
    *,
    (candidate_score - second_best_score) AS score_gap
FROM ranked;
""")

print("✅ cds_to_mb_exact_scored created!")

print("\n--- Scored candidate sample ---")
print(con.execute("""
SELECT
    cds_id, artist, title, cds_year,
    release_id, release_title, main_artist_name, mb_year,
    candidate_score, candidate_rank, second_best_score, score_gap,
    has_real_barcode, artist_raw_exact, title_raw_exact, edition_penalty
FROM cds_to_mb_exact_scored
ORDER BY candidate_score DESC, cds_id
LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_to_mb_exact_scored without year scoring...
✅ cds_to_mb_exact_scored created!

--- Scored candidate sample ---
    cds_id         artist            title  cds_year  release_id    release_title main_artist_name  mb_year  candidate_score  candidate_rank  second_best_score  score_gap  has_real_barcode  artist_raw_exact  \
0     1884   leon redbone  up a lazy river      <NA>       25586  Up a Lazy River     Leon Redbone     2022             82.0               1               82.0        0.0              True              True   
1     1884   leon redbone  up a lazy river      <NA>     2940444  Up a Lazy River     Leon Redbone     2021             82.0               2               82.0        0.0              True              True   
2     1884   leon redbone  up a lazy river      <NA>     4939641  Up a Lazy River     Leon Redbone     2025             82.0               3               56.0       26.0              True              True   
3     2026  alkaline trio

In [ ]:
#cds_to_mb_exact_scored stores all exact CDS–MusicBrainz candidate pairs together with their heuristic ranking features and scores.
#In contrast, cds_to_mb_best_exact_v4 is the reduced output of this ranking step and contains only the top-ranked candidate (candidate_rank = 1) for each CDS record.

In [212]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Recreating cds_to_mb_best_exact_v4 without year-based confidence...")

con.execute("""
CREATE OR REPLACE TABLE cds_to_mb_best_exact_v4 AS
SELECT
    cds_id,
    artist,
    title,
    cds_year,
    category,
    genre,
    tracks,
    release_id,
    release_title,
    main_artist_name,
    artist_credit_full,
    total_tracks,
    barcode,
    mb_year,
    n_candidates,
    candidate_score,
    candidate_rank,
    second_best_score,
    score_gap,
    has_real_barcode,
    artist_raw_exact,
    title_raw_exact,
    edition_penalty,

    CASE
        WHEN n_candidates = 1 THEN 'high'
        WHEN n_candidates > 1
             AND candidate_score >= 60
             AND coalesce(score_gap, 0) >= 8 THEN 'high'
        WHEN n_candidates > 1
             AND candidate_score >= 45
             AND coalesce(score_gap, 0) >= 3 THEN 'medium'
        ELSE 'low'
    END AS match_confidence
FROM cds_to_mb_exact_scored
WHERE candidate_rank = 1;
""")

print("✅ cds_to_mb_best_exact created!")

print("\n--- Check schema ---")
print(con.execute("DESCRIBE cds_to_mb_best_exact_v4").fetchdf())

print("\n--- Best exact match stats ---")
print(con.execute("""
SELECT
    COUNT(*) AS total_best_matches,
    SUM(CASE WHEN match_confidence = 'high' THEN 1 ELSE 0 END) AS high_confidence,
    SUM(CASE WHEN match_confidence = 'medium' THEN 1 ELSE 0 END) AS medium_confidence,
    SUM(CASE WHEN match_confidence = 'low' THEN 1 ELSE 0 END) AS low_confidence
FROM cds_to_mb_best_exact_v4
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
SELECT
    cds_id, artist, title, cds_year,
    release_id, release_title, main_artist_name, mb_year,
    candidate_score, score_gap, n_candidates, match_confidence
FROM cds_to_mb_best_exact_v4
ORDER BY candidate_score DESC, cds_id
LIMIT 20
""").fetchdf())

Connected ✅
Recreating cds_to_mb_best_exact_v4 without year-based confidence...
✅ cds_to_mb_best_exact created!

--- Check schema ---
           column_name    column_type null   key default extra
0               cds_id         BIGINT  YES  None    None  None
1               artist        VARCHAR  YES  None    None  None
2                title        VARCHAR  YES  None    None  None
3             cds_year        INTEGER  YES  None    None  None
4             category        VARCHAR  YES  None    None  None
5                genre        VARCHAR  YES  None    None  None
6               tracks        VARCHAR  YES  None    None  None
7           release_id        INTEGER  YES  None    None  None
8        release_title        VARCHAR  YES  None    None  None
9     main_artist_name        VARCHAR  YES  None    None  None
10  artist_credit_full        VARCHAR  YES  None    None  None
11        total_tracks        HUGEINT  YES  None    None  None
12             barcode        VARCHAR  YES  Non

In [185]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Create a table containing all CDS rows that did NOT receive
# an exact match in cds_to_mb_best_exact_v4
con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_after_exact_v4 AS
    SELECT c.*
    FROM cds_norm c
    LEFT JOIN cds_to_mb_best_exact_v4 m
      ON c.cds_id = m.cds_id
    WHERE m.cds_id IS NULL;
""")

# Export the unmatched rows to CSV for inspection / reporting
con.execute("""
    COPY cds_unmatched_after_exact_v4
    TO '../data_processed/exports/cds_unmatched_after_exact_v4.csv'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Table created: cds_unmatched_after_exact_v4")
print("✅ Saved: ../data_processed/exports/cds_unmatched_after_exact_v4.csv")

# Show how many CDS rows are still unmatched after the exact stage
print(con.execute("""
    SELECT COUNT(*) AS unmatched_rows
    FROM cds_unmatched_after_exact_v4
""").fetchdf())

Connected ✅
✅ Table created: cds_unmatched_after_exact_v4
✅ Saved: ../data_processed/exports/cds_unmatched_after_exact_v4.csv
   unmatched_rows
0            6281


In [213]:
#deletes the old CSVs, then writes the new ones.
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Output folder
out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

tables = [
    "cds_to_mb_exact_matches",
    "cds_to_mb_exact_scored",
    "cds_to_mb_best_exact_v4"
]

for table in tables:
    out_path = os.path.join(out_dir, f"{table}.csv")

    # Delete old CSV if it already exists
    if os.path.exists(out_path):
        os.remove(out_path)
        print(f"🗑️ Deleted old file: {out_path}")

    # Export fresh CSV
    con.execute(f"""
        COPY (
            SELECT *
            FROM {table}
        ) TO '{out_path}'
        (HEADER, DELIMITER ',');
    """)
    print(f"✅ Exported new file: {out_path}")

con.close()
print("Done ✅")

Connected ✅
🗑️ Deleted old file: ../data_processed/exports\cds_to_mb_exact_matches.csv
✅ Exported new file: ../data_processed/exports\cds_to_mb_exact_matches.csv
🗑️ Deleted old file: ../data_processed/exports\cds_to_mb_exact_scored.csv
✅ Exported new file: ../data_processed/exports\cds_to_mb_exact_scored.csv
🗑️ Deleted old file: ../data_processed/exports\cds_to_mb_best_exact_v4.csv
✅ Exported new file: ../data_processed/exports\cds_to_mb_best_exact_v4.csv
Done ✅


In [223]:
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cds_id) AS distinct_cds_id,
        COUNT(*) - COUNT(DISTINCT cds_id) AS duplicate_cds_id_rows
    FROM cds_to_mb_best_exact_v4
""").fetchdf())

Connected ✅
   total_rows  distinct_cds_id  duplicate_cds_id_rows
0        3482             3482                      0


In [222]:
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT release_id) AS distinct_release_id,
        COUNT(*) - COUNT(DISTINCT release_id) AS repeated_release_rows
    FROM cds_to_mb_best_exact_v4
""").fetchdf())

Connected ✅
   total_rows  distinct_release_id  repeated_release_rows
0        3482                 3340                    142


In [220]:
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        COUNT(*) AS cds_links
    FROM cds_to_mb_best_exact_v4
    GROUP BY release_id, release_title, main_artist_name
    HAVING COUNT(*) > 1
    ORDER BY cds_links DESC, release_id
    LIMIT 20
""").fetchdf())

    release_id      release_title   main_artist_name  cds_links
0         7518          Nevermind            Nirvana          5
1          678          ANThology     Alien Ant Farm          4
2         3827      Hybrid Theory        Linkin Park          4
3         8194       Ray of Light            Madonna          4
4         1784          The Bends          Radiohead          3
5         2783     Disintegration           The Cure          3
6         8006    Laundry Service            Shakira          3
7         8013             Let Go      Avril Lavigne          3
8         8417     We Can’t Dance            Genesis          3
9        13268  Money for Nothing       Dire Straits          3
10       18490       Whoa, Nelly!      Nelly Furtado          3
11       38994          Justified  Justin Timberlake          3
12       70115     Mental Jewelry               Live          3
13      246990    The Kick Inside          Kate Bush          3
14     1007028   King James Bible  Alexa

In [235]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Create a profiled version of the unmatched table
# This adds quality flags and cleaned helper columns
con.execute(r"""
    CREATE OR REPLACE TABLE cds_unmatched_profiled_v4 AS
    SELECT
        *,

        -- -----------------------------
        -- Missingness indicators
        -- -----------------------------
        CASE WHEN artist IS NULL OR trim(artist) = '' THEN 1 ELSE 0 END AS missing_artist,
        CASE WHEN title  IS NULL OR trim(title)  = '' THEN 1 ELSE 0 END AS missing_title,
        CASE WHEN tracks IS NULL OR trim(tracks) = '' THEN 1 ELSE 0 END AS missing_tracks,
        CASE WHEN cds_year IS NULL THEN 1 ELSE 0 END AS missing_year,

        -- -----------------------------
        -- Weak / short text indicators
        -- Very short artist/title strings are high-risk for matching
        -- -----------------------------
        CASE WHEN length(trim(coalesce(artist, ''))) < 4 THEN 1 ELSE 0 END AS short_artist,
        CASE WHEN length(trim(coalesce(title,  ''))) < 4 THEN 1 ELSE 0 END AS short_title,

        -- -----------------------------
        -- Mojibake / encoding corruption markers
        -- These patterns often indicate broken text encoding
        -- -----------------------------
        CASE
            WHEN regexp_matches(coalesce(artist, ''), 'Ã|â|Â|�')
              OR regexp_matches(coalesce(title,  ''), 'Ã|â|Â|�')
              OR regexp_matches(coalesce(tracks, ''), 'Ã|â|Â|�')
            THEN 1 ELSE 0
        END AS is_mojibake,

        -- -----------------------------
        -- Various Artists markers
        -- These rows often need a special matching strategy
        -- -----------------------------
        CASE
            WHEN lower(trim(coalesce(artist, ''))) IN ('various', 'various artists', 'va', 'v a', 'v.a.')
            THEN 1 ELSE 0
        END AS is_various_artists,

        -- -----------------------------
        -- disc/cd/vol markers in title
        -- These often create noisy title variants
        -- -----------------------------
        CASE
            WHEN regexp_matches(lower(coalesce(title, '')), '(disc|cd|vol|volume)\s*0*\d+')
            THEN 1 ELSE 0
        END AS has_disc_cd_vol,

        -- -----------------------------
        -- Generic titles
        -- These are risky because they match too many releases
        -- -----------------------------
        CASE
            WHEN lower(trim(coalesce(title, ''))) IN (
                'greatest hits', 'live', 'best of', 'the best of', 'now', 'soundtrack', 'hits', 'mix', 'mixes'
            )
            THEN 1 ELSE 0
        END AS is_generic_title,

        -- -----------------------------
        -- Cleaned helper text
        -- Lowercase + normalized whitespace
        -- -----------------------------
        regexp_replace(lower(coalesce(artist, '')), '\s+', ' ', 'g') AS artist_clean,
        regexp_replace(lower(coalesce(title,  '')), '\s+', ' ', 'g') AS title_clean,

        -- Remove disc/cd/vol number patterns from title
        regexp_replace(
            regexp_replace(lower(coalesce(title, '')), '(disc|cd|vol|volume)\s*0*\d+', '', 'g'),
            '\s+', ' ', 'g'
        ) AS title_no_disc,

        -- -----------------------------
        -- Clean alphanumeric keys for later matching / grouping
        -- -----------------------------
        lower(regexp_replace(coalesce(artist, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key_clean,
        lower(regexp_replace(
            regexp_replace(
                regexp_replace(lower(coalesce(title, '')), '(disc|cd|vol|volume)\s*0*\d+', '', 'g'),
                '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
            ),
            '[^a-zA-Z0-9]+', '', 'g'
        )) AS title_key_clean

    FROM cds_unmatched_after_exact_v4
""")

print("✅ Table created: cds_unmatched_profiled_v4")

Connected ✅
✅ Table created: cds_unmatched_profiled_v4


In [236]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Summarize the main data-quality issues in the unmatched set
print(con.execute("""
    SELECT
        COUNT(*) AS total_unmatched,
        SUM(missing_artist) AS missing_artist,
        SUM(missing_title) AS missing_title,
        SUM(missing_tracks) AS missing_tracks,
        SUM(missing_year) AS missing_year,
        SUM(short_artist) AS short_artist,
        SUM(short_title) AS short_title,
        SUM(is_mojibake) AS mojibake_rows,
        SUM(is_various_artists) AS various_artist_rows,
        SUM(has_disc_cd_vol) AS disc_cd_vol_rows,
        SUM(is_generic_title) AS generic_title_rows
    FROM cds_unmatched_profiled_v4
""").fetchdf())

Connected ✅
   total_unmatched  missing_artist  missing_title  missing_tracks  missing_year  short_artist  short_title  mojibake_rows  various_artist_rows  disc_cd_vol_rows  generic_title_rows
0             6281             7.0           11.0             1.0        3019.0         178.0        114.0          509.0               1087.0            1163.0                38.0


In [248]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating improved unmatched profiling with score + reason + priority...")

con.execute(r"""
CREATE OR REPLACE TABLE cds_unmatched_profiled_v5 AS
WITH scored AS (
    SELECT
        u.*,

        CASE
            WHEN missing_artist = 1 OR missing_title = 1 THEN 'missing_core_fields'
            WHEN is_mojibake = 1 THEN 'encoding_corruption'
            WHEN short_artist = 1 OR short_title = 1 THEN 'short_text'
            WHEN is_various_artists = 1 THEN 'various_artists_case'
            WHEN is_generic_title = 1 THEN 'generic_title_case'
            WHEN has_disc_cd_vol = 1 THEN 'disc_volume_variant'
            ELSE 'clean_but_unmatched'
        END AS primary_unmatched_reason,

        (
            5 * CAST((missing_artist = 1 OR missing_title = 1) AS INTEGER)
          + 4 * CAST(is_mojibake AS INTEGER)
          + 2 * CAST((short_artist = 1 OR short_title = 1) AS INTEGER)
          + 2 * CAST(is_various_artists AS INTEGER)
          + 2 * CAST(is_generic_title AS INTEGER)
          + 1 * CAST(has_disc_cd_vol AS INTEGER)
          + 1 * CAST(missing_year AS INTEGER)
          + 1 * CAST(missing_tracks AS INTEGER)
        ) AS unmatched_risk_score

    FROM cds_unmatched_profiled_v4 u
),
final AS (
    SELECT
        scored.*,
        CASE
            WHEN unmatched_risk_score >= 5 THEN 'low'     -- noisy / not worth much effort
            WHEN unmatched_risk_score >= 2 THEN 'medium'  -- special handling might help
            ELSE 'high'                                   -- clean candidates for next-step matching
        END AS next_step_priority
    FROM scored
)
SELECT * FROM final;
""")

print("✅ Table created: cds_unmatched_profiled_v5")

print("\n--- Row count ---")
print(con.execute("SELECT COUNT(*) AS n FROM cds_unmatched_profiled_v5").fetchdf())

print("\n--- Priority distribution ---")
print(con.execute("""
    SELECT next_step_priority, COUNT(*) AS n
    FROM cds_unmatched_profiled_v5
    GROUP BY next_step_priority
    ORDER BY
        CASE next_step_priority
            WHEN 'high' THEN 1
            WHEN 'medium' THEN 2
            ELSE 3
        END
""").fetchdf())

print("\n--- Main unmatched reasons ---")
print(con.execute("""
    SELECT primary_unmatched_reason, COUNT(*) AS n
    FROM cds_unmatched_profiled_v5
    GROUP BY primary_unmatched_reason
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        unmatched_risk_score,
        primary_unmatched_reason,
        next_step_priority
    FROM cds_unmatched_profiled_v5
    LIMIT 20
""").fetchdf())

Connected ✅
Creating improved unmatched profiling with score + reason + priority...
✅ Table created: cds_unmatched_profiled_v5

--- Row count ---
      n
0  6281

--- Priority distribution ---
  next_step_priority     n
0               high  4178
1             medium  1800
2                low   303

--- Main unmatched reasons ---
  primary_unmatched_reason     n
0      clean_but_unmatched  3799
1     various_artists_case  1002
2      disc_volume_variant   697
3      encoding_corruption   508
4               short_text   230
5       generic_title_case    28
6      missing_core_fields    17

--- Preview ---
    cds_id                        artist                                                                         title  unmatched_risk_score primary_unmatched_reason next_step_priority
0   100001                       various                                                   frankfurt trance vol 04 cd1                     4     various_artists_case             medium
1   100003      

In [249]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Count how many unmatched rows fall into each priority bucket
print(con.execute("""
    SELECT next_step_priority, COUNT(*) AS n
    FROM cds_unmatched_profiled_v5
    GROUP BY next_step_priority
    ORDER BY n DESC
""").fetchdf())

Connected ✅
  next_step_priority     n
0               high  4178
1             medium  1800
2                low   303


In [250]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Export the profiled unmatched table to CSV
# This file is useful for manual review and thesis reporting
con.execute("""
    COPY cds_unmatched_profiled_v5
    TO '../data_processed/exports/cds_unmatched_profiled_v5.csv'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Saved: ../data_processed/exports/cds_unmatched_profiled_v5.csv")

Connected ✅
✅ Saved: ../data_processed/exports/cds_unmatched_profiled_v5.csv


In [251]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Show a sample of unmatched rows with the new quality flags
# Useful for sanity-checking the profiling logic
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year,
        is_mojibake, is_various_artists, has_disc_cd_vol, is_generic_title,
        next_step_priority
    FROM cds_unmatched_profiled_v5
    ORDER BY
        next_step_priority DESC,
        is_mojibake DESC,
        is_various_artists DESC,
        has_disc_cd_vol DESC
    LIMIT 50
""").fetchdf())

Connected ✅
    cds_id                                           artist                                                                                                       title  cds_year  is_mojibake  is_various_artists  \
0   100003                          ã â â ã â ã â ºâ ã â å                                                                        ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ       1989            1                   0   
1   101581                                      hajime sato                                                                                         blue best 1998 2003      2003            1                   0   
2   100034                          ã â šã å â ã å å ã â â¾                                                                                                        lucy      2001            1                   0   
3   100051                  ã â â ã â å½ã â¾ã â ã â šë ã â                                                                          

In [252]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Show the most frequent artist names in the unmatched set
# This can reveal repeated failure patterns such as Various Artists or noisy artist names
print(con.execute("""
    SELECT artist, COUNT(*) AS n
    FROM cds_unmatched_profiled_v5
    GROUP BY artist
    ORDER BY n DESC
    LIMIT 20
""").fetchdf())

Connected ✅
             artist    n
0           various  618
1   various artists  436
2            varios   38
3           diverse   18
4    various artist   15
5                va   14
6       compilation   14
7            mozart   12
8         bob dylan   11
9         metallica   10
10             v a    10
11          vã rios    9
12               u2    9
13       pink floyd    8
14      the beatles    8
15             bach    8
16    elvis presley    8
17          sampler    8
18  varios artistas    7
19         j s bach    7


In [247]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Show the most frequent titles in the unmatched set
# This helps identify generic titles such as "Greatest Hits" or "Live"
print(con.execute("""
    SELECT title, COUNT(*) AS n
    FROM cds_unmatched_profiled_v5
    GROUP BY title
    ORDER BY n DESC
    LIMIT 20
""").fetchdf())

Connected ✅
                     title   n
0                     live  11
1            greatest hits  10
2                     None   8
3                      now   6
4      greatest hits vol 2   3
5           the collection   3
6                 volume 1   3
7               soundtrack   3
8                            3
9   ã ã ã ã ã ã ã ã ã ã ã    3
10          grandes exitos   3
11                     ten   3
12              next world   3
13            instrumental   3
14                     cd2   3
15                       3   3
16                 best of   3
17          it s christmas   3
18                the best   3
19             the best of   3


In [366]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating corrected cds_unmatched_for_enrichment...")

con.execute(r"""
CREATE OR REPLACE TABLE cds_unmatched_for_enrichment AS
SELECT
    cds_id,
    artist,
    title,
    cds_year,
    tracks,
    next_step_priority,
    is_mojibake,
    is_various_artists,
    has_disc_cd_vol,
    is_generic_title,

    trim(regexp_replace(lower(coalesce(artist, '')), '\s+', ' ', 'g')) AS artist_clean,
    trim(regexp_replace(lower(coalesce(title,  '')), '\s+', ' ', 'g')) AS title_clean,

    -- Extract disc/cd/vol number BEFORE removing it
    TRY_CAST(
        regexp_extract(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*([0-9]+)', 2)
        AS INTEGER
    ) AS part_number,

    -- Core title without disc/cd/vol marker
    trim(regexp_replace(
        regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*[0-9]+', '', 'g'),
        '\s+', ' ', 'g'
    )) AS title_core,

    lower(regexp_replace(coalesce(artist, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key_clean,

    lower(regexp_replace(
        regexp_replace(
            regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*[0-9]+', '', 'g'),
            '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
        ),
        '[^a-zA-Z0-9]+', '', 'g'
    )) AS title_key_clean

FROM cds_unmatched_profiled_v5
WHERE next_step_priority IN ('high', 'medium')
""")

print("✅ Table created: cds_unmatched_for_enrichment")

Connected ✅
Creating corrected cds_unmatched_for_enrichment...
✅ Table created: cds_unmatched_for_enrichment


In [367]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating corrected mb_integration_view...")

con.execute(r"""
CREATE OR REPLACE VIEW mb_integration_view AS
SELECT
    release_id,
    release_title,
    main_artist_name,
    artist_credit_full,
    total_tracks,
    barcode,
    mb_year,

    trim(regexp_replace(lower(coalesce(main_artist_name, '')), '\s+', ' ', 'g')) AS artist_clean,
    trim(regexp_replace(lower(coalesce(release_title,  '')), '\s+', ' ', 'g')) AS title_clean,

    TRY_CAST(
        regexp_extract(lower(coalesce(release_title, '')), '(disc|disk|cd|vol|volume)\s*0*([0-9]+)', 2)
        AS INTEGER
    ) AS part_number,

    trim(regexp_replace(
        regexp_replace(lower(coalesce(release_title, '')), '(disc|disk|cd|vol|volume)\s*0*[0-9]+', '', 'g'),
        '\s+', ' ', 'g'
    )) AS title_core,

    lower(regexp_replace(coalesce(main_artist_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key_clean,

    lower(regexp_replace(
        regexp_replace(
            regexp_replace(lower(coalesce(release_title, '')), '(disc|disk|cd|vol|volume)\s*0*[0-9]+', '', 'g'),
            '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
        ),
        '[^a-zA-Z0-9]+', '', 'g'
    )) AS title_key_clean
FROM mb_release_features
""")

print("✅ View created: mb_integration_view")

Connected ✅
Creating corrected mb_integration_view...
✅ View created: mb_integration_view


In [368]:
#deletes the old CSVs, then writes the new ones.
import duckdb
import os

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Output folder
out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

tables = [

    "mb_integration_view",
    "cds_unmatched_for_enrichment"
]

for table in tables:
    out_path = os.path.join(out_dir, f"{table}.csv")

    # Delete old CSV if it already exists
    if os.path.exists(out_path):
        os.remove(out_path)
        print(f"🗑️ Deleted old file: {out_path}")

    # Export fresh CSV
    con.execute(f"""
        COPY (
            SELECT *
            FROM {table}
        ) TO '{out_path}'
        (HEADER, DELIMITER ',');
    """)
    print(f"✅ Exported new file: {out_path}")

con.close()
print("Done ✅")

Connected ✅
🗑️ Deleted old file: ../data_processed/exports\mb_integration_view.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported new file: ../data_processed/exports\mb_integration_view.csv
🗑️ Deleted old file: ../data_processed/exports\cds_unmatched_for_enrichment.csv
✅ Exported new file: ../data_processed/exports\cds_unmatched_for_enrichment.csv
Done ✅


In [369]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating corrected mb_candidate_pool_v2...")

con.execute("""
CREATE OR REPLACE TABLE mb_candidate_pool_v2 AS
SELECT
    release_id,
    release_title,
    main_artist_name,
    artist_credit_full,
    total_tracks,
    barcode,
    mb_year,
    artist_clean,
    title_clean,
    title_core,
    artist_key_clean,
    title_key_clean,
    part_number,
    substr(artist_key_clean, 1, 1) AS artist_first,
    substr(title_key_clean,  1, 1) AS title_first,
    length(artist_key_clean) AS artist_len,
    length(title_key_clean)  AS title_len
FROM mb_integration_view
WHERE
    artist_key_clean IS NOT NULL
    AND title_key_clean IS NOT NULL
    AND length(artist_key_clean) >= 4
    AND length(title_key_clean) >= 4
""")

print("✅ Table created: mb_candidate_pool_v2")

Connected ✅
Creating corrected mb_candidate_pool_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: mb_candidate_pool_v2


In [372]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step: Generate blocked candidate pairs for enrichment
# ------------------------------------------------------------
# Comments:
# - This creates plausible CDS–MB release candidate pairs before scoring.
# - We block on:
#     * same first artist-key letter
#     * same first title-key letter
#     * similar artist/title key lengths
# - We exclude the noisiest CDS rows here:
#     * mojibake
#     * Various Artists (handled separately)
# - Important correction:
#     * if both CDS and MB titles contain an explicit disc/cd/volume number,
#       they must agree (e.g. "Vol. 1" should NOT match "Vol. 3").
print("Creating corrected cds_mb_enrichment_pairs...")

con.execute("""
CREATE OR REPLACE TABLE cds_mb_enrichment_pairs AS
SELECT
    -- --------------------------------------------------------
    -- CDS side
    -- --------------------------------------------------------
    c.cds_id,
    c.artist,
    c.title,
    c.cds_year,
    c.tracks,
    c.artist_clean AS cds_artist_clean,
    c.title_core AS cds_title_core,
    c.artist_key_clean AS cds_artist_key,
    c.title_key_clean AS cds_title_key,
    c.part_number AS cds_part_number,

    -- --------------------------------------------------------
    -- MusicBrainz side
    -- --------------------------------------------------------
    m.release_id,
    m.main_artist_name,
    m.release_title,
    m.artist_credit_full,
    m.total_tracks,
    m.barcode,
    m.mb_year,
    m.artist_clean AS mb_artist_clean,
    m.title_core AS mb_title_core,
    m.artist_key_clean AS mb_artist_key,
    m.title_key_clean AS mb_title_key,
    m.part_number AS mb_part_number

FROM (
    SELECT
        *,
        substr(artist_key_clean, 1, 1) AS artist_first,
        substr(title_key_clean, 1, 1) AS title_first,
        length(artist_key_clean) AS artist_len,
        length(title_key_clean) AS title_len
    FROM cds_unmatched_for_enrichment
    WHERE
        is_mojibake = 0
        AND is_various_artists = 0
        AND artist_key_clean IS NOT NULL
        AND title_key_clean IS NOT NULL
        AND length(artist_key_clean) >= 4
        AND length(title_key_clean) >= 4
) c
JOIN mb_candidate_pool_v2 m
  ON m.artist_first = c.artist_first
 AND m.title_first = c.title_first
 AND m.artist_len BETWEEN c.artist_len - 3 AND c.artist_len + 3
 AND m.title_len BETWEEN c.title_len - 5 AND c.title_len + 5

 -- ----------------------------------------------------------
 -- Part-number compatibility rule
 -- If both sides explicitly have a part number, they must match.
 -- This prevents false links like:
 --   CDS: "Greatest Hits Vol. 1"
 --   MB : "Greatest Hits Vol. 3"
 -- ----------------------------------------------------------
 AND (
      c.part_number IS NULL
      OR m.part_number IS NULL
      OR c.part_number = m.part_number
 )
""")

print("✅ Table created: cds_mb_enrichment_pairs")

print("\n--- Pair count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_enrichment_pairs
""").fetchdf())

print("\n--- Rows with explicit part-number agreement on both sides ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_enrichment_pairs
    WHERE cds_part_number IS NOT NULL
      AND mb_part_number IS NOT NULL
      AND cds_part_number = mb_part_number
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_part_number,
        release_id,
        main_artist_name,
        release_title,
        mb_part_number
    FROM cds_mb_enrichment_pairs
    LIMIT 25
""").fetchdf())

Connected ✅
Creating corrected cds_mb_enrichment_pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_mb_enrichment_pairs

--- Pair count ---
          n
0  11845921

--- Rows with explicit part-number agreement on both sides ---
      n
0  6357

--- Preview ---
    cds_id                artist                                title  cds_part_number  release_id        main_artist_name                    release_title  mb_part_number
0   109006  mela tenenbaum viola                   richard kapp piano             <NA>     2750291  Manic Street Preachers        Rip It Up 5 Track Sampler            <NA>
1     7072            spineshank             the height of callousnes             <NA>     4965994           She is Legend             This Game Needs Guns            <NA>
2     1823        carlos santana                               carlos             <NA>     2912117       Corpsefucking Art                          Corpsex            <NA>
3   109077   sir john barbirolli                           halle orch             <NA>     3237600         Solemn Assembly        

In [373]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Scoring corrected candidate pairs...")

con.execute("""
CREATE OR REPLACE VIEW cds_mb_enrichment_pairs_indexed AS
SELECT row_number() OVER () AS rid, *
FROM cds_mb_enrichment_pairs
""")

total_pairs = con.execute("SELECT COUNT(*) FROM cds_mb_enrichment_pairs").fetchone()[0]
print(f"Total candidate pairs to score: {total_pairs:,}")

CHUNK_SIZE = 250_000
KEEP_PER_CHUNK = 3
all_best = []

for start in range(1, total_pairs + 1, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE - 1, total_pairs)

    df = con.execute("""
        SELECT *
        FROM cds_mb_enrichment_pairs_indexed
        WHERE rid BETWEEN ? AND ?
    """, [start, end]).fetchdf()

    if df.empty:
        continue

    artist_sims = []
    title_sims = []
    scores = []

    for _, row in df.iterrows():
        a_sim = fuzz.token_set_ratio(str(row["cds_artist_clean"]), str(row["mb_artist_clean"]))
        t_sim = fuzz.token_set_ratio(str(row["cds_title_core"]), str(row["mb_title_core"]))

        score = 0.4 * a_sim + 0.6 * t_sim

        artist_sims.append(a_sim)
        title_sims.append(t_sim)
        scores.append(score)

    df["artist_sim"] = artist_sims
    df["title_sim"] = title_sims
    df["score"] = scores

    df = (
        df.sort_values(["cds_id", "score"], ascending=[True, False])
          .groupby("cds_id")
          .head(KEEP_PER_CHUNK)
    )

    all_best.append(df)

best_df = pd.concat(all_best, ignore_index=True) if all_best else pd.DataFrame()

best_df = (
    best_df.sort_values(["cds_id", "score"], ascending=[True, False])
           .groupby("cds_id")
           .head(5)
)

con.register("best_df_view", best_df)

con.execute("""
CREATE OR REPLACE TABLE cds_to_mb_enrichment_candidates AS
SELECT * FROM best_df_view
""")

print("✅ Table created: cds_to_mb_enrichment_candidates")

Connected ✅
Scoring corrected candidate pairs...
Total candidate pairs to score: 11,845,921


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_to_mb_enrichment_candidates


In [375]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

SCORE_TH = 90.0
TITLE_TH = 92.0

print(f"Creating cds_to_mb_best_enrichment with score >= {SCORE_TH} and title_sim >= {TITLE_TH}...")

con.execute(f"""
CREATE OR REPLACE TABLE cds_to_mb_best_enrichment AS
WITH filtered AS (
    SELECT *
    FROM cds_to_mb_enrichment_candidates
    WHERE score >= {SCORE_TH}
      AND title_sim >= {TITLE_TH}
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY cds_id
            ORDER BY
                score DESC,
                title_sim DESC,
                artist_sim DESC,
                release_id ASC
        ) AS rn,
        LEAD(score) OVER (
            PARTITION BY cds_id
            ORDER BY
                score DESC,
                title_sim DESC,
                artist_sim DESC,
                release_id ASC
        ) AS second_best_score
    FROM filtered
),
cds_level AS (
    SELECT
        cds_id,
        COUNT(DISTINCT mb_artist_key) AS n_artist_keys,
        COUNT(DISTINCT mb_title_key) AS n_title_keys
    FROM filtered
    GROUP BY cds_id
)
SELECT
    r.cds_id,
    r.artist,
    r.title,
    r.cds_year,
    r.release_id,
    r.release_title,
    r.main_artist_name,
    r.artist_credit_full,
    r.total_tracks,
    r.barcode,
    r.mb_year,
    r.score,
    r.artist_sim,
    r.title_sim,
    r.second_best_score,
    (r.score - r.second_best_score) AS score_gap,
    CASE
        WHEN r.second_best_score IS NULL THEN 'high'
        WHEN c.n_artist_keys = 1 AND c.n_title_keys = 1 THEN 'high'
        WHEN (r.score - r.second_best_score) >= 5 THEN 'high'
        WHEN (r.score - r.second_best_score) >= 2 THEN 'medium'
        ELSE 'low'
    END AS enrichment_confidence
FROM ranked r
JOIN cds_level c
  ON r.cds_id = c.cds_id
WHERE r.rn = 1
""")

print("✅ Table created: cds_to_mb_best_enrichment")

Connected ✅
Creating cds_to_mb_best_enrichment with score >= 90.0 and title_sim >= 92.0...
✅ Table created: cds_to_mb_best_enrichment


In [376]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("\n--- Low-confidence enrichment pairs ---")
low_df = con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        mb_year,
        score,
        title_sim,
        artist_sim,
        second_best_score,
        score_gap,
        enrichment_confidence
    FROM cds_to_mb_best_enrichment
    WHERE enrichment_confidence = 'low'
    ORDER BY score DESC, score_gap ASC, cds_id
    LIMIT 50
""").fetchdf()

low_df

Connected ✅

--- Low-confidence enrichment pairs ---


,cds_id,artist,title,cds_year,release_id,release_title,main_artist_name,mb_year,score,title_sim,artist_sim,second_best_score,score_gap,enrichment_confidence
0,102424,armin van buuren,a state of trance 2004 cd2,2004,81056,A State of Trance 2004,Armin van Buuren,2023,100.000000,100.000000,100.000000,100.000000,0.000000,low
1,103702,peter rauhofer,live roxy cd1,2002,441999,Live @ Roxy 4,Peter Rauhofer,2024,100.000000,100.000000,100.000000,100.000000,0.000000,low
2,104080,roy orbison,roy orbison cd2,<NA>,621375,Roy Orbison,Roy Orbison,2025,100.000000,100.000000,100.000000,100.000000,0.000000,low
3,104673,butter 08,butter 08,<NA>,163082,Butter,Butter 08,2024,100.000000,100.000000,100.000000,100.000000,0.000000,low
4,104846,the beatles,the beatles vol 9,<NA>,212391,The Beatles,The Beatles,2020,100.000000,100.000000,100.000000,100.000000,0.000000,low
5,105070,the beatles,the black album disc 3,<NA>,861020,The Black Album,The Beatles,2016,100.000000,100.000000,100.000000,100.000000,0.000000,low
6,105433,new order,substance 1987 disc 1,1987,16498,Substance 1987,New Order,2022,100.000000,100.000000,100.000000,100.000000,0.000000,low
7,107480,armin van buuren,a state of trance 2004 cd1,2004,81056,A State of Trance 2004,Armin van Buuren,2023,100.000000,100.000000,100.000000,100.000000,0.000000,low
8,108846,joseph haydn,the 12 london symphonies cd 2 4,1972,365121,The 12 London Symphonies,Joseph Haydn,2024,100.000000,100.000000,100.000000,100.000000,0.000000,low
9,108885,johann sebastian bach,cantatas vol 5 cd 11,1996,1026137,Cantatas BWV 4-6,Johann Sebastian Bach,2022,100.000000,100.000000,100.000000,100.000000,0.000000,low


In [377]:
# 1) Inspect all candidates for one problematic cds_id, e.g. 1838
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        cds_id,
        release_id,
        mb_artist_key,
        mb_title_key,
        total_tracks,
        score
    FROM cds_to_mb_enrichment_candidates
    WHERE cds_id = 1838
    ORDER BY score DESC, release_id
""").fetchdf())

# 2) See how cds_level aggregates look for the same cds_id
print(con.execute("""
    SELECT *
    FROM (
        SELECT
            cds_id,
            MIN(score)  AS min_score,
            MAX(score)  AS max_score,
            COUNT(DISTINCT mb_artist_key) AS n_artist_keys,
            COUNT(DISTINCT mb_title_key)  AS n_title_keys,
            COUNT(DISTINCT total_tracks)  AS n_track_counts
        FROM cds_to_mb_enrichment_candidates
        WHERE score >= 90 AND title_sim >= 92
        GROUP BY cds_id
    )
    WHERE cds_id IN (1838, 2707, 6413, 100492)
""").fetchdf())

Connected ✅
   cds_id  release_id mb_artist_key  mb_title_key  total_tracks  score
0    1838      137707          2pac  greatesthits          25.0  100.0
1    1838     2208058          2pac  greatesthits          25.0  100.0
2    1838     2610338          2pac  greatesthits          25.0  100.0
3    1838     3303486          2pac  greatesthits          25.0  100.0
4    1838     3473055          2pac  greatesthits          25.0  100.0
   cds_id  min_score  max_score  n_artist_keys  n_title_keys  n_track_counts
0    6413      100.0      100.0              1             1               2
1  100492      100.0      100.0              1             1               2
2    1838      100.0      100.0              1             1               1
3    2707      100.0      100.0              1             1               3


In [378]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating corrected cds_to_mb_final_mapping_v5...")

con.execute("""
CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v5 AS

SELECT
    x.cds_id,
    x.artist,
    x.title,
    x.cds_year,
    x.release_id,
    x.release_title,
    x.main_artist_name,
    x.artist_credit_full,
    x.total_tracks,
    x.barcode,
    x.mb_year,
    'exact' AS match_type,
    x.match_confidence AS confidence_label,
    x.candidate_score AS score,
    NULL::DOUBLE AS title_sim,
    NULL::DOUBLE AS artist_sim
FROM cds_to_mb_best_exact_v4 x

UNION ALL

SELECT
    e.cds_id,
    e.artist,
    e.title,
    e.cds_year,
    e.release_id,
    e.release_title,
    e.main_artist_name,
    e.artist_credit_full,
    e.total_tracks,
    e.barcode,
    e.mb_year,
    'fuzzy_second_pass' AS match_type,
    e.enrichment_confidence AS confidence_label,
    e.score,
    e.title_sim,
    e.artist_sim
FROM cds_to_mb_best_enrichment e
LEFT JOIN cds_to_mb_best_exact_v4 x
  ON x.cds_id = e.cds_id
WHERE x.cds_id IS NULL
  AND e.enrichment_confidence = 'high'
""")

print("✅ Table created: cds_to_mb_final_mapping_v5")

print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cds_id) AS distinct_cds_id,
        COUNT(*) - COUNT(DISTINCT cds_id) AS duplicate_cds_id_rows
    FROM cds_to_mb_final_mapping_v5
""").fetchdf())

Connected ✅
Creating corrected cds_to_mb_final_mapping_v5...
✅ Table created: cds_to_mb_final_mapping_v5
   total_rows  distinct_cds_id  duplicate_cds_id_rows
0        4017             4017                      0


In [379]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 8: Export useful outputs to CSV
# ------------------------------------------------------------
print("Exporting enrichment outputs...")

con.execute("""
    COPY cds_to_mb_best_enrichment
    TO '../data_processed/exports/cds_to_mb_best_enrichment.csv'
    WITH (HEADER, DELIMITER ',')
""")

con.execute("""
    COPY cds_to_mb_final_mapping_v5
    TO '../data_processed/exports/cds_to_mb_final_mapping_v5.csv'
    WITH (HEADER, DELIMITER ',')
""")

print("✅ Saved: ../data_processed/exports/cds_to_mb_best_enrichment.csv")
print("✅ Saved: ../data_processed/exports/cds_to_mb_final_mapping_v5.csv")

Connected ✅
Exporting enrichment outputs...
✅ Saved: ../data_processed/exports/cds_to_mb_best_enrichment.csv
✅ Saved: ../data_processed/exports/cds_to_mb_final_mapping_v5.csv


In [380]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Thesis report table 1:
# Overall matching coverage after exact matching and enrichment
# ------------------------------------------------------------
report_coverage = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(*) FROM cds_to_mb_best_exact_v4) AS exact_v4_matched,
        (SELECT COUNT(*) FROM cds_unmatched_after_exact_v4) AS unmatched_after_exact_v4,
        (SELECT COUNT(*) FROM cds_to_mb_best_enrichment) AS enrichment_added,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v5) AS total_matched_after_enrichment,
        (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v5) AS still_unmatched_after_enrichment
""").fetchdf()

print("\n--- Thesis Report: Coverage Summary ---")
print(report_coverage)

Connected ✅

--- Thesis Report: Coverage Summary ---
   cds_total  exact_v4_matched  unmatched_after_exact_v4  enrichment_added  total_matched_after_enrichment  still_unmatched_after_enrichment
0       9763              3482                      6281               564                            4017                              5746


In [381]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Thesis report table 2:
# Data profiling of the unmatched CDS remainder after exact matching
# ------------------------------------------------------------
report_unmatched_profile = con.execute("""
    SELECT
        COUNT(*) AS total_unmatched,
        SUM(missing_artist) AS missing_artist,
        SUM(missing_title) AS missing_title,
        SUM(missing_tracks) AS missing_tracks,
        SUM(missing_year) AS missing_year,
        SUM(short_artist) AS short_artist,
        SUM(short_title) AS short_title,
        SUM(is_mojibake) AS mojibake_rows,
        SUM(is_various_artists) AS various_artist_rows,
        SUM(has_disc_cd_vol) AS disc_cd_vol_rows,
        SUM(is_generic_title) AS generic_title_rows
    FROM cds_unmatched_profiled_v4
""").fetchdf()

print("\n--- Thesis Report: Unmatched Data Profiling ---")
print(report_unmatched_profile)

Connected ✅

--- Thesis Report: Unmatched Data Profiling ---
   total_unmatched  missing_artist  missing_title  missing_tracks  missing_year  short_artist  short_title  mojibake_rows  various_artist_rows  disc_cd_vol_rows  generic_title_rows
0             6281             7.0           11.0             1.0        3019.0         178.0        114.0          509.0               1087.0            1163.0                38.0


In [293]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step: Standardize and isolate Various Artists rows
# from the unmatched set for a title-driven matching pipeline
# ------------------------------------------------------------
print("Creating cds_various_unmatched_cleaned...")

con.execute(r"""
    CREATE OR REPLACE TABLE cds_various_unmatched_cleaned AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        next_step_priority,
        is_mojibake,
        has_disc_cd_vol,
        is_generic_title,

        -- Canonicalized artist label
        'various_artists' AS artist_canonical,

        -- Clean title text
        trim(regexp_replace(lower(coalesce(title, '')), '\s+', ' ', 'g')) AS title_clean,

        -- Remove disc/cd/vol number patterns
        trim(regexp_replace(
            regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
            '\s+', ' ', 'g'
        )) AS title_no_disc,

        -- Title-only matching key
        lower(regexp_replace(
            regexp_replace(
                regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
                '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
            ),
            '[^a-zA-Z0-9]+', '', 'g'
        )) AS title_key_various

    FROM cds_unmatched_profiled_v5
    WHERE lower(trim(coalesce(artist, ''))) IN (
        'various', 'various artists', 'various artist',
        'va', 'v a', 'v.a.',
        'varios', 'varios artistas', 'vàrios'
    )
      AND coalesce(title, '') <> ''
      AND is_mojibake = 0
""")

print("✅ Table created: cds_various_unmatched_cleaned")

print("\n--- Preview: standardized Various Artists rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        artist_canonical,
        title,
        title_no_disc,
        title_key_various,
        next_step_priority
    FROM cds_various_unmatched_cleaned
    LIMIT 20
""").fetchdf())

print("\n--- Count of standardized Various Artists rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_various_unmatched_cleaned
""").fetchdf())

Connected ✅
Creating cds_various_unmatched_cleaned...
✅ Table created: cds_various_unmatched_cleaned

--- Preview: standardized Various Artists rows ---
    cds_id           artist artist_canonical                                            title                                    title_no_disc                         title_key_various next_step_priority
0   100001          various  various_artists                      frankfurt trance vol 04 cd1                                 frankfurt trance                           frankfurttrance             medium
1   100007  various artists  various_artists                                        nop pop 2                                        nop pop 2                                   noppop2             medium
2   100010          various  various_artists                      frankfurt trance vol 05 cd1                                 frankfurt trance                           frankfurttrance             medium
3   100017          various  va

In [294]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Various Artists enrichment step:
# Build a MusicBrainz title-only integration view
# ------------------------------------------------------------
# Comments:
# - For Various Artists rows, artist is not discriminative enough.
# - So we prepare MusicBrainz release titles for title-only matching.
# - We apply similar cleaning:
#   * lowercase
#   * collapse whitespace
#   * remove disc/cd/vol numbers
#   * remove edition markers
#   * create a title-only key
print("Creating mb_various_title_integration_view...")

con.execute(r"""
    CREATE OR REPLACE VIEW mb_various_title_integration_view AS
    SELECT
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,

        -- Clean title text
        trim(regexp_replace(lower(coalesce(release_title, '')), '\s+', ' ', 'g')) AS title_clean,

        -- Remove disc/cd/vol number patterns
        trim(regexp_replace(
            regexp_replace(lower(coalesce(release_title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
            '\s+', ' ', 'g'
        )) AS title_no_disc,

        -- Title-only matching key
        lower(regexp_replace(
            regexp_replace(
                regexp_replace(lower(coalesce(release_title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
                '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
            ),
            '[^a-zA-Z0-9]+', '', 'g'
        )) AS title_key_various

    FROM mb_release_features
    WHERE coalesce(release_title, '') <> ''
""")

print("✅ View created: mb_various_title_integration_view")

print("\n--- Preview: MB title-only integration view ---")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        title_no_disc,
        title_key_various
    FROM mb_various_title_integration_view
    WHERE title_key_various IS NOT NULL
      AND length(title_key_various) > 0
    LIMIT 20
""").fetchdf())

Connected ✅
Creating mb_various_title_integration_view...
✅ View created: mb_various_title_integration_view

--- Preview: MB title-only integration view ---
    release_id                         release_title                         title_no_disc                title_key_various
0       178087  Blechdance: Ritter Rost zum Abtanzen  blechdance: ritter rost zum abtanzen  blechdanceritterrostzumabtanzen
1       178088                                 Verte                                 verte                            verte
2       178097                             Egouality                             egouality                        egouality
3       178098                             Egouality                             egouality                        egouality
4       178100                             Démo 2005                             démo 2005                          dmo2005
5       178101                             Démo 2005                             démo 2005         

In [295]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Various Artists enrichment step:
# Generate safe title-only candidate pairs
# ------------------------------------------------------------
# Comments:
# - We match CDS Various Artists rows to MB by cleaned title-only key.
# - This is safer than fuzzy matching on artist because artist is too generic here.
# - We require a minimum key length to reduce collisions.
# - We also exclude mojibake rows from this step.
print("Creating cds_various_candidate_pairs...")

con.execute("""
    CREATE OR REPLACE TABLE cds_various_candidate_pairs AS
    SELECT
        c.cds_id,
        c.artist,
        c.artist_canonical,
        c.title,
        c.title_no_disc,
        c.title_key_various,
        c.cds_year,
        c.next_step_priority,

        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        m.title_no_disc AS mb_title_no_disc

    FROM cds_various_unmatched_cleaned c
    JOIN mb_various_title_integration_view m
      ON c.title_key_various = m.title_key_various
    WHERE
        c.is_mojibake = 0
        AND c.title_key_various IS NOT NULL
        AND length(c.title_key_various) >= 8
""")

print("✅ Table created: cds_various_candidate_pairs")

print("\n--- Candidate pair count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_various_candidate_pairs
""").fetchdf())

print("\n--- Distinct CDS rows with candidates ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_candidates
    FROM cds_various_candidate_pairs
""").fetchdf())

print("\n--- Preview: Various Artists candidate pairs ---")
print(con.execute("""
    SELECT
        cds_id,
        title,
        title_no_disc,
        release_id,
        release_title,
        mb_title_no_disc,
        total_tracks,
        barcode,
        mb_year,
        next_step_priority
    FROM cds_various_candidate_pairs
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_various_candidate_pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_various_candidate_pairs

--- Candidate pair count ---
      n
0  3817

--- Distinct CDS rows with candidates ---
   cds_rows_with_candidates
0                       388

--- Preview: Various Artists candidate pairs ---
    cds_id                          title                  title_no_disc  release_id                             release_title                mb_title_no_disc  total_tracks        barcode  mb_year next_step_priority
0   102701  disco pogo for punks in pumps  disco pogo for punks in pumps      179806  Disco Pogo for Punks in Pumps, Volume 11  disco pogo for punks in pumps,          12.0           None     2013             medium
1   102701  disco pogo for punks in pumps  disco pogo for punks in pumps      179809  Disco Pogo for Punks in Pumps, Volume 12  disco pogo for punks in pumps,          12.0           None     2013             medium
2   109704                    love song s                    love song s      180462                            

In [297]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Scoring cds_various_candidate_pairs and creating cds_various_best_match...")

df = con.execute("""
    SELECT
        p.cds_id,
        p.artist,
        p.artist_canonical,
        p.title,
        p.title_no_disc,
        p.title_key_various,
        p.cds_year,
        p.next_step_priority,

        p.release_id,
        p.release_title,
        p.main_artist_name,
        p.artist_credit_full,
        p.total_tracks,
        p.barcode,
        p.mb_year,
        p.mb_title_no_disc,
        m.title_key_various AS mb_title_key_various
    FROM cds_various_candidate_pairs p
    LEFT JOIN mb_various_title_integration_view m
      ON p.release_id = m.release_id
""").fetchdf()

print(f"Candidate pairs loaded: {len(df):,}")

if df.empty:
    best_df = pd.DataFrame()
else:
    title_token_sims = []
    title_char_sims = []
    scores = []
    has_barcode = []

    for t1, k1, t2, k2, bc in zip(
        df["title_no_disc"],
        df["title_key_various"],
        df["mb_title_no_disc"],
        df["mb_title_key_various"],
        df["barcode"]
    ):
        t1 = "" if pd.isna(t1) else str(t1)
        k1 = "" if pd.isna(k1) else str(k1)
        t2 = "" if pd.isna(t2) else str(t2)
        k2 = "" if pd.isna(k2) else str(k2)

        token_sim = fuzz.token_set_ratio(t1, t2)
        char_sim = fuzz.ratio(k1, k2)

        # title-only score
        score = 0.7 * token_sim + 0.3 * char_sim

        title_token_sims.append(token_sim)
        title_char_sims.append(char_sim)
        scores.append(score)
        has_barcode.append(0 if pd.isna(bc) or str(bc).strip() == "" else 1)

    df["title_token_sim"] = title_token_sims
    df["title_char_sim"] = title_char_sims
    df["score"] = scores
    df["has_barcode"] = has_barcode

    # rank candidates per CDS row
    df = df.sort_values(
        ["cds_id", "score", "title_token_sim", "title_char_sim", "has_barcode", "release_id"],
        ascending=[True, False, False, False, False, True]
    ).copy()

    # second-best score per cds_id
    df["second_best_score"] = df.groupby("cds_id")["score"].shift(-1)
    df["score_gap"] = df["score"] - df["second_best_score"]

    # keep best row per cds_id
    best_df = df.groupby("cds_id", as_index=False).head(1).copy()

    # confidence rules
    def classify(row):
        if pd.isna(row["second_best_score"]):
            return "high"
        if row["score"] >= 98 and row["score_gap"] >= 2:
            return "high"
        if row["score"] >= 95 and row["score_gap"] >= 1:
            return "medium"
        return "low"

    best_df["various_confidence"] = best_df.apply(classify, axis=1)

con.register("best_various_df_view", best_df)

con.execute("""
    CREATE OR REPLACE TABLE cds_various_best_match AS
    SELECT * FROM best_various_df_view
""")

print("✅ Table created: cds_various_best_match")

print("\n--- Best Various Artists match count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_various_best_match
""").fetchdf())

print("\n--- Confidence distribution ---")
print(con.execute("""
    SELECT various_confidence, COUNT(*) AS n
    FROM cds_various_best_match
    GROUP BY various_confidence
    ORDER BY
        CASE various_confidence
            WHEN 'high' THEN 1
            WHEN 'medium' THEN 2
            ELSE 3
        END
""").fetchdf())

print("\n--- Preview best Various Artists matches ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        release_title,
        main_artist_name,
        score,
        title_token_sim,
        title_char_sim,
        second_best_score,
        score_gap,
        various_confidence
    FROM cds_various_best_match
    ORDER BY score DESC, cds_id
    LIMIT 25
""").fetchdf())

Connected ✅
Scoring cds_various_candidate_pairs and creating cds_various_best_match...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate pairs loaded: 3,817
✅ Table created: cds_various_best_match

--- Best Various Artists match count ---
     n
0  388

--- Confidence distribution ---
  various_confidence    n
0               high  184
1             medium   17
2                low  187

--- Preview best Various Artists matches ---
    cds_id           artist                                title  release_id                  release_title             main_artist_name  score  title_token_sim  title_char_sim  second_best_score  score_gap  \
0   100001          various          frankfurt trance vol 04 cd1      190028               Frankfurt Trance              Various Artists  100.0            100.0           100.0                NaN        NaN   
1   100010          various          frankfurt trance vol 05 cd1      190028               Frankfurt Trance              Various Artists  100.0            100.0           100.0                NaN        NaN   
2   100089  various artists                       dorival cay

In [298]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("\n--- Low-confidence Various Artists matches (187 rows) ---")
low_df = con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        mb_year,
        score,
        title_token_sim,
        title_char_sim,
        second_best_score,
        score_gap,
        various_confidence
    FROM cds_various_best_match
    WHERE various_confidence = 'low'
    ORDER BY score DESC, cds_id
    LIMIT 50
""").fetchdf()

low_df

Connected ✅

--- Low-confidence Various Artists matches (187 rows) ---


,cds_id,artist,title,cds_year,release_id,release_title,main_artist_name,mb_year,score,title_token_sim,title_char_sim,second_best_score,score_gap,various_confidence
0,100432,various,celtic heritage,2000,1186775,Celtic Heritage,Various Artists,2012,100.0,100.0,100.0,100.0,0.0,low
1,100600,various,children of the world,1994,167800,Children of the World,Bee Gees,2014,100.0,100.0,100.0,100.0,0.0,low
2,100773,various artists,celtic mystique,<NA>,497351,Celtic Mystique,Howard Baer,2010,100.0,100.0,100.0,100.0,0.0,low
3,100793,various,pipes drums of scotland,<NA>,2683017,Pipes & Drums of Scotland,Bun'ber E Pipe & Drum Corps,2020,100.0,100.0,100.0,100.0,0.0,low
4,100863,various artists,hed kandi back to love cd2,1999,2193551,Hed Kandi Back to Love,Various Artists,2018,100.0,100.0,100.0,100.0,0.0,low
5,100989,various,how low can you go,<NA>,2892115,How Low Can You Go,LP,2020,100.0,100.0,100.0,100.0,0.0,low
6,100996,various,jazz in the house 3 cd1,1996,122194,Jazz in the House 3,Various Artists,2009,100.0,100.0,100.0,100.0,0.0,low
7,101158,various,trumpet summit,<NA>,295332,Trumpet Summit,Allen Vizzutti,2022,100.0,100.0,100.0,100.0,0.0,low
8,101229,various artists,jazz moods cd 2,<NA>,82879,Jazz Moods,Various Artists,2023,100.0,100.0,100.0,100.0,0.0,low
9,101326,various,jazz for lovers,2000,512197,Jazz for Lovers,Various Artists,2021,100.0,100.0,100.0,100.0,0.0,low


In [339]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating cds_to_mb_final_mapping_v6...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v6 AS

    -- 1) Exact-stage matches
    SELECT
        x.cds_id,
        x.artist,
        x.title,
        x.cds_year,
        x.release_id,
        x.release_title,
        x.main_artist_name,
        x.artist_credit_full,
        x.total_tracks,
        x.barcode,
        x.mb_year,
        'exact' AS match_type,
        x.match_confidence AS confidence_label,
        x.candidate_score AS score,
        NULL::DOUBLE AS title_sim,
        NULL::DOUBLE AS artist_sim
    FROM cds_to_mb_best_exact_v4 x

    UNION ALL

    -- 2) Second-pass fuzzy matches
    SELECT
        f.cds_id,
        f.artist,
        f.title,
        f.cds_year,
        f.release_id,
        f.release_title,
        f.main_artist_name,
        f.artist_credit_full,
        f.total_tracks,
        f.barcode,
        f.mb_year,
        'fuzzy_second_pass' AS match_type,
        f.enrichment_confidence AS confidence_label,
        f.score,
        f.title_sim,
        f.artist_sim
    FROM cds_to_mb_best_enrichment f
    LEFT JOIN cds_to_mb_best_exact_v4 x
      ON x.cds_id = f.cds_id
    WHERE x.cds_id IS NULL
      AND f.enrichment_confidence = 'high'

    UNION ALL

    -- 3) Various Artists title-only matches
    SELECT
        v.cds_id,
        v.artist,
        v.title,
        v.cds_year,
        v.release_id,
        v.release_title,
        v.main_artist_name,
        v.artist_credit_full,
        v.total_tracks,
        v.barcode,
        v.mb_year,
        'various_title_only' AS match_type,
        v.various_confidence AS confidence_label,
        v.score,
        v.title_token_sim AS title_sim,
        NULL::DOUBLE AS artist_sim
    FROM cds_various_best_match v
    LEFT JOIN cds_to_mb_best_exact_v4 x
      ON x.cds_id = v.cds_id
    LEFT JOIN cds_to_mb_best_enrichment f
      ON f.cds_id = v.cds_id
         AND f.enrichment_confidence = 'high'
    WHERE x.cds_id IS NULL
      AND f.cds_id IS NULL
      AND v.various_confidence = 'high'
""")

print("✅ Table created: cds_to_mb_final_mapping_v6")

print("\n--- Coverage report ---")
print(con.execute("""
    WITH
    exact_cte AS (
        SELECT COUNT(*) AS n
        FROM cds_to_mb_best_exact_v4
    ),
    fuzzy_high_cte AS (
        SELECT COUNT(*) AS n
        FROM cds_to_mb_best_enrichment f
        LEFT JOIN cds_to_mb_best_exact_v4 x
          ON x.cds_id = f.cds_id
        WHERE x.cds_id IS NULL
          AND f.enrichment_confidence = 'high'
    ),
    various_high_cte AS (
        SELECT COUNT(*) AS n
        FROM cds_various_best_match v
        LEFT JOIN cds_to_mb_best_exact_v4 x
          ON x.cds_id = v.cds_id
        LEFT JOIN cds_to_mb_best_enrichment f
          ON f.cds_id = v.cds_id
             AND f.enrichment_confidence = 'high'
        WHERE x.cds_id IS NULL
          AND f.cds_id IS NULL
          AND v.various_confidence = 'high'
    ),
    final_cte AS (
        SELECT COUNT(*) AS n
        FROM cds_to_mb_final_mapping_v6
    )
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT n FROM exact_cte) AS exact_matched,
        (SELECT n FROM fuzzy_high_cte) AS fuzzy_high_added,
        (SELECT n FROM various_high_cte) AS various_high_added,
        (SELECT n FROM final_cte) AS total_matched,
        (SELECT COUNT(*) FROM cds) - (SELECT n FROM final_cte) AS still_unmatched
""").fetchdf())

print("\n--- Match type distribution ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v6
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Confidence distribution ---")
print(con.execute("""
    SELECT confidence_label, COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v6
    GROUP BY confidence_label
    ORDER BY n DESC
""").fetchdf())

print("\n--- Uniqueness check by cds_id ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cds_id) AS distinct_cds_id,
        COUNT(*) - COUNT(DISTINCT cds_id) AS duplicate_cds_id_rows
    FROM cds_to_mb_final_mapping_v6
""").fetchdf())

print("\n--- Preview final mapping ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        mb_year,
        match_type,
        confidence_label,
        score,
        title_sim,
        artist_sim
    FROM cds_to_mb_final_mapping_v6
    ORDER BY cds_id
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_to_mb_final_mapping_v6...
✅ Table created: cds_to_mb_final_mapping_v6

--- Coverage report ---
   cds_total  exact_matched  fuzzy_high_added  various_high_added  total_matched  still_unmatched
0       9763           3482               543                 183           4208             5555

--- Match type distribution ---
           match_type     n
0               exact  3482
1   fuzzy_second_pass   543
2  various_title_only   183

--- Confidence distribution ---
  confidence_label     n
0             high  2551
1              low  1578
2           medium    79

--- Uniqueness check by cds_id ---
   total_rows  distinct_cds_id  duplicate_cds_id_rows
0        4208             4208                      0

--- Preview final mapping ---
    cds_id                  artist                                title  cds_year  release_id                release_title       main_artist_name  mb_year         match_type confidence_label       score   title_sim  \
0     1818   

In [305]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Thesis report table:
# Final coverage with percentages after all current matching steps
# ------------------------------------------------------------
report_percentages = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,

        (SELECT COUNT(*) FROM cds_to_mb_best_exact_v4) AS exact_v4_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_to_mb_best_exact_v4)
            / (SELECT COUNT(*) FROM cds), 2
        ) AS exact_v4_pct,

        (SELECT COUNT(*) FROM cds_to_mb_best_enrichment) AS enrichment_fuzzy_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_to_mb_best_enrichment)
            / (SELECT COUNT(*) FROM cds), 2
        ) AS enrichment_fuzzy_pct,

        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6 WHERE match_type = 'various_title_integration') AS various_title_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6 WHERE match_type = 'various_title_integration')
            / (SELECT COUNT(*) FROM cds), 2
        ) AS various_title_pct,

        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS total_matched_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6)
            / (SELECT COUNT(*) FROM cds), 2
        ) AS total_matched_pct,

        (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS still_unmatched_n,
        ROUND(
            100.0 * (
                (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6)
            ) / (SELECT COUNT(*) FROM cds), 2
        ) AS still_unmatched_pct
""").fetchdf()

print("\n--- Thesis Report: Final Coverage with Percentages ---")
print(report_percentages)

Connected ✅

--- Thesis Report: Final Coverage with Percentages ---
   cds_total  exact_v4_n  exact_v4_pct  enrichment_fuzzy_n  enrichment_fuzzy_pct  various_title_n  various_title_pct  total_matched_n  total_matched_pct  still_unmatched_n  still_unmatched_pct
0       9763        3482         35.67                 565                  5.79                0                0.0             4208               43.1               5555                 56.9


In [306]:
import duckdb
import os

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Make sure exports folder exists
os.makedirs("../data_processed/exports", exist_ok=True)

# ------------------------------------------------------------
# Export new thesis/enrichment tables to CSV
# ------------------------------------------------------------
# Comments:
# - Only real tables are exported here.
# - Views are exported through SELECT * FROM view_name.
# - Existing files with the same name will be overwritten.
exports = [
    ("mb_integration_view", "../data_processed/exports/mb_integration_view.csv", "view"),
    ("mb_candidate_pool_v2", "../data_processed/exports/mb_candidate_pool_v2.csv", "table"),
    ("cds_mb_enrichment_pairs", "../data_processed/exports/cds_mb_enrichment_pairs.csv", "table"),
    ("cds_to_mb_enrichment_candidates", "../data_processed/exports/cds_to_mb_enrichment_candidates.csv", "table"),
    ("cds_to_mb_best_enrichment", "../data_processed/exports/cds_to_mb_best_enrichment.csv", "table"),

    ("cds_to_mb_final_mapping_v5", "../data_processed/exports/cds_to_mb_final_mapping_v5.csv", "table"),
    ("cds_to_mb_final_mapping_v6", "../data_processed/exports/cds_to_mb_final_mapping_v6.csv", "table"),

    ("cds_various_unmatched_cleaned", "../data_processed/exports/cds_various_unmatched_cleaned.csv", "table"),
    ("mb_various_title_integration_view", "../data_processed/exports/mb_various_title_integration_view.csv", "view"),
    ("cds_various_candidate_pairs", "../data_processed/exports/cds_various_candidate_pairs.csv", "table"),
    ("cds_various_best_match", "../data_processed/exports/cds_various_best_match.csv", "table"),
]

for obj_name, out_path, obj_type in exports:
    try:
        if obj_type == "table":
            con.execute(f"""
                COPY {obj_name}
                TO '{out_path}'
                WITH (HEADER, DELIMITER ',')
            """)
        else:
            con.execute(f"""
                COPY (SELECT * FROM {obj_name})
                TO '{out_path}'
                WITH (HEADER, DELIMITER ',')
            """)

        print(f"✅ Exported: {obj_name} -> {out_path}")

    except Exception as e:
        print(f"❌ Failed: {obj_name}")
        print("   ", type(e).__name__, e)

print("\nDone ✅")

Connected ✅


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: mb_integration_view -> ../data_processed/exports/mb_integration_view.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: mb_candidate_pool_v2 -> ../data_processed/exports/mb_candidate_pool_v2.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: cds_mb_enrichment_pairs -> ../data_processed/exports/cds_mb_enrichment_pairs.csv
✅ Exported: cds_to_mb_enrichment_candidates -> ../data_processed/exports/cds_to_mb_enrichment_candidates.csv
✅ Exported: cds_to_mb_best_enrichment -> ../data_processed/exports/cds_to_mb_best_enrichment.csv
✅ Exported: cds_to_mb_final_mapping_v5 -> ../data_processed/exports/cds_to_mb_final_mapping_v5.csv
✅ Exported: cds_to_mb_final_mapping_v6 -> ../data_processed/exports/cds_to_mb_final_mapping_v6.csv
✅ Exported: cds_various_unmatched_cleaned -> ../data_processed/exports/cds_various_unmatched_cleaned.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: mb_various_title_integration_view -> ../data_processed/exports/mb_various_title_integration_view.csv
✅ Exported: cds_various_candidate_pairs -> ../data_processed/exports/cds_various_candidate_pairs.csv
✅ Exported: cds_various_best_match -> ../data_processed/exports/cds_various_best_match.csv

Done ✅


In [309]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating cds_to_mb_enrichment_ranked...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_enrichment_ranked AS
    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS candidate_rank,

            FIRST_VALUE(score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS best_score,

            LEAD(score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS second_best_score

        FROM cds_to_mb_enrichment_candidates
    )
    SELECT
        *,
        CASE
            WHEN candidate_rank = 1 THEN (best_score - second_best_score)
            ELSE NULL
        END AS score_gap,

        CASE
            WHEN candidate_rank = 1
                 AND score >= 92
                 AND title_sim >= 94
                 AND ((best_score - second_best_score) >= 8 OR second_best_score IS NULL)
            THEN 'auto_accept'

            WHEN candidate_rank = 1
                 AND score >= 88
                 AND title_sim >= 90
            THEN 'review'

            ELSE 'reject'
        END AS confidence_class
    FROM ranked
""")

print("✅ Table created: cds_to_mb_enrichment_ranked")

print("\n--- Top-ranked enrichment candidates only ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        main_artist_name,
        release_title,
        score,
        artist_sim,
        title_sim,
        best_score,
        second_best_score,
        score_gap,
        confidence_class
    FROM cds_to_mb_enrichment_ranked
    WHERE candidate_rank = 1
    ORDER BY score DESC, score_gap DESC NULLS LAST
    LIMIT 30
""").fetchdf())

print("\n--- Confidence class summary (top-ranked only) ---")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*) AS n
    FROM cds_to_mb_enrichment_ranked
    WHERE candidate_rank = 1
    GROUP BY confidence_class
    ORDER BY n DESC
""").fetchdf())

Connected ✅
Creating cds_to_mb_enrichment_ranked...
✅ Table created: cds_to_mb_enrichment_ranked

--- Top-ranked enrichment candidates only ---
    cds_id                    artist                                      title  release_id          main_artist_name                                 release_title  score  artist_sim  title_sim  best_score  \
0   102089               101 strings              unforgettable melodies disc 1     1893295               101 Strings                        Unforgettable Melodies  100.0       100.0      100.0       100.0   
1   101970                 tom waits             jockey full of bourbon disc 1      4816131                 Tom Waits                        Jockey Full of Bourbon  100.0       100.0      100.0       100.0   
2   107708             vernon oxford                 keeper of the flame disc 4      863661             Vernon Oxford                           Keeper of the Flame  100.0       100.0      100.0       100.0   
3   100556          

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating mb_release_enriched_from_cds_v1...")

con.execute("""
    CREATE OR REPLACE TABLE mb_release_enriched_from_cds_v1 AS
    WITH ranked AS (
        SELECT
            m.*,

            ROW_NUMBER() OVER (
                PARTITION BY m.release_id
                ORDER BY
                    CASE m.match_type
                        WHEN 'exact' THEN 1
                        WHEN 'fuzzy_second_pass' THEN 2
                        WHEN 'various_title_only' THEN 3
                        ELSE 9
                    END ASC,
                    CASE m.confidence_label
                        WHEN 'high' THEN 1
                        WHEN 'auto_accept' THEN 1
                        WHEN 'medium' THEN 2
                        WHEN 'review' THEN 2
                        WHEN 'low' THEN 3
                        WHEN 'reject' THEN 4
                        ELSE 9
                    END ASC,
                    COALESCE(m.score, -1) DESC,
                    m.cds_id ASC
            ) AS rn,

            COUNT(*) OVER (PARTITION BY m.release_id) AS cds_links_to_release
        FROM cds_to_mb_final_mapping_v6 m
    )
    SELECT
        -- MB anchor
        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.mb_year,
        r.barcode,
        r.total_tracks,

        -- CDS enrichment attributes
        r.cds_id,
        r.cds_year,

        -- Pull CDS metadata directly from the CDS table
        c.category AS cds_category,
        c.genre AS cds_genre,
        c.tracks AS cds_tracklist,

        -- Provenance / integration metadata
        r.match_type,
        r.confidence_label,
        r.score,
        r.title_sim,
        r.artist_sim,
        r.cds_links_to_release,
        'CDS' AS enrichment_source

    FROM ranked r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
    WHERE r.rn = 1
""")

print("✅ Table created: mb_release_enriched_from_cds_v1")

print("\n--- Row count / uniqueness check ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT release_id) AS distinct_release_id,
        COUNT(*) - COUNT(DISTINCT release_id) AS duplicate_release_rows
    FROM mb_release_enriched_from_cds_v1
""").fetchdf())

print("\n--- Enrichment coverage report ---")
print(con.execute("""
    WITH mb_total AS (
        SELECT COUNT(*) AS n
        FROM mb_release_features
    ),
    enriched_total AS (
        SELECT COUNT(*) AS n
        FROM mb_release_enriched_from_cds_v1
    ),
    enriched_with_new_metadata AS (
        SELECT COUNT(*) AS n
        FROM mb_release_enriched_from_cds_v1
        WHERE cds_category IS NOT NULL
           OR cds_genre IS NOT NULL
           OR cds_year IS NOT NULL
    )
    SELECT
        (SELECT n FROM mb_total) AS mb_total_releases,
        (SELECT n FROM enriched_total) AS mb_releases_enriched,
        (SELECT n FROM enriched_with_new_metadata) AS mb_releases_with_new_metadata,
        ROUND(
            100.0 * (SELECT n FROM enriched_with_new_metadata) / (SELECT n FROM mb_total),
            4
        ) AS pct_of_mb_with_new_metadata
""").fetchdf())

print("\n--- Match type distribution in final MB enrichment table ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM mb_release_enriched_from_cds_v1
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Confidence distribution in final MB enrichment table ---")
print(con.execute("""
    SELECT confidence_label, COUNT(*) AS n
    FROM mb_release_enriched_from_cds_v1
    GROUP BY confidence_label
    ORDER BY n DESC
""").fetchdf())

print("\n--- Releases with multiple CDS links before consolidation ---")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        cds_links_to_release,
        match_type,
        confidence_label,
        score,
        cds_id,
        cds_category,
        cds_genre
    FROM mb_release_enriched_from_cds_v1
    WHERE cds_links_to_release > 1
    ORDER BY cds_links_to_release DESC, COALESCE(score, -1) DESC, release_id
    LIMIT 20
""").fetchdf())

print("\n--- Preview final MusicBrainz-centered enrichment table ---")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        mb_year,
        barcode,
        total_tracks,
        cds_id,
        cds_year,
        cds_category,
        cds_genre,
        match_type,
        confidence_label,
        score,
        enrichment_source
    FROM mb_release_enriched_from_cds_v1
    LIMIT 25
""").fetchdf())

print("\nExporting CSV...")
con.execute("""
    COPY mb_release_enriched_from_cds_v1
    TO '../data_processed/exports/mb_release_enriched_from_cds_v1.csv'
    WITH (HEADER, DELIMITER ',')
""")
print("✅ CSV saved: ../data_processed/exports/mb_release_enriched_from_cds_v1.csv")

Connected ✅
Creating mb_release_enriched_from_cds_v1...


In [329]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Create a new automatic enrichment table using only auto_accept rows
# ------------------------------------------------------------
# Comments:
# - We keep only candidate_rank = 1 and confidence_class = 'auto_accept'.
# - This is the ambiguity-controlled automatic enrichment subset.
print("Creating cds_to_mb_best_enrichment_auto...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_enrichment_auto AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,
        score,
        artist_sim,
        title_sim,
        best_score,
        second_best_score,
        score_gap,
        confidence_class
    FROM cds_to_mb_enrichment_ranked
    WHERE candidate_rank = 1
      AND confidence_class = 'auto_accept'
""")

print("✅ Table created: cds_to_mb_best_enrichment_auto")

print("\n--- Count of auto-accepted enrichment matches ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_best_enrichment_auto
""").fetchdf())

# ------------------------------------------------------------
# Merge auto-accepted enrichment matches into a new final mapping version
# ------------------------------------------------------------
# Comments:
# - Keep exact_v4 matches
# - Add only ambiguity-controlled auto-accepted enrichment matches
# - Do not duplicate cds_ids already matched by exact_v4
print("\nCreating cds_to_mb_final_mapping_v7_auto...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v7_auto AS
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,
        'exact_v4' AS match_type,
        NULL::DOUBLE AS score
    FROM cds_to_mb_best_exact_v4

    UNION ALL

    SELECT
        e.cds_id,
        e.artist,
        e.title,
        e.release_id,
        e.release_title,
        e.main_artist_name,
        e.artist_credit_full,
        e.total_tracks,
        e.barcode,
        e.mb_year,
        'enrichment_fuzzy_auto' AS match_type,
        e.score
    FROM cds_to_mb_best_enrichment_auto e
    LEFT JOIN cds_to_mb_best_exact_v4 x
      ON x.cds_id = e.cds_id
    WHERE x.cds_id IS NULL
""")

print("✅ Table created: cds_to_mb_final_mapping_v7_auto")

# ------------------------------------------------------------
# Coverage report for the ambiguity-controlled automatic version
# ------------------------------------------------------------
report_auto = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(*) FROM cds_to_mb_best_exact_v4) AS exact_v4_matched,
        (SELECT COUNT(*) FROM cds_to_mb_best_enrichment_auto) AS enrichment_auto_added,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto) AS total_matched_v7_auto,
        (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto) AS still_unmatched_v7_auto
""").fetchdf()

print("\n--- Thesis Report: Ambiguity-Controlled Automatic Coverage ---")
print(report_auto)

print("\n--- Match type breakdown in v7_auto ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v7_auto
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

Connected ✅
Creating cds_to_mb_best_enrichment_auto...
✅ Table created: cds_to_mb_best_enrichment_auto

--- Count of auto-accepted enrichment matches ---
     n
0  238

Creating cds_to_mb_final_mapping_v7_auto...
✅ Table created: cds_to_mb_final_mapping_v7_auto

--- Thesis Report: Ambiguity-Controlled Automatic Coverage ---
   cds_total  exact_v4_matched  enrichment_auto_added  total_matched_v7_auto  still_unmatched_v7_auto
0       9763              3482                    238                   3720                     6043

--- Match type breakdown in v7_auto ---
              match_type     n
0               exact_v4  3482
1  enrichment_fuzzy_auto   238


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

comparison = con.execute("""
    SELECT 'v6_broader_mapping' AS version,
           (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS matched_rows,
           ROUND(100.0 * (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) / (SELECT COUNT(*) FROM cds), 2) AS matched_pct,
           (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS unmatched_rows,
           ROUND(100.0 * ((SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6)) / (SELECT COUNT(*) FROM cds), 2) AS unmatched_pct

    UNION ALL

    SELECT 'v7_auto_high_confidence' AS version,
           (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto) AS matched_rows,
           ROUND(100.0 * (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto) / (SELECT COUNT(*) FROM cds), 2) AS matched_pct,
           (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto) AS unmatched_rows,
           ROUND(100.0 * ((SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v7_auto)) / (SELECT COUNT(*) FROM cds), 2) AS unmatched_pct
""").fetchdf()

print("\n--- Thesis Report: Comparison of Final Mapping Variants ---")
print(comparison)

In [331]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Create the unmatched CDS remainder after v6
# ------------------------------------------------------------
# Comments:
# - v6 is the broader working mapping.
# - We now isolate the CDS rows that still remain unmatched after v6.
# - This will be the input for the next cleaning and enrichment cycle.
print("Creating cds_unmatched_after_v6...")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_after_v6 AS
    SELECT c.*
    FROM cds_norm c
    LEFT JOIN cds_to_mb_final_mapping_v6 m
      ON c.cds_id = m.cds_id
    WHERE m.cds_id IS NULL
""")

print("✅ Table created: cds_unmatched_after_v6")

print("\n--- Count of unmatched CDS rows after v6 ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_unmatched_after_v6
""").fetchdf())

print("\n--- Preview: unmatched CDS rows after v6 ---")
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year, artist_key, title_key
    FROM cds_unmatched_after_v6
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_unmatched_after_v6...
✅ Table created: cds_unmatched_after_v6

--- Count of unmatched CDS rows after v6 ---
      n
0  5555

--- Preview: unmatched CDS rows after v6 ---
    cds_id                        artist                                                                         title  cds_year                artist_key  \
0   100003       ã â â ã â ã â ºâ ã â å                                          ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ       1989                             
1   100004                       emanuel                                                                    felicidade      1998                   emanuel   
2   100005   ã å â ã â ºâ ã â ºâ ºã â â     catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½â ã â šâ ã æ â ã â šâ¹ã æ ë       <NA>                             
3   100007               various artists                                                                     nop pop 2      2002            variousartists   
4   100009     

In [332]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Creating cds_unmatched_profiled_v6...")

con.execute(r"""
    CREATE OR REPLACE TABLE cds_unmatched_profiled_v6 AS
    SELECT
        *,

        CASE WHEN artist IS NULL OR trim(artist) = '' THEN 1 ELSE 0 END AS missing_artist,
        CASE WHEN title  IS NULL OR trim(title)  = '' THEN 1 ELSE 0 END AS missing_title,
        CASE WHEN tracks IS NULL OR trim(tracks) = '' THEN 1 ELSE 0 END AS missing_tracks,
        CASE WHEN cds_year IS NULL THEN 1 ELSE 0 END AS missing_year,

        CASE WHEN length(trim(coalesce(artist, ''))) < 4 THEN 1 ELSE 0 END AS short_artist,
        CASE WHEN length(trim(coalesce(title,  ''))) < 4 THEN 1 ELSE 0 END AS short_title,

        CASE
            WHEN regexp_matches(coalesce(artist, ''), 'Ã|â|Â|�')
              OR regexp_matches(coalesce(title,  ''), 'Ã|â|Â|�')
              OR regexp_matches(coalesce(tracks, ''), 'Ã|â|Â|�')
            THEN 1 ELSE 0
        END AS is_mojibake,

        CASE
            WHEN lower(trim(coalesce(artist, ''))) IN (
                'various', 'various artists', 'various artist',
                'va', 'v a', 'v.a.',
                'varios', 'varios artistas', 'vàrios'
            )
            THEN 1 ELSE 0
        END AS is_various_artists,

        CASE
            WHEN regexp_matches(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+')
            THEN 1 ELSE 0
        END AS has_disc_cd_vol,

        CASE
            WHEN lower(trim(coalesce(title, ''))) IN (
                'live', 'greatest hits', 'best of', 'the best of',
                'hits', 'soundtrack', 'mix', 'mixes', 'now'
            )
            THEN 1 ELSE 0
        END AS is_generic_title,

        trim(regexp_replace(lower(coalesce(artist, '')), '\s+', ' ', 'g')) AS artist_clean,
        trim(regexp_replace(lower(coalesce(title,  '')), '\s+', ' ', 'g')) AS title_clean,

        trim(regexp_replace(
            regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
            '\s+', ' ', 'g'
        )) AS title_no_disc,

        lower(regexp_replace(coalesce(artist, ''), '[^a-zA-Z0-9]+', '', 'g')) AS artist_key_clean,

        lower(regexp_replace(
            regexp_replace(
                regexp_replace(lower(coalesce(title, '')), '(disc|disk|cd|vol|volume)\s*0*\d+', '', 'g'),
                '\b(remastered|remaster|deluxe|edition|bonus|expanded|reissue|live)\b', '', 'g'
            ),
            '[^a-zA-Z0-9]+', '', 'g'
        )) AS title_key_clean

    FROM cds_unmatched_after_v6
""")

print("✅ Table created: cds_unmatched_profiled_v6")

print("\n--- Profiling summary for unmatched CDS after v6 ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_unmatched,
        SUM(missing_artist) AS missing_artist,
        SUM(missing_title) AS missing_title,
        SUM(missing_tracks) AS missing_tracks,
        SUM(missing_year) AS missing_year,
        SUM(short_artist) AS short_artist,
        SUM(short_title) AS short_title,
        SUM(is_mojibake) AS mojibake_rows,
        SUM(is_various_artists) AS various_artist_rows,
        SUM(has_disc_cd_vol) AS disc_cd_vol_rows,
        SUM(is_generic_title) AS generic_title_rows
    FROM cds_unmatched_profiled_v6
""").fetchdf())

Connected ✅
Creating cds_unmatched_profiled_v6...
✅ Table created: cds_unmatched_profiled_v6

--- Profiling summary for unmatched CDS after v6 ---
   total_unmatched  missing_artist  missing_title  missing_tracks  missing_year  short_artist  short_title  mojibake_rows  various_artist_rows  disc_cd_vol_rows  generic_title_rows
0             5555             7.0           11.0             1.0        2670.0         171.0        114.0          509.0                959.0             972.0                35.0


In [333]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Create a cleaned working subset from the v6 unmatched remainder
# ------------------------------------------------------------
# Comments:
# - We build a reusable working table for the next enrichment cycle.
# - We assign priority classes:
#     high   = relatively clean and promising
#     medium = recoverable but needs special handling
#     low    = noisy / risky / weak
print("Creating cds_unmatched_working_v6...")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_working_v6 AS
    SELECT
        *,

        CASE
            WHEN missing_artist = 1 OR missing_title = 1 THEN 'low'
            WHEN is_mojibake = 1 THEN 'low'
            WHEN short_artist = 1 OR short_title = 1 THEN 'low'
            WHEN is_generic_title = 1 THEN 'medium'
            WHEN is_various_artists = 1 THEN 'medium'
            WHEN has_disc_cd_vol = 1 THEN 'medium'
            ELSE 'high'
        END AS enrichment_priority_v6

    FROM cds_unmatched_profiled_v6
""")

print("✅ Table created: cds_unmatched_working_v6")

print("\n--- Priority distribution in v6 remainder ---")
print(con.execute("""
    SELECT enrichment_priority_v6, COUNT(*) AS n
    FROM cds_unmatched_working_v6
    GROUP BY enrichment_priority_v6
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: working unmatched rows after v6 ---")
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year,
        is_mojibake, is_various_artists, has_disc_cd_vol, is_generic_title,
        enrichment_priority_v6
    FROM cds_unmatched_working_v6
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_unmatched_working_v6...
✅ Table created: cds_unmatched_working_v6

--- Priority distribution in v6 remainder ---
  enrichment_priority_v6     n
0                   high  3348
1                 medium  1459
2                    low   748

--- Preview: working unmatched rows after v6 ---
    cds_id                        artist                                                                         title  cds_year  is_mojibake  is_various_artists  has_disc_cd_vol  is_generic_title  \
0   100003       ã â â ã â ã â ºâ ã â å                                          ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ       1989            1                   0                0                 0   
1   100004                       emanuel                                                                    felicidade      1998            0                   0                0                 0   
2   100005   ã å â ã â ºâ ã â ºâ ºã â â     catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã 

In [383]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build the next cleaned enrichment subset from the v6 remainder
# ------------------------------------------------------------
# Comments:
# - Keep all high-priority rows
# - Also keep medium-priority rows that are not mojibake and not Various Artists
# - This gives us a cleaner subset for the next standard enrichment pass
# - Separate pipelines can later handle mojibake and Various Artists again
print("Creating cds_unmatched_for_enrichment_v6_round2...")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_for_enrichment_v6_round2 AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        artist_key,
        title_key,

        artist_clean,
        title_clean,
        title_no_disc,
        artist_key_clean,
        title_key_clean,

        is_mojibake,
        is_various_artists,
        has_disc_cd_vol,
        is_generic_title,
        enrichment_priority_v6

    FROM cds_unmatched_working_v6
    WHERE
        enrichment_priority_v6 = 'high'
        OR (
            enrichment_priority_v6 = 'medium'
            AND is_mojibake = 0
            AND is_various_artists = 0
        )
""")

print("✅ Table created: cds_unmatched_for_enrichment_v6_round2")

print("\n--- Count of rows selected for round-2 enrichment ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_unmatched_for_enrichment_v6_round2
""").fetchdf())

print("\n--- Breakdown by priority in round-2 subset ---")
print(con.execute("""
    SELECT enrichment_priority_v6, COUNT(*) AS n
    FROM cds_unmatched_for_enrichment_v6_round2
    GROUP BY enrichment_priority_v6
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: round-2 enrichment subset ---")
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year,
        is_mojibake, is_various_artists, has_disc_cd_vol, is_generic_title,
        enrichment_priority_v6
    FROM cds_unmatched_for_enrichment_v6_round2
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_unmatched_for_enrichment_v6_round2...
✅ Table created: cds_unmatched_for_enrichment_v6_round2

--- Count of rows selected for round-2 enrichment ---
      n
0  3931

--- Breakdown by priority in round-2 subset ---
  enrichment_priority_v6     n
0                   high  3348
1                 medium   583

--- Preview: round-2 enrichment subset ---
    cds_id                        artist                                                                         title  cds_year  is_mojibake  is_various_artists  has_disc_cd_vol  is_generic_title  \
0   100004                       emanuel                                                                    felicidade      1998            0                   0                0                 0   
1   100009                        4 p m                                                                     sweet soul      2001            0                   0                0                 0   
2   100025  variousce it

In [335]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build round-2 candidate pairs from the v6 cleaned enrichment subset
# ------------------------------------------------------------
# Comments:
# - We compare the cleaned CDS remainder against the cleaned MB pool.
# - Blocking is based on:
#     * first letter of artist_key_clean
#     * first letter of title_key_clean
#     * similar key lengths
# - This is the input for the next fuzzy scoring round.
print("Creating cds_mb_enrichment_pairs_v6_round2...")

con.execute("""
    CREATE OR REPLACE TABLE cds_mb_enrichment_pairs_v6_round2 AS
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.tracks,
        c.enrichment_priority_v6,

        c.artist_key_clean AS cds_artist_key,
        c.title_key_clean  AS cds_title_key,

        m.release_id,
        m.main_artist_name,
        m.release_title,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        m.artist_key_clean AS mb_artist_key,
        m.title_key_clean  AS mb_title_key

    FROM (
        SELECT
            *,
            substr(artist_key_clean, 1, 1) AS artist_first,
            substr(title_key_clean,  1, 1) AS title_first,
            length(artist_key_clean) AS artist_len,
            length(title_key_clean)  AS title_len
        FROM cds_unmatched_for_enrichment_v6_round2
        WHERE
            artist_key_clean IS NOT NULL
            AND title_key_clean IS NOT NULL
            AND length(artist_key_clean) >= 4
            AND length(title_key_clean)  >= 4
    ) c

    JOIN mb_candidate_pool_v2 m
      ON m.artist_first = c.artist_first
     AND m.title_first  = c.title_first
     AND m.artist_len BETWEEN c.artist_len - 3 AND c.artist_len + 3
     AND m.title_len  BETWEEN c.title_len  - 5 AND c.title_len  + 5
""")

print("✅ Table created: cds_mb_enrichment_pairs_v6_round2")

print("\n--- Pair count for round-2 enrichment ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_enrichment_pairs_v6_round2
""").fetchdf())

print("\n--- Distinct CDS rows with candidate pairs in round 2 ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_candidates
    FROM cds_mb_enrichment_pairs_v6_round2
""").fetchdf())

print("\n--- Preview: round-2 candidate pairs ---")
print(con.execute("""
    SELECT
        cds_id, artist, title,
        release_id, main_artist_name, release_title,
        cds_year, mb_year, total_tracks, barcode
    FROM cds_mb_enrichment_pairs_v6_round2
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_mb_enrichment_pairs_v6_round2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_mb_enrichment_pairs_v6_round2

--- Pair count for round-2 enrichment ---
         n
0  9818133

--- Distinct CDS rows with candidate pairs in round 2 ---
   cds_rows_with_candidates
0                      3707

--- Preview: round-2 candidate pairs ---
    cds_id                           artist                                    title  release_id               main_artist_name                                     release_title  cds_year  mb_year  total_tracks  \
0     3123                philosopher kings                    the philosopher kings     1468481                Parasite Single                                The Hunt (Remixed)      <NA>     2014           2.0   
1   103412                 eisuke mochiduki                           waitin for you     1664976            Enrico Rava Quartet                                        Wild Dance      <NA>     2015          14.0   
2   107234  gavin friday and the man seezer        each man kills the thing he loves 

In [47]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Score round-2 candidate pairs in chunks
# ------------------------------------------------------------
# Comments:
# - We score CDS vs MB using cleaned keys:
#     0.4 * artist similarity + 0.6 * title similarity
# - We process in chunks because the pair table is large.
# - Per chunk, we keep only the top few candidates per CDS row.
print("Preparing indexed view for round-2 scoring...")

con.execute("""
    CREATE OR REPLACE VIEW cds_mb_enrichment_pairs_v6_round2_indexed AS
    SELECT
        row_number() OVER () AS rid,
        *
    FROM cds_mb_enrichment_pairs_v6_round2
""")

total_pairs = con.execute("""
    SELECT COUNT(*) FROM cds_mb_enrichment_pairs_v6_round2
""").fetchone()[0]

print(f"Total round-2 pairs to score: {total_pairs:,}")

CHUNK_SIZE = 250_000
KEEP_PER_CHUNK = 3
all_best = []

print("Scoring round-2 candidate pairs in chunks...")

for start in range(1, total_pairs + 1, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE - 1, total_pairs)
    print(f"  Processing rid {start:,} .. {end:,}")

    df = con.execute("""
        SELECT
            cds_id,
            artist,
            title,
            cds_year,
            enrichment_priority_v6,
            cds_artist_key,
            cds_title_key,
            release_id,
            main_artist_name,
            release_title,
            artist_credit_full,
            total_tracks,
            barcode,
            mb_year,
            mb_artist_key,
            mb_title_key
        FROM cds_mb_enrichment_pairs_v6_round2_indexed
        WHERE rid BETWEEN ? AND ?
    """, [start, end]).fetchdf()

    if df.empty:
        continue

    artist_sims = []
    title_sims = []
    scores = []

    for a1, t1, a2, t2 in zip(df["cds_artist_key"], df["cds_title_key"], df["mb_artist_key"], df["mb_title_key"]):
        a_sim = fuzz.token_set_ratio(a1, a2)
        t_sim = fuzz.token_set_ratio(t1, t2)
        score = 0.4 * a_sim + 0.6 * t_sim
        artist_sims.append(a_sim)
        title_sims.append(t_sim)
        scores.append(score)

    df["artist_sim"] = artist_sims
    df["title_sim"] = title_sims
    df["score"] = scores

    # Keep only the best few candidates per CDS row in this chunk
    df = (
        df.sort_values(["cds_id", "score"], ascending=[True, False])
          .groupby("cds_id")
          .head(KEEP_PER_CHUNK)
    )

    all_best.append(df)

best_df = pd.concat(all_best, ignore_index=True) if all_best else pd.DataFrame()

print("Combined candidates after chunk pruning:", len(best_df))

# Keep final top 5 per CDS row across all chunks
TOP_K = 5
best_df = (
    best_df.sort_values(["cds_id", "score"], ascending=[True, False])
           .groupby("cds_id")
           .head(TOP_K)
)

print("Final top candidates:", len(best_df))

con.register("best_df_round2", best_df)

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_enrichment_candidates_v6_round2 AS
    SELECT *
    FROM best_df_round2
""")

print("✅ Table created: cds_to_mb_enrichment_candidates_v6_round2")

print("\n--- Preview: top round-2 enrichment candidates ---")
print(con.execute("""
    SELECT
        cds_id, artist, title,
        release_id, main_artist_name, release_title,
        score, artist_sim, title_sim,
        enrichment_priority_v6
    FROM cds_to_mb_enrichment_candidates_v6_round2
    ORDER BY score DESC
    LIMIT 25
""").fetchdf())

Connected ✅
Preparing indexed view for round-2 scoring...
Total round-2 pairs to score: 9,898,596
Scoring round-2 candidate pairs in chunks...
  Processing rid 1 .. 250,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing rid 250,001 .. 500,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing rid 500,001 .. 750,000


...

  Processing rid 9,500,001 .. 9,750,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing rid 9,750,001 .. 9,898,596


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Combined candidates after chunk pruning: 392886
Final top candidates: 18189
✅ Table created: cds_to_mb_enrichment_candidates_v6_round2

--- Preview: top round-2 enrichment candidates ---
    cds_id                artist                                 title  release_id      main_artist_name                 release_title  score  artist_sim  title_sim enrichment_priority_v6
0     4348           frank zappa                  joe s garage disk 1      4387319           Frank Zappa                  Joe’s Garage  100.0       100.0      100.0                 medium
1   105572                 cream                wheels of fire disk 1       454410                 Cream                Wheels of Fire  100.0       100.0      100.0                 medium
2   101998           abbacadabra                     abbasalute disk 1     2748831           Abbacadabra                    Abbasalute  100.0       100.0      100.0                 medium
3   102027           jules verne  20 000 leagues under the se

In [365]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build ambiguity-aware ranking for round-2 enrichment candidates
# ------------------------------------------------------------
# Comments:
# - Input: cds_to_mb_enrichment_candidates_v6_round2
# - For each cds_id:
#     * rank candidates
#     * compute best_score
#     * compute second_best_score
#     * compute score_gap
# - Then assign confidence classes:
#     * auto_accept
#     * review
#     * reject
print("Creating cds_to_mb_enrichment_ranked_v6_round2...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_enrichment_ranked_v6_round2 AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS candidate_rank,
            

            first_value(score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS best_score,

            lead(score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    score DESC,
                    title_sim DESC,
                    artist_sim DESC,
                    release_id ASC
            ) AS second_best_score
        FROM cds_to_mb_enrichment_candidates_v6_round2
    )
    SELECT
        *,
        (best_score - second_best_score) AS score_gap,

        CASE
            WHEN candidate_rank = 1
                 AND score >= 92
                 AND title_sim >= 94
                 AND (best_score - second_best_score >= 8 OR second_best_score IS NULL)
            THEN 'auto_accept'

            WHEN candidate_rank = 1
                 AND score >= 88
                 AND title_sim >= 90
            THEN 'review'

            ELSE 'reject'
        END AS confidence_class

    FROM ranked
""")

print("✅ Table created: cds_to_mb_enrichment_ranked_v6_round2")

print("\n--- Confidence class summary (top-ranked only, round 2) ---")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*) AS n
    FROM cds_to_mb_enrichment_ranked_v6_round2
    WHERE candidate_rank = 1
    GROUP BY confidence_class
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: top-ranked round-2 candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        main_artist_name,
        release_title,
        score,
        artist_sim,
        title_sim,
        best_score,
        second_best_score,
        score_gap,
        confidence_class,
        enrichment_priority_v6
    FROM cds_to_mb_enrichment_ranked_v6_round2
    WHERE candidate_rank = 1
    ORDER BY score DESC, score_gap DESC NULLS LAST
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_to_mb_enrichment_ranked_v6_round2...
✅ Table created: cds_to_mb_enrichment_ranked_v6_round2

--- Confidence class summary (top-ranked only, round 2) ---
  confidence_class     n
0           reject  3611
1           review    64
2      auto_accept     5

--- Preview: top-ranked round-2 candidates ---
    cds_id                artist                                   title  release_id      main_artist_name                      release_title       score  artist_sim   title_sim  best_score  second_best_score  \
0   104819              hawkwind            golden void 1969 1979 disk 1      298862              Hawkwind              Golden Void 1969–1979  100.000000  100.000000  100.000000  100.000000          61.176471   
1   102968           randy greif             alice in wonderland disk 1       250197           Randy Greif                Alice in Wonderland  100.000000  100.000000  100.000000  100.000000          61.443932   
2   105837                prince      

In [344]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Create automatic round-2 enrichment table
# ------------------------------------------------------------
print("Creating cds_to_mb_best_enrichment_v6_round2_auto...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_best_enrichment_v6_round2_auto AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,
        score,
        artist_sim,
        title_sim,
        best_score,
        second_best_score,
        score_gap,
        confidence_class,
        enrichment_priority_v6
    FROM cds_to_mb_enrichment_ranked_v6_round2
    WHERE candidate_rank = 1
      AND confidence_class = 'auto_accept'
""")

print("✅ Table created: cds_to_mb_best_enrichment_v6_round2_auto")

print("\n--- Count of round-2 auto-accepted matches ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_best_enrichment_v6_round2_auto
""").fetchdf())

# ------------------------------------------------------------
# Step 2: Merge round-2 auto matches into a new final mapping
# ------------------------------------------------------------
print("\nCreating cds_to_mb_final_mapping_v8...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v8 AS

    -- Existing v6 mapping
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,
        match_type,
        confidence_label,
        score,
        title_sim,
        artist_sim
    FROM cds_to_mb_final_mapping_v6

    UNION ALL

    -- New round-2 auto-accepted matches
    SELECT
        e.cds_id,
        e.artist,
        e.title,
        e.cds_year,
        e.release_id,
        e.release_title,
        e.main_artist_name,
        e.artist_credit_full,
        e.total_tracks,
        e.barcode,
        e.mb_year,
        'enrichment_round2_auto' AS match_type,
        e.confidence_class AS confidence_label,
        e.score,
        e.title_sim,
        e.artist_sim
    FROM cds_to_mb_best_enrichment_v6_round2_auto e
    LEFT JOIN cds_to_mb_final_mapping_v6 m
      ON m.cds_id = e.cds_id
    WHERE m.cds_id IS NULL
""")

print("✅ Table created: cds_to_mb_final_mapping_v8")

# ------------------------------------------------------------
# Step 3: Coverage report after round-2 automatic enrichment
# ------------------------------------------------------------
report_v8 = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS matched_before_round2,
        (SELECT COUNT(*) FROM cds_to_mb_best_enrichment_v6_round2_auto) AS round2_auto_candidates,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v8) AS matched_after_round2,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v8)
          - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v6) AS newly_added_by_round2,
        (SELECT COUNT(*) FROM cds)
          - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v8) AS still_unmatched_after_round2
""").fetchdf()

print("\n--- Thesis Report: Coverage After Round-2 Automatic Enrichment ---")
print(report_v8)

print("\n--- Match type breakdown in v8 ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v8
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Uniqueness check by cds_id ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cds_id) AS distinct_cds_id,
        COUNT(*) - COUNT(DISTINCT cds_id) AS duplicate_cds_id_rows
    FROM cds_to_mb_final_mapping_v8
""").fetchdf())

Connected ✅
Creating cds_to_mb_best_enrichment_v6_round2_auto...
✅ Table created: cds_to_mb_best_enrichment_v6_round2_auto

--- Count of round-2 auto-accepted matches ---
   n
0  5

Creating cds_to_mb_final_mapping_v8...
✅ Table created: cds_to_mb_final_mapping_v8

--- Thesis Report: Coverage After Round-2 Automatic Enrichment ---
   cds_total  matched_before_round2  round2_auto_candidates  matched_after_round2  newly_added_by_round2  still_unmatched_after_round2
0       9763                   4208                       5                  4211                      3                          5552

--- Match type breakdown in v8 ---
               match_type     n
0                   exact  3482
1       fuzzy_second_pass   543
2      various_title_only   183
3  enrichment_round2_auto     3

--- Uniqueness check by cds_id ---
   total_rows  distinct_cds_id  duplicate_cds_id_rows
0        4211             4211                      0


In [345]:
import duckdb
import os
import pandas as pd

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

# ------------------------------------------------------------
# Export round-2 review candidates for manual inspection
# ------------------------------------------------------------
# Comments:
# - These are the top-ranked round-2 candidates classified as "review"
# - They are not safe enough for auto-accept
# - But they are the best manual-review set for the thesis
print("Creating cds_to_mb_review_candidates_v6_round2...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_review_candidates_v6_round2 AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        release_id,
        main_artist_name,
        release_title,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,
        score,
        artist_sim,
        title_sim,
        best_score,
        second_best_score,
        score_gap,
        confidence_class,
        enrichment_priority_v6
    FROM cds_to_mb_enrichment_ranked_v6_round2
    WHERE candidate_rank = 1
      AND confidence_class = 'review'
""")

print("✅ Table created: cds_to_mb_review_candidates_v6_round2")

print("\n--- Count of review candidates ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_review_candidates_v6_round2
""").fetchdf())

# Export to CSV
out_path = "../data_processed/exports/cds_to_mb_review_candidates_v6_round2.csv"
con.execute(f"""
    COPY cds_to_mb_review_candidates_v6_round2
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"\n✅ Exported: {out_path}")

print("\n--- Preview: review candidates ---")
print(con.execute("""
    SELECT
        cds_id, artist, title,
        release_id, main_artist_name, release_title,
        score, artist_sim, title_sim, score_gap
    FROM cds_to_mb_review_candidates_v6_round2
    ORDER BY score DESC, score_gap DESC NULLS LAST
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_to_mb_review_candidates_v6_round2...
✅ Table created: cds_to_mb_review_candidates_v6_round2

--- Count of review candidates ---
    n
0  64

✅ Exported: ../data_processed/exports/cds_to_mb_review_candidates_v6_round2.csv

--- Preview: review candidates ---
    cds_id                artist                                   title  release_id      main_artist_name                      release_title       score  artist_sim   title_sim  score_gap
0   101998           abbacadabra                       abbasalute disk 1      376867           Abbacadabra                         Abbasalute  100.000000  100.000000  100.000000   0.000000
1   107890          bessie smith   the complete recordings vol 5 disk 1       770956          Bessie Smith  The Complete Recordings, Volume 1  100.000000  100.000000  100.000000   0.000000
2   104641            pink floyd                      prism disk 2 live       907959            Pink Floyd                              Prism  100.0000

In [346]:
import duckdb
import os

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Make sure exports folder exists
os.makedirs("../data_processed/exports", exist_ok=True)

# Export final mapping v8
out_path = "../data_processed/exports/cds_to_mb_final_mapping_v8.csv"

con.execute(f"""
    COPY cds_to_mb_final_mapping_v8
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

# Quick check
print("\n--- Row count in cds_to_mb_final_mapping_v8 ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v8
""").fetchdf())

Connected ✅
✅ Exported: ../data_processed/exports/cds_to_mb_final_mapping_v8.csv

--- Row count in cds_to_mb_final_mapping_v8 ---
      n
0  4211


In [347]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Master status table for the full CDS source population
# ------------------------------------------------------------
# Comments:
# - This table is built from the whole CDS dataset.
# - It tracks, for every cds_id:
#     * whether it was matched
#     * by which route it was matched
#     * whether it is in the manual-review set
#     * whether it remains unmatched after v8
# - This is useful for thesis reporting because it proves that
#   the whole CDS source participated in the enrichment workflow.
print("Creating cds_enrichment_master_status...")

con.execute("""
    CREATE OR REPLACE TABLE cds_enrichment_master_status AS
    WITH base AS (
        SELECT
            c.cds_id,
            c.artist,
            c.title,
            c.cds_year,
            c.tracks,
            c.artist_key,
            c.title_key
        FROM cds_norm c
    ),

    final_match AS (
        SELECT
            cds_id,
            release_id,
            release_title,
            main_artist_name,
            artist_credit_full,
            total_tracks,
            barcode,
            mb_year,
            match_type,
            score
        FROM cds_to_mb_final_mapping_v8
    ),

    review AS (
        SELECT
            cds_id,
            release_id AS review_release_id,
            release_title AS review_release_title,
            main_artist_name AS review_main_artist_name,
            score AS review_score,
            artist_sim,
            title_sim,
            best_score,
            second_best_score,
            score_gap,
            confidence_class
        FROM cds_to_mb_review_candidates_v6_round2
    )

    SELECT
        b.cds_id,
        b.artist,
        b.title,
        b.cds_year,
        b.tracks,
        b.artist_key,
        b.title_key,

        -- Final automatic match information
        f.release_id,
        f.release_title,
        f.main_artist_name,
        f.artist_credit_full,
        f.total_tracks,
        f.barcode,
        f.mb_year,
        f.match_type,
        f.score AS final_score,

        -- Review candidate information
        r.review_release_id,
        r.review_release_title,
        r.review_main_artist_name,
        r.review_score,
        r.artist_sim AS review_artist_sim,
        r.title_sim AS review_title_sim,
        r.best_score AS review_best_score,
        r.second_best_score AS review_second_best_score,
        r.score_gap AS review_score_gap,
        r.confidence_class AS review_confidence_class,

        -- Master pipeline status
        CASE
            WHEN f.cds_id IS NOT NULL THEN 'matched_automatic'
            WHEN r.cds_id IS NOT NULL THEN 'review_candidate'
            ELSE 'still_unmatched'
        END AS pipeline_status

    FROM base b
    LEFT JOIN final_match f
      ON b.cds_id = f.cds_id
    LEFT JOIN review r
      ON b.cds_id = r.cds_id
""")

print("✅ Table created: cds_enrichment_master_status")

print("\n--- Master status breakdown ---")
print(con.execute("""
    SELECT pipeline_status, COUNT(*) AS n
    FROM cds_enrichment_master_status
    GROUP BY pipeline_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Match route breakdown (automatic only) ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_enrichment_master_status
    WHERE pipeline_status = 'matched_automatic'
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: master status table ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        pipeline_status,
        match_type,
        release_id,
        release_title,
        review_release_id,
        review_release_title
    FROM cds_enrichment_master_status
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_enrichment_master_status...
✅ Table created: cds_enrichment_master_status

--- Master status breakdown ---
     pipeline_status     n
0    still_unmatched  5509
1  matched_automatic  4211
2   review_candidate    43

--- Match route breakdown (automatic only) ---
               match_type     n
0                   exact  3482
1       fuzzy_second_pass   543
2      various_title_only   183
3  enrichment_round2_auto     3

--- Preview: master status table ---
    cds_id                artist                            title    pipeline_status          match_type  release_id                  release_title  review_release_id review_release_title
0    10000       backstreet boys                       millennium  matched_automatic               exact        8069                     Millennium               <NA>                 None
1   100001               various      frankfurt trance vol 04 cd1  matched_automatic  various_title_only      190028               Frankfu

In [348]:
import duckdb
import os

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_enrichment_master_status.csv"

con.execute(f"""
    COPY cds_enrichment_master_status
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

Connected ✅
✅ Exported: ../data_processed/exports/cds_enrichment_master_status.csv


In [349]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

report_full_source = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds_enrichment_master_status) AS cds_total,

        (SELECT COUNT(*) FROM cds_enrichment_master_status
         WHERE pipeline_status = 'matched_automatic') AS matched_automatic_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_enrichment_master_status
                     WHERE pipeline_status = 'matched_automatic')
            / (SELECT COUNT(*) FROM cds_enrichment_master_status), 2
        ) AS matched_automatic_pct,

        (SELECT COUNT(*) FROM cds_enrichment_master_status
         WHERE pipeline_status = 'review_candidate') AS review_candidate_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_enrichment_master_status
                     WHERE pipeline_status = 'review_candidate')
            / (SELECT COUNT(*) FROM cds_enrichment_master_status), 2
        ) AS review_candidate_pct,

        (SELECT COUNT(*) FROM cds_enrichment_master_status
         WHERE pipeline_status = 'still_unmatched') AS still_unmatched_n,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM cds_enrichment_master_status
                     WHERE pipeline_status = 'still_unmatched')
            / (SELECT COUNT(*) FROM cds_enrichment_master_status), 2
        ) AS still_unmatched_pct
""").fetchdf()

print("\n--- Thesis Report: Full CDS Source Participation ---")
print(report_full_source)

Connected ✅

--- Thesis Report: Full CDS Source Participation ---
   cds_total  matched_automatic_n  matched_automatic_pct  review_candidate_n  review_candidate_pct  still_unmatched_n  still_unmatched_pct
0       9763                 4211                  43.13                  43                  0.44               5509                56.43


In [350]:
import duckdb
import pandas as pd

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Professional curation of unmatched CDS after v6
# ------------------------------------------------------------
# Comments:
# - We do NOT delete from the source.
# - We create:
#     1) curated rows for active enrichment
#     2) excluded rows for documentation / audit trail
# - Exclusion is only for rows that are too damaged or too weak.
print("Creating curated and excluded unmatched tables...")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_curated_v1 AS
    SELECT
        *,
        CASE
            WHEN is_various_artists = 1 THEN 'special_various_route'
            WHEN has_disc_cd_vol = 1 THEN 'special_disc_cd_vol_route'
            WHEN enrichment_priority_v6 = 'high' THEN 'standard_enrichment'
            WHEN enrichment_priority_v6 = 'medium' THEN 'careful_enrichment'
            ELSE 'other'
        END AS curation_route
    FROM cds_unmatched_working_v6
    WHERE NOT (
        -- Exclude rows that are too damaged for trustworthy enrichment
        is_mojibake = 1
        OR missing_artist = 1
        OR missing_title = 1
        OR (short_artist = 1 AND short_title = 1)
        OR (
            is_generic_title = 1
            AND (artist IS NULL OR trim(artist) = '' OR lower(trim(artist)) IN (
                'various', 'various artists', 'various artist', 'va', 'v a', 'v.a.'
            ))
        )
    )
""")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_excluded_v1 AS
    SELECT
        *,
        CASE
            WHEN is_mojibake = 1 THEN 'unrecoverable_mojibake'
            WHEN missing_artist = 1 THEN 'missing_artist'
            WHEN missing_title = 1 THEN 'missing_title'
            WHEN short_artist = 1 AND short_title = 1 THEN 'too_short_artist_and_title'
            WHEN is_generic_title = 1
                 AND (artist IS NULL OR trim(artist) = '' OR lower(trim(artist)) IN (
                     'various', 'various artists', 'various artist', 'va', 'v a', 'v.a.'
                 ))
            THEN 'generic_and_weak_metadata'
            ELSE 'other_exclusion'
        END AS exclusion_reason
    FROM cds_unmatched_working_v6
    WHERE (
        is_mojibake = 1
        OR missing_artist = 1
        OR missing_title = 1
        OR (short_artist = 1 AND short_title = 1)
        OR (
            is_generic_title = 1
            AND (artist IS NULL OR trim(artist) = '' OR lower(trim(artist)) IN (
                'various', 'various artists', 'various artist', 'va', 'v a', 'v.a.'
            ))
        )
    )
""")

print("✅ Tables created: cds_unmatched_curated_v1, cds_unmatched_excluded_v1")

print("\n--- Curated vs excluded summary ---")
print(con.execute("""
    SELECT 'curated_for_enrichment' AS bucket, COUNT(*) AS n
    FROM cds_unmatched_curated_v1
    UNION ALL
    SELECT 'excluded_from_active_enrichment' AS bucket, COUNT(*) AS n
    FROM cds_unmatched_excluded_v1
""").fetchdf())

print("\n--- Exclusion reason breakdown ---")
print(con.execute("""
    SELECT exclusion_reason, COUNT(*) AS n
    FROM cds_unmatched_excluded_v1
    GROUP BY exclusion_reason
    ORDER BY n DESC
""").fetchdf())

print("\n--- Curation route breakdown ---")
print(con.execute("""
    SELECT curation_route, COUNT(*) AS n
    FROM cds_unmatched_curated_v1
    GROUP BY curation_route
    ORDER BY n DESC
""").fetchdf())

Connected ✅
Creating curated and excluded unmatched tables...
✅ Tables created: cds_unmatched_curated_v1, cds_unmatched_excluded_v1

--- Curated vs excluded summary ---
                            bucket     n
0           curated_for_enrichment  5015
1  excluded_from_active_enrichment   540

--- Exclusion reason breakdown ---
             exclusion_reason    n
0      unrecoverable_mojibake  509
1  too_short_artist_and_title   15
2               missing_title   10
3              missing_artist    6

--- Curation route breakdown ---
              curation_route     n
0        standard_enrichment  3348
1      special_various_route   896
2  special_disc_cd_vol_route   579
3                      other   167
4         careful_enrichment    25


In [351]:
import duckdb
import pandas as pd

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Main curated enrichment subset
# ------------------------------------------------------------
# Comments:
# - This is the safest main route from the curated unmatched set.
# - We keep:
#     * standard_enrichment
#     * careful_enrichment
# - We leave special_various_route and special_disc_cd_vol_route
#   for dedicated strategies.
print("Creating cds_unmatched_curated_main_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_unmatched_curated_main_v1 AS
    SELECT *
    FROM cds_unmatched_curated_v1
    WHERE curation_route IN ('standard_enrichment', 'careful_enrichment')
""")

print("✅ Table created: cds_unmatched_curated_main_v1")

print("\n--- Main curated enrichment subset ---")
print(con.execute("""
    SELECT curation_route, COUNT(*) AS n
    FROM cds_unmatched_curated_main_v1
    GROUP BY curation_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Count of rows in main curated enrichment subset ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_unmatched_curated_main_v1
""").fetchdf())

print("\n--- Preview: main curated enrichment subset ---")
print(con.execute("""
    SELECT
        cds_id, artist, title, cds_year,
        artist_key_clean, title_key_clean,
        curation_route
    FROM cds_unmatched_curated_main_v1
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_unmatched_curated_main_v1...
✅ Table created: cds_unmatched_curated_main_v1

--- Main curated enrichment subset ---
        curation_route     n
0  standard_enrichment  3348
1   careful_enrichment    25

--- Count of rows in main curated enrichment subset ---
      n
0  3373

--- Preview: main curated enrichment subset ---
    cds_id                             artist                                                             title  cds_year              artist_key_clean                                           title_key_clean  \
0   100004                            emanuel                                                        felicidade      1998                       emanuel                                                felicidade   
1   100009                             4 p m                                                         sweet soul      2001                           4pm                                                 sweetsoul   
2   100028 

In [352]:
import duckdb
import pandas as pd

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Candidate pairs for the main curated enrichment subset
# ------------------------------------------------------------
# Comments:
# - Uses cleaned CDS keys from cds_unmatched_curated_main_v1
# - Matches against the cleaned MusicBrainz pool
# - Blocking:
#     * same first artist letter
#     * same first title letter
#     * similar artist/title key lengths
print("Creating cds_mb_curated_main_pairs_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_mb_curated_main_pairs_v1 AS
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.tracks,
        c.curation_route,

        c.artist_key_clean AS cds_artist_key,
        c.title_key_clean  AS cds_title_key,

        m.release_id,
        m.main_artist_name,
        m.release_title,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        m.artist_key_clean AS mb_artist_key,
        m.title_key_clean  AS mb_title_key

    FROM (
        SELECT
            *,
            substr(artist_key_clean, 1, 1) AS artist_first,
            substr(title_key_clean,  1, 1) AS title_first,
            length(artist_key_clean) AS artist_len,
            length(title_key_clean)  AS title_len
        FROM cds_unmatched_curated_main_v1
        WHERE
            artist_key_clean IS NOT NULL
            AND title_key_clean IS NOT NULL
            AND length(artist_key_clean) >= 4
            AND length(title_key_clean)  >= 4
    ) c

    JOIN mb_candidate_pool_v2 m
      ON m.artist_first = c.artist_first
     AND m.title_first  = c.title_first
     AND m.artist_len BETWEEN c.artist_len - 3 AND c.artist_len + 3
     AND m.title_len  BETWEEN c.title_len  - 5 AND c.title_len  + 5
""")

print("✅ Table created: cds_mb_curated_main_pairs_v1")

print("\n--- Pair count for curated main subset ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_curated_main_pairs_v1
""").fetchdf())

print("\n--- Distinct CDS rows with candidate pairs ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_candidates
    FROM cds_mb_curated_main_pairs_v1
""").fetchdf())

print("\n--- Preview: curated main candidate pairs ---")
print(con.execute("""
    SELECT
        cds_id, artist, title,
        release_id, main_artist_name, release_title,
        cds_year, mb_year, total_tracks, barcode
    FROM cds_mb_curated_main_pairs_v1
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_mb_curated_main_pairs_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_mb_curated_main_pairs_v1

--- Pair count for curated main subset ---
         n
0  8725533

--- Distinct CDS rows with candidate pairs ---
   cds_rows_with_candidates
0                      3206

--- Preview: curated main candidate pairs ---
    cds_id                artist                         title  release_id   main_artist_name             release_title  cds_year  mb_year  total_tracks        barcode
0   108372                 thebe                        mokone     3649233          The BREED              Making a Way      2002     2022           1.0   745051303228
1   108488      mofunk colective              fresh jazz vibes     3649237   Maynard Ferguson   Live From San Francisco      1995     2022           7.0   081971807720
2   103432      ben zion shekner          v lirusholayim ircho     2918800  Bruce Springsteen          VH1 Storytellers      2003     2023          21.0  0828767352992
3   108798              schubert                     lieder 31   

In [353]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE lower(table_name) LIKE '%artist%'
       OR lower(table_name) LIKE '%alias%'
    ORDER BY table_name
""").fetchdf())

Connected ✅
                                            table_name
0                         cds_alias_enrichment_auto_v1
1                  cds_alias_enrichment_auto_v2_review
2                    cds_alias_enrichment_auto_v2_safe
3                  cds_alias_enrichment_auto_v3_review
4                    cds_alias_enrichment_auto_v3_safe
5                       cds_alias_enrichment_ranked_v1
6                    cds_artist_alias_bridge_unique_v1
7                           cds_artist_alias_bridge_v1
8   cds_artist_resolved_release_candidates_filtered_v1
9            cds_artist_resolved_release_candidates_v1
10                                           mb_artist
11                                     mb_artist_alias
12                                 mb_artist_alias_raw
13                               mb_artist_credit_core
14            mb_artist_credit_for_resolved_artists_v1
15                               mb_artist_credit_name
16                                mb_artist_credit_ra

In [62]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    PRAGMA table_info('mb_artist')
""").fetchdf())

Connected ✅
    cid              name       type  notnull dflt_value     pk
0     0                id    INTEGER    False       None  False
1     1               gid    VARCHAR    False       None  False
2     2              name    VARCHAR    False       None  False
3     3         sort_name    VARCHAR    False       None  False
4     4   begin_date_year    INTEGER    False       None  False
5     5  begin_date_month    INTEGER    False       None  False
6     6    begin_date_day    INTEGER    False       None  False
7     7     end_date_year    INTEGER    False       None  False
8     8    end_date_month    INTEGER    False       None  False
9     9      end_date_day    INTEGER    False       None  False
10   10              type    INTEGER    False       None  False
11   11              area    INTEGER    False       None  False
12   12            gender    INTEGER    False       None  False
13   13           comment    VARCHAR    False       None  False
14   14     edits_pending   

In [63]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    PRAGMA table_info('mb_artist')
""").fetchdf())

Connected ✅
    cid              name       type  notnull dflt_value     pk
0     0                id    INTEGER    False       None  False
1     1               gid    VARCHAR    False       None  False
2     2              name    VARCHAR    False       None  False
3     3         sort_name    VARCHAR    False       None  False
4     4   begin_date_year    INTEGER    False       None  False
5     5  begin_date_month    INTEGER    False       None  False
6     6    begin_date_day    INTEGER    False       None  False
7     7     end_date_year    INTEGER    False       None  False
8     8    end_date_month    INTEGER    False       None  False
9     9      end_date_day    INTEGER    False       None  False
10   10              type    INTEGER    False       None  False
11   11              area    INTEGER    False       None  False
12   12            gender    INTEGER    False       None  False
13   13           comment    VARCHAR    False       None  False
14   14     edits_pending   

In [354]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

artist_alias_path = "../data_raw/musicbrainz/mbdump/mbdump/artist_alias"

print("Importing artist_alias...")

con.execute(f"""
    CREATE OR REPLACE TABLE mb_artist_alias_raw AS
    SELECT *
    FROM read_csv_auto(
        '{artist_alias_path}',
        delim = '\t',
        header = false,
        null_padding = true,
        ignore_errors = true,
        strict_mode = false
    )
""")

print("✅ Table created: mb_artist_alias_raw")

print("\n--- Schema: mb_artist_alias_raw ---")
print(con.execute("""
    PRAGMA table_info('mb_artist_alias_raw')
""").fetchdf())

print("\n--- Preview: mb_artist_alias_raw ---")
print(con.execute("""
    SELECT *
    FROM mb_artist_alias_raw
    LIMIT 10
""").fetchdf())

Connected ✅
Importing artist_alias...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: mb_artist_alias_raw

--- Schema: mb_artist_alias_raw ---
    cid      name     type  notnull dflt_value     pk
0     0  column00   BIGINT    False       None  False
1     1  column01   BIGINT    False       None  False
2     2  column02  VARCHAR    False       None  False
3     3  column03  VARCHAR    False       None  False
4     4  column04  VARCHAR    False       None  False
5     5  column05  VARCHAR    False       None  False
6     6  column06  VARCHAR    False       None  False
7     7  column07  VARCHAR    False       None  False
8     8  column08  VARCHAR    False       None  False
9     9  column09  VARCHAR    False       None  False
10   10  column10  VARCHAR    False       None  False
11   11  column11  VARCHAR    False       None  False
12   12  column12  VARCHAR    False       None  False
13   13  column13  VARCHAR    False       None  False
14   14  column14  BOOLEAN    False       None  False
15   15  column15  BOOLEAN    False       None  False

--- Pre

In [355]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Clean and rename the raw MusicBrainz artist_alias table
# ------------------------------------------------------------
# Comments:
# - The raw MusicBrainz dump has no headers.
# - It uses '\N' for NULL values.
# - Boolean fields are stored as t / f / \N, so we map them manually.
# - We also create normalized alias keys for later matching.
print("Creating cleaned MusicBrainz artist alias table...")

con.execute(r"""
    CREATE OR REPLACE TABLE mb_artist_alias AS
    SELECT
        CAST(column00 AS BIGINT) AS alias_id,
        CAST(column01 AS BIGINT) AS artist_id,

        CASE WHEN column02 = '\N' THEN NULL ELSE column02 END AS alias_name,
        CASE WHEN column03 = '\N' THEN NULL ELSE column03 END AS locale,
        TRY_CAST(CASE WHEN column04 = '\N' THEN NULL ELSE column04 END AS INTEGER) AS edits_pending,
        TRY_CAST(CASE WHEN column05 = '\N' THEN NULL ELSE column05 END AS TIMESTAMP) AS last_updated,
        TRY_CAST(CASE WHEN column06 = '\N' THEN NULL ELSE column06 END AS INTEGER) AS alias_type_id,
        CASE WHEN column07 = '\N' THEN NULL ELSE column07 END AS alias_sort_name,

        TRY_CAST(CASE WHEN column08 = '\N' THEN NULL ELSE column08 END AS INTEGER) AS begin_date_year,
        TRY_CAST(CASE WHEN column09 = '\N' THEN NULL ELSE column09 END AS INTEGER) AS begin_date_month,
        TRY_CAST(CASE WHEN column10 = '\N' THEN NULL ELSE column10 END AS INTEGER) AS begin_date_day,
        TRY_CAST(CASE WHEN column11 = '\N' THEN NULL ELSE column11 END AS INTEGER) AS end_date_year,
        TRY_CAST(CASE WHEN column12 = '\N' THEN NULL ELSE column12 END AS INTEGER) AS end_date_month,
        TRY_CAST(CASE WHEN column13 = '\N' THEN NULL ELSE column13 END AS INTEGER) AS end_date_day,

        CASE
            WHEN column14 IN ('t', 'true', '1') THEN TRUE
            WHEN column14 IN ('f', 'false', '0') THEN FALSE
            ELSE NULL
        END AS primary_for_locale,

        CASE
            WHEN column15 IN ('t', 'true', '1') THEN TRUE
            WHEN column15 IN ('f', 'false', '0') THEN FALSE
            ELSE NULL
        END AS ended,

        -- ----------------------------------------------------
        -- Normalized keys for artist alias matching
        -- ----------------------------------------------------
        lower(
            regexp_replace(
                coalesce(CASE WHEN column02 = '\N' THEN NULL ELSE column02 END, ''),
                '[^a-zA-Z0-9]+', '', 'g'
            )
        ) AS alias_name_key,

        lower(
            regexp_replace(
                coalesce(CASE WHEN column07 = '\N' THEN NULL ELSE column07 END, ''),
                '[^a-zA-Z0-9]+', '', 'g'
            )
        ) AS alias_sort_name_key

    FROM mb_artist_alias_raw
""")

print("✅ Table created: mb_artist_alias")

print("\n--- Schema: mb_artist_alias ---")
print(con.execute("""
    PRAGMA table_info('mb_artist_alias')
""").fetchdf())

print("\n--- Preview: cleaned artist aliases ---")
print(con.execute("""
    SELECT
        alias_id,
        artist_id,
        alias_name,
        locale,
        alias_type_id,
        alias_sort_name,
        primary_for_locale,
        ended,
        alias_name_key,
        alias_sort_name_key
    FROM mb_artist_alias
    LIMIT 20
""").fetchdf())

print("\n--- Basic counts ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_alias_rows,
        COUNT(DISTINCT artist_id) AS distinct_artists_with_aliases,
        COUNT(*) FILTER (WHERE alias_name IS NOT NULL) AS alias_name_present,
        COUNT(*) FILTER (WHERE primary_for_locale = TRUE) AS primary_locale_aliases
    FROM mb_artist_alias
""").fetchdf())

Connected ✅
Creating cleaned MusicBrainz artist alias table...
✅ Table created: mb_artist_alias

--- Schema: mb_artist_alias ---
    cid                 name       type  notnull dflt_value     pk
0     0             alias_id     BIGINT    False       None  False
1     1            artist_id     BIGINT    False       None  False
2     2           alias_name    VARCHAR    False       None  False
3     3               locale    VARCHAR    False       None  False
4     4        edits_pending    INTEGER    False       None  False
5     5         last_updated  TIMESTAMP    False       None  False
6     6        alias_type_id    INTEGER    False       None  False
7     7      alias_sort_name    VARCHAR    False       None  False
8     8      begin_date_year    INTEGER    False       None  False
9     9     begin_date_month    INTEGER    False       None  False
10   10       begin_date_day    INTEGER    False       None  False
11   11        end_date_year    INTEGER    False       None  False


In [356]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build a unified MusicBrainz artist name lookup
# ------------------------------------------------------------
# Comments:
# - We combine canonical MB artist names with MB artist aliases.
# - This creates a richer lookup source for artist-side resolution.
# - Later, CDS artist strings can be matched against:
#     * canonical artist names
#     * canonical sort names
#     * alias names
#     * alias sort names
print("Creating mb_artist_name_lookup_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE mb_artist_name_lookup_v1 AS

    -- --------------------------------------------------------
    -- Canonical artist name from mb_artist.name
    -- --------------------------------------------------------
    SELECT
        a.id AS artist_id,
        a.gid AS artist_gid,
        a.name AS artist_display_name,
        a.sort_name AS artist_sort_name,

        a.name AS lookup_name,
        'canonical_name' AS lookup_source,
        lower(regexp_replace(coalesce(a.name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS lookup_name_key,
        NULL::VARCHAR AS locale,
        NULL::INTEGER AS alias_type_id,
        NULL::BOOLEAN AS primary_for_locale

    FROM mb_artist a
    WHERE a.name IS NOT NULL

    UNION ALL

    -- --------------------------------------------------------
    -- Canonical sort_name from mb_artist.sort_name
    -- --------------------------------------------------------
    SELECT
        a.id AS artist_id,
        a.gid AS artist_gid,
        a.name AS artist_display_name,
        a.sort_name AS artist_sort_name,

        a.sort_name AS lookup_name,
        'canonical_sort_name' AS lookup_source,
        lower(regexp_replace(coalesce(a.sort_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS lookup_name_key,
        NULL::VARCHAR AS locale,
        NULL::INTEGER AS alias_type_id,
        NULL::BOOLEAN AS primary_for_locale

    FROM mb_artist a
    WHERE a.sort_name IS NOT NULL

    UNION ALL

    -- --------------------------------------------------------
    -- Alias name from mb_artist_alias.alias_name
    -- --------------------------------------------------------
    SELECT
        a.id AS artist_id,
        a.gid AS artist_gid,
        a.name AS artist_display_name,
        a.sort_name AS artist_sort_name,

        aa.alias_name AS lookup_name,
        'alias_name' AS lookup_source,
        aa.alias_name_key AS lookup_name_key,
        aa.locale,
        aa.alias_type_id,
        aa.primary_for_locale

    FROM mb_artist_alias aa
    JOIN mb_artist a
      ON a.id = aa.artist_id
    WHERE aa.alias_name IS NOT NULL

    UNION ALL

    -- --------------------------------------------------------
    -- Alias sort_name from mb_artist_alias.alias_sort_name
    -- --------------------------------------------------------
    SELECT
        a.id AS artist_id,
        a.gid AS artist_gid,
        a.name AS artist_display_name,
        a.sort_name AS artist_sort_name,

        aa.alias_sort_name AS lookup_name,
        'alias_sort_name' AS lookup_source,
        aa.alias_sort_name_key AS lookup_name_key,
        aa.locale,
        aa.alias_type_id,
        aa.primary_for_locale

    FROM mb_artist_alias aa
    JOIN mb_artist a
      ON a.id = aa.artist_id
    WHERE aa.alias_sort_name IS NOT NULL
""")

print("✅ Table created: mb_artist_name_lookup_v1")

print("\n--- Lookup source breakdown ---")
print(con.execute("""
    SELECT lookup_source, COUNT(*) AS n
    FROM mb_artist_name_lookup_v1
    GROUP BY lookup_source
    ORDER BY n DESC
""").fetchdf())

print("\n--- Distinct artists represented in lookup ---")
print(con.execute("""
    SELECT COUNT(DISTINCT artist_id) AS distinct_artists
    FROM mb_artist_name_lookup_v1
""").fetchdf())

print("\n--- Preview: artist lookup table ---")
print(con.execute("""
    SELECT
        artist_id,
        artist_display_name,
        artist_sort_name,
        lookup_name,
        lookup_source,
        lookup_name_key,
        locale,
        alias_type_id,
        primary_for_locale
    FROM mb_artist_name_lookup_v1
    LIMIT 25
""").fetchdf())

Connected ✅
Creating mb_artist_name_lookup_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: mb_artist_name_lookup_v1

--- Lookup source breakdown ---
         lookup_source        n
0       canonical_name  2807098
1  canonical_sort_name  2807098
2           alias_name   497697
3      alias_sort_name   497668

--- Distinct artists represented in lookup ---
   distinct_artists
0           2807098

--- Preview: artist lookup table ---
    artist_id       artist_display_name            artist_sort_name               lookup_name   lookup_source         lookup_name_key locale  alias_type_id  primary_for_locale
0     1214795              Radek Kašpar               Kašpar, Radek              Radek Kašpar  canonical_name              radekkapar   None           <NA>                <NA>
1     1819567        Oracle da Goldsmif         Goldsmif, Oracle da        Oracle da Goldsmif  canonical_name        oracledagoldsmif   None           <NA>                <NA>
2     1215271               Kari Jerkku                 Kari Jerkku               Kari Jerkku  canonical_name  

In [357]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Alias-assisted CDS artist resolution
# ------------------------------------------------------------
# Comments:
# - We use the curated unmatched CDS subset from v6.
# - We try to resolve CDS artist strings against:
#     * canonical MB artist names
#     * canonical MB sort names
#     * alias names
#     * alias sort names
# - This step only resolves ARTIST identity.
# - It does not yet decide the final MB release match.
print("Creating cds_artist_alias_bridge_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_artist_alias_bridge_v1 AS
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.curation_route,
        c.enrichment_priority_v6,
        c.artist_key_clean,
        c.title_key_clean,

        l.artist_id,
        l.artist_gid,
        l.artist_display_name,
        l.artist_sort_name,
        l.lookup_name,
        l.lookup_source,
        l.locale,
        l.alias_type_id,
        l.primary_for_locale

    FROM cds_unmatched_curated_v1 c
    JOIN mb_artist_name_lookup_v1 l
      ON c.artist_key_clean = l.lookup_name_key
    WHERE c.artist_key_clean IS NOT NULL
      AND length(c.artist_key_clean) >= 4
""")

print("✅ Table created: cds_artist_alias_bridge_v1")

print("\n--- Total bridge rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_alias_bridge_v1
""").fetchdf())

print("\n--- Distinct CDS rows with artist resolution ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_artist_match
    FROM cds_artist_alias_bridge_v1
""").fetchdf())

print("\n--- Lookup source breakdown ---")
print(con.execute("""
    SELECT lookup_source, COUNT(*) AS n
    FROM cds_artist_alias_bridge_v1
    GROUP BY lookup_source
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: alias-assisted CDS artist bridge ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_key_clean,
        artist_id,
        artist_display_name,
        artist_sort_name,
        lookup_name,
        lookup_source,
        locale,
        primary_for_locale
    FROM cds_artist_alias_bridge_v1
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_artist_alias_bridge_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_artist_alias_bridge_v1

--- Total bridge rows ---
       n
0  26404

--- Distinct CDS rows with artist resolution ---
   cds_rows_with_artist_match
0                        3238

--- Lookup source breakdown ---
         lookup_source      n
0       canonical_name  10185
1  canonical_sort_name   9242
2      alias_sort_name   3565
3           alias_name   3412

--- Preview: alias-assisted CDS artist bridge ---
    cds_id      artist                                                        title artist_key_clean  artist_id artist_display_name artist_sort_name lookup_name   lookup_source locale  primary_for_locale
0   109759     various                                           warriors of virtue          various    3151913             Various          Various     Various  canonical_name   None                <NA>
1   105478  drew young                                             hide out with me        drewyoung    3152215          Drew Young      Young, Drew  Drew Youn

In [ ]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build artist-resolved release candidates
# ------------------------------------------------------------
# Comments:
# - We already resolved CDS artist strings to candidate MB artists
#   using canonical names + alias names.
# - Now we expand each resolved MB artist to candidate releases.
# - We do this through:
#     mb_artist_credit_name  -> links artist to artist_credit
#     mb_release_features    -> links artist_credit_id to releases
# - Important fix:
#     mb_release_features uses artist_credit_id, not artist_credit.
print("Creating cds_artist_resolved_release_candidates_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_artist_resolved_release_candidates_v1 AS
    SELECT DISTINCT
        b.cds_id,
        b.artist,
        b.title,
        b.cds_year,
        b.curation_route,
        b.enrichment_priority_v6,
        b.artist_key_clean,
        b.title_key_clean,

        b.artist_id,
        b.artist_gid,
        b.artist_display_name,
        b.artist_sort_name,
        b.lookup_name,
        b.lookup_source,
        b.locale,
        b.alias_type_id,
        b.primary_for_locale,

        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.artist_credit_id,
        r.total_tracks,
        r.barcode,
        r.mb_year

    FROM cds_artist_alias_bridge_v1 b
    JOIN mb_artist_credit_name acn
      ON acn.artist = b.artist_id
    JOIN mb_release_features r
      ON r.artist_credit_id = acn.artist_credit
""")

print("✅ Table created: cds_artist_resolved_release_candidates_v1")

print("\n--- Total candidate rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_resolved_release_candidates_v1
""").fetchdf())

print("\n--- Distinct CDS rows with release candidates ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_release_candidates
    FROM cds_artist_resolved_release_candidates_v1
""").fetchdf())

print("\n--- Lookup source breakdown inside release candidates ---")
print(con.execute("""
    SELECT lookup_source, COUNT(*) AS n
    FROM cds_artist_resolved_release_candidates_v1
    GROUP BY lookup_source
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: artist-resolved release candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        lookup_name,
        lookup_source,
        release_id,
        main_artist_name,
        release_title,
        mb_year,
        total_tracks,
        barcode
    FROM cds_artist_resolved_release_candidates_v1
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_artist_resolved_release_candidates_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Memory-safe settings
# ------------------------------------------------------------
# Comments:
# - Fewer threads usually reduce peak memory.
# - Turning off insertion-order preservation can reduce overhead.
# - A temp directory lets DuckDB spill intermediate data to disk.
# - memory_limit should stay conservative to avoid crashes.
print("Applying DuckDB memory-safe settings...")

con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")
con.execute("PRAGMA memory_limit='8GB'")

print("✅ Settings applied")

# ------------------------------------------------------------
# Step 1: Deduplicate CDS x artist_id pairs
# ------------------------------------------------------------
# Comments:
# - The alias bridge may contain multiple rows for the same cds_id + artist_id
#   because the same artist can be reached via canonical name, sort name, alias, etc.
# - We keep only the best lookup source per cds_id + artist_id.
print("\nCreating cds_artist_alias_bridge_unique_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_artist_alias_bridge_unique_v1 AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id, artist_id
                ORDER BY
                    CASE lookup_source
                        WHEN 'canonical_name' THEN 1
                        WHEN 'alias_name' THEN 2
                        WHEN 'canonical_sort_name' THEN 3
                        WHEN 'alias_sort_name' THEN 4
                        ELSE 5
                    END,
                    CASE WHEN primary_for_locale = TRUE THEN 0 ELSE 1 END,
                    alias_type_id ASC NULLS LAST
            ) AS rn
        FROM cds_artist_alias_bridge_v1
    )
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        curation_route,
        enrichment_priority_v6,
        artist_key_clean,
        title_key_clean,
        artist_id,
        artist_gid,
        artist_display_name,
        artist_sort_name,
        lookup_name,
        lookup_source,
        locale,
        alias_type_id,
        primary_for_locale
    FROM ranked
    WHERE rn = 1
""")

print("✅ Table created: cds_artist_alias_bridge_unique_v1")

print("\n--- Unique CDS x artist pairs ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_alias_bridge_unique_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 2: Build reduced artist-credit map only for resolved artists
# ------------------------------------------------------------
# Comments:
# - This reduces mb_artist_credit_name to only the artist IDs we actually need.
print("\nCreating mb_artist_credit_for_resolved_artists_v1...")

con.execute("""
    CREATE OR REPLACE TABLE mb_artist_credit_for_resolved_artists_v1 AS
    SELECT DISTINCT
        acn.artist,
        acn.artist_credit
    FROM mb_artist_credit_name acn
    JOIN (
        SELECT DISTINCT artist_id
        FROM cds_artist_alias_bridge_unique_v1
    ) x
      ON acn.artist = x.artist_id
""")

print("✅ Table created: mb_artist_credit_for_resolved_artists_v1")

print("\n--- Artist-credit links for resolved artists ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM mb_artist_credit_for_resolved_artists_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 3: Build reduced artist-to-release map first
# ------------------------------------------------------------
# Comments:
# - This is much smaller than joining the full bridge directly to releases.
# - We materialize only artist -> release links for already-resolved artists.
print("\nCreating mb_artist_release_for_resolved_artists_v1...")

con.execute("""
    CREATE OR REPLACE TABLE mb_artist_release_for_resolved_artists_v1 AS
    SELECT DISTINCT
        ac.artist,
        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.artist_credit_id,
        r.total_tracks,
        r.barcode,
        r.mb_year
    FROM mb_artist_credit_for_resolved_artists_v1 ac
    JOIN mb_release_features r
      ON r.artist_credit_id = ac.artist_credit
""")

print("✅ Table created: mb_artist_release_for_resolved_artists_v1")

print("\n--- Reduced artist-release rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM mb_artist_release_for_resolved_artists_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 4: Create empty final candidate table
# ------------------------------------------------------------
# Comments:
# - We will insert into this table in chunks of artist_id values.
# - Chunking avoids one giant join result in memory.
print("\nPreparing cds_artist_resolved_release_candidates_v1...")

con.execute("""
    DROP TABLE IF EXISTS cds_artist_resolved_release_candidates_v1
""")

con.execute("""
    CREATE TABLE cds_artist_resolved_release_candidates_v1 AS
    SELECT
        b.cds_id,
        b.artist,
        b.title,
        b.cds_year,
        b.curation_route,
        b.enrichment_priority_v6,
        b.artist_key_clean,
        b.title_key_clean,
        b.artist_id,
        b.artist_gid,
        b.artist_display_name,
        b.artist_sort_name,
        b.lookup_name,
        b.lookup_source,
        b.locale,
        b.alias_type_id,
        b.primary_for_locale,
        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.artist_credit_id,
        r.total_tracks,
        r.barcode,
        r.mb_year
    FROM cds_artist_alias_bridge_unique_v1 b
    JOIN mb_artist_release_for_resolved_artists_v1 r
      ON r.artist = b.artist_id
    WHERE 1 = 0
""")

print("✅ Empty table created: cds_artist_resolved_release_candidates_v1")

# ------------------------------------------------------------
# Step 5: Insert candidates in artist_id chunks
# ------------------------------------------------------------
# Comments:
# - We chunk on artist_id, not cds_id, because the explosion happens on artist->release expansion.
# - Adjust CHUNK_SIZE if needed.
artist_ids = con.execute("""
    SELECT DISTINCT artist_id
    FROM cds_artist_alias_bridge_unique_v1
    ORDER BY artist_id
""").fetchdf()["artist_id"].tolist()

CHUNK_SIZE = 5000
print(f"\nInserting candidate rows in chunks of {CHUNK_SIZE} artist IDs...")

for start in range(0, len(artist_ids), CHUNK_SIZE):
    chunk = artist_ids[start:start + CHUNK_SIZE]
    lo = chunk[0]
    hi = chunk[-1]
    print(f"  Processing artist_id chunk {start:,}..{min(start + CHUNK_SIZE, len(artist_ids)):,}  "
          f"(artist_id {lo}..{hi})")

    values_sql = ",".join(str(int(x)) for x in chunk)

    con.execute(f"""
        INSERT INTO cds_artist_resolved_release_candidates_v1
        SELECT
            b.cds_id,
            b.artist,
            b.title,
            b.cds_year,
            b.curation_route,
            b.enrichment_priority_v6,
            b.artist_key_clean,
            b.title_key_clean,
            b.artist_id,
            b.artist_gid,
            b.artist_display_name,
            b.artist_sort_name,
            b.lookup_name,
            b.lookup_source,
            b.locale,
            b.alias_type_id,
            b.primary_for_locale,
            r.release_id,
            r.release_title,
            r.main_artist_name,
            r.artist_credit_full,
            r.artist_credit_id,
            r.total_tracks,
            r.barcode,
            r.mb_year
        FROM cds_artist_alias_bridge_unique_v1 b
        JOIN mb_artist_release_for_resolved_artists_v1 r
          ON r.artist = b.artist_id
        WHERE b.artist_id IN ({values_sql})
    """)

print("✅ Table created: cds_artist_resolved_release_candidates_v1")

print("\n--- Total candidate rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_resolved_release_candidates_v1
""").fetchdf())

print("\n--- Distinct CDS rows with release candidates ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_release_candidates
    FROM cds_artist_resolved_release_candidates_v1
""").fetchdf())

print("\n--- Average candidates per CDS row ---")
print(con.execute("""
    SELECT ROUND(AVG(k), 2) AS avg_candidates_per_cds
    FROM (
        SELECT cds_id, COUNT(*) AS k
        FROM cds_artist_resolved_release_candidates_v1
        GROUP BY cds_id
    )
""").fetchdf())

print("\n--- Preview: artist-resolved release candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        lookup_name,
        lookup_source,
        release_id,
        main_artist_name,
        release_title,
        mb_year,
        total_tracks,
        barcode
    FROM cds_artist_resolved_release_candidates_v1
    LIMIT 25
""").fetchdf())

Connected ✅
Applying DuckDB memory-safe settings...
✅ Settings applied

Creating cds_artist_alias_bridge_unique_v1...
✅ Table created: cds_artist_alias_bridge_unique_v1

--- Unique CDS x artist pairs ---
       n
0  11585

Creating mb_artist_credit_for_resolved_artists_v1...
✅ Table created: mb_artist_credit_for_resolved_artists_v1

--- Artist-credit links for resolved artists ---
        n
0  177386

Creating mb_artist_release_for_resolved_artists_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: mb_artist_release_for_resolved_artists_v1

--- Reduced artist-release rows ---
        n
0  631241

Preparing cds_artist_resolved_release_candidates_v1...
✅ Empty table created: cds_artist_resolved_release_candidates_v1

Inserting candidate rows in chunks of 5000 artist IDs...
  Processing artist_id chunk 0..5,000  (artist_id 1..2674420)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Processing artist_id chunk 5,000..5,796  (artist_id 2674601..3208878)
✅ Table created: cds_artist_resolved_release_candidates_v1

--- Total candidate rows ---
           n
0  241255822

--- Distinct CDS rows with release candidates ---
   cds_rows_with_release_candidates
0                              2820

--- Average candidates per CDS row ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   avg_candidates_per_cds
0                85551.71

--- Preview: artist-resolved release candidates ---
    cds_id   artist                  title artist_display_name lookup_name  \
0   103671  various  the ultimate hardcore     Various Artists     Various   
1   103671  various  the ultimate hardcore     Various Artists     Various   
2   103671  various  the ultimate hardcore     Various Artists     Various   
3   103671  various  the ultimate hardcore     Various Artists     Various   
4   103671  various  the ultimate hardcore     Various Artists     Various   
5   103671  various  the ultimate hardcore     Various Artists     Various   
6   103671  various  the ultimate hardcore     Various Artists     Various   
7   103671  various  the ultimate hardcore     Various Artists     Various   
8   103671  various  the ultimate hardcore     Various Artists     Various   
9   103671  various  the ultimate hardcore     Various Artists     Various   
10  103671  various  the ultimate har

In [19]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Filter alias-assisted release candidates before scoring
# ------------------------------------------------------------
# Comments:
# - The error came from subtracting r.total_tracks - c.tracks
# - c.tracks is text (VARCHAR), so we must convert it to numeric first.
# - We use TRY_CAST so bad/non-numeric values become NULL instead of failing.
print("Creating cds_artist_resolved_release_candidates_filtered_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_artist_resolved_release_candidates_filtered_v1 AS
    WITH cds_curated_numeric AS (
        SELECT
            cds_id,
            TRY_CAST(tracks AS BIGINT) AS cds_trackcount_num
        FROM cds_unmatched_curated_main_v1
    )
    SELECT
        r.*,
        c.cds_trackcount_num
    FROM cds_artist_resolved_release_candidates_v1 r
    LEFT JOIN cds_curated_numeric c
      ON r.cds_id = c.cds_id
    WHERE
        -- Exclude non-discriminative artist identities
        lower(coalesce(r.artist_display_name, '')) NOT IN ('various artists', 'various')
        AND lower(coalesce(r.main_artist_name, '')) NOT IN ('various artists', 'various')

        -- Release title must exist
        AND r.release_title IS NOT NULL
        AND trim(r.release_title) <> ''

        -- Year compatibility
        AND (
            r.cds_year IS NULL
            OR r.mb_year IS NULL
            OR abs(r.cds_year - r.mb_year) <= 5
        )

        -- Rough track compatibility
        AND (
            c.cds_trackcount_num IS NULL
            OR r.total_tracks IS NULL
            OR abs(r.total_tracks - c.cds_trackcount_num) <= 8
        )
""")

print("✅ Table created: cds_artist_resolved_release_candidates_filtered_v1")

print("\n--- Candidate count before filtering ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_resolved_release_candidates_v1
""").fetchdf())

print("\n--- Candidate count after filtering ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_artist_resolved_release_candidates_filtered_v1
""").fetchdf())

print("\n--- Distinct CDS rows after filtering ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS cds_rows_with_candidates
    FROM cds_artist_resolved_release_candidates_filtered_v1
""").fetchdf())

print("\n--- Average candidates per CDS row after filtering ---")
print(con.execute("""
    SELECT ROUND(AVG(k), 2) AS avg_candidates_per_cds
    FROM (
        SELECT cds_id, COUNT(*) AS k
        FROM cds_artist_resolved_release_candidates_filtered_v1
        GROUP BY cds_id
    )
""").fetchdf())

print("\n--- Preview: filtered alias-assisted candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        lookup_name,
        lookup_source,
        release_id,
        main_artist_name,
        release_title,
        cds_year,
        mb_year,
        cds_trackcount_num,
        total_tracks,
        barcode
    FROM cds_artist_resolved_release_candidates_filtered_v1
    LIMIT 25
""").fetchdf())

Connected ✅
Creating cds_artist_resolved_release_candidates_filtered_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_artist_resolved_release_candidates_filtered_v1

--- Candidate count before filtering ---
           n
0  241255822

--- Candidate count after filtering ---
        n
0  544138

--- Distinct CDS rows after filtering ---
   cds_rows_with_candidates
0                      1839

--- Average candidates per CDS row after filtering ---
   avg_candidates_per_cds
0                  295.89

--- Preview: filtered alias-assisted candidates ---
    cds_id                            artist  \
0   102060                      scott joplin   
1   107053                      king crimson   
2   109175                   igor stravinsky   
3   107117                      joe satriani   
4     9068                      depeche mode   
5   107388                     reel big fish   
6   101188               cannonball adderley   
7   105763                         john cale   
8   101939  the royal philharmonic orchestra   
9   108994         london symphony orchestra   
10  100981     

In [20]:
import duckdb
import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Connect
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Score and rank filtered alias-assisted release candidates
# ------------------------------------------------------------
# Comments:
# - We already resolved artist identity through MB aliases.
# - So now the main discriminative signal is the CDS title vs MB release title.
# - We also use year compatibility and lookup-source quality.
# - Then we rank candidates per cds_id and assign confidence classes.
print("Loading filtered alias-assisted candidates...")

df = con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        curation_route,
        enrichment_priority_v6,
        artist_key_clean,
        title_key_clean,
        artist_id,
        artist_display_name,
        artist_sort_name,
        lookup_name,
        lookup_source,
        locale,
        alias_type_id,
        primary_for_locale,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year
    FROM cds_artist_resolved_release_candidates_filtered_v1
""").fetchdf()

print(f"✅ Loaded rows: {len(df):,}")

# ------------------------------------------------------------
# Compute title similarity + small metadata bonuses
# ------------------------------------------------------------
# Comments:
# - Artist is already "resolved" by the alias bridge, so title gets higher weight.
# - We add:
#     * year bonus if close
#     * barcode bonus
#     * lookup-source preference
print("Scoring alias-assisted candidates...")

title_sims = []
year_bonus = []
barcode_bonus = []
lookup_bonus = []
final_scores = []

for _, row in df.iterrows():
    cds_title = row["title"] if row["title"] is not None else ""
    mb_title = row["release_title"] if row["release_title"] is not None else ""

    t_sim = fuzz.token_set_ratio(str(cds_title), str(mb_title))
    title_sims.append(t_sim)

    # Year bonus
    if pd.isna(row["cds_year"]) or pd.isna(row["mb_year"]):
        y_bonus = 0.0
    else:
        diff = abs(int(row["cds_year"]) - int(row["mb_year"]))
        if diff == 0:
            y_bonus = 8.0
        elif diff <= 1:
            y_bonus = 5.0
        elif diff <= 3:
            y_bonus = 2.0
        else:
            y_bonus = -3.0
    year_bonus.append(y_bonus)

    # Barcode bonus
    b_bonus = 3.0 if pd.notna(row["barcode"]) and str(row["barcode"]).strip() != "" else 0.0
    barcode_bonus.append(b_bonus)

    # Lookup source bonus
    source = row["lookup_source"]
    if source == "canonical_name":
        l_bonus = 4.0
    elif source == "alias_name":
        l_bonus = 2.5
    elif source == "canonical_sort_name":
        l_bonus = 1.5
    elif source == "alias_sort_name":
        l_bonus = 1.0
    else:
        l_bonus = 0.0
    lookup_bonus.append(l_bonus)

    # Final weighted score
    score = 0.85 * t_sim + y_bonus + b_bonus + l_bonus
    final_scores.append(score)

df["title_sim"] = title_sims
df["year_bonus"] = year_bonus
df["barcode_bonus"] = barcode_bonus
df["lookup_bonus"] = lookup_bonus
df["alias_final_score"] = final_scores

# ------------------------------------------------------------
# Keep top candidates per CDS row
# ------------------------------------------------------------
df = df.sort_values(
    ["cds_id", "alias_final_score", "title_sim", "release_id"],
    ascending=[True, False, False, True]
)

# Keep top 5 per cds_id
df_top = df.groupby("cds_id").head(5).copy()

print("Top candidates retained:", len(df_top))

# Register and rank in DuckDB
con.register("alias_rank_df", df_top)

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_ranked_v1 AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY cds_id
                ORDER BY
                    alias_final_score DESC,
                    title_sim DESC,
                    release_id ASC
            ) AS candidate_rank,

            first_value(alias_final_score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    alias_final_score DESC,
                    title_sim DESC,
                    release_id ASC
            ) AS best_score,

            lead(alias_final_score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    alias_final_score DESC,
                    title_sim DESC,
                    release_id ASC
            ) AS second_best_score
        FROM alias_rank_df
    )
    SELECT
        *,
        (best_score - second_best_score) AS score_gap,
        CASE
            WHEN candidate_rank = 1
                 AND title_sim >= 92
                 AND alias_final_score >= 86
                 AND (best_score - second_best_score >= 6 OR second_best_score IS NULL)
            THEN 'auto_accept'

            WHEN candidate_rank = 1
                 AND title_sim >= 86
                 AND alias_final_score >= 80
            THEN 'review'

            ELSE 'reject'
        END AS confidence_class
    FROM ranked
""")

print("✅ Table created: cds_alias_enrichment_ranked_v1")

print("\n--- Confidence class summary (top-ranked only) ---")
print(con.execute("""
    SELECT
        confidence_class,
        COUNT(*) AS n
    FROM cds_alias_enrichment_ranked_v1
    WHERE candidate_rank = 1
    GROUP BY confidence_class
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: top-ranked alias-assisted candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        lookup_name,
        lookup_source,
        release_id,
        main_artist_name,
        release_title,
        title_sim,
        alias_final_score,
        best_score,
        second_best_score,
        score_gap,
        confidence_class
    FROM cds_alias_enrichment_ranked_v1
    WHERE candidate_rank = 1
    ORDER BY alias_final_score DESC, score_gap DESC NULLS LAST
    LIMIT 30
""").fetchdf())

Connected ✅
Loading filtered alias-assisted candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded rows: 544,138
Scoring alias-assisted candidates...
Top candidates retained: 6737
✅ Table created: cds_alias_enrichment_ranked_v1

--- Confidence class summary (top-ranked only) ---
  confidence_class     n
0           reject  1807
1           review    23
2      auto_accept     9

--- Preview: top-ranked alias-assisted candidates ---
    cds_id                      artist                                          title           artist_display_name                 lookup_name   lookup_source  release_id  \
0   100136                   joan baez                                              5                     Joan Baez                   Joan Baez  canonical_name     5080207   
1   108936            paul baumgartner                                          piano              Paul Baumgartner            Paul Baumgartner  canonical_name     1458913   
2   103932                  zã ramalho                               grandes sucessos                    Zé Ramalho               

In [359]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Stricter alias-assisted automatic acceptance
# ------------------------------------------------------------
# Comments:
# - Keep only top-ranked alias candidates
# - Keep only auto_accept rows
# - Add an extra safeguard:
#     * reject titles that are too short
#     * reject titles that are numeric-only
print("Creating cds_alias_enrichment_auto_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_auto_v1 AS
    SELECT *
    FROM cds_alias_enrichment_ranked_v1
    WHERE candidate_rank = 1
      AND confidence_class = 'auto_accept'
      AND length(trim(coalesce(title, ''))) >= 4
      AND NOT regexp_matches(trim(coalesce(title, '')), '^[0-9]+$')
""")

print("✅ Table created: cds_alias_enrichment_auto_v1")

print("\n--- Count of stricter alias auto-accept rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_alias_enrichment_auto_v1
""").fetchdf())

print("\n--- Preview: stricter alias auto-accept rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        lookup_name,
        lookup_source,
        release_id,
        main_artist_name,
        release_title,
        title_sim,
        alias_final_score,
        score_gap
    FROM cds_alias_enrichment_auto_v1
    ORDER BY alias_final_score DESC, score_gap DESC NULLS LAST
""").fetchdf())

Connected ✅
Creating cds_alias_enrichment_auto_v1...
✅ Table created: cds_alias_enrichment_auto_v1

--- Count of stricter alias auto-accept rows ---
   n
0  8

--- Preview: stricter alias auto-accept rows ---
   cds_id            artist                      title artist_display_name       lookup_name   lookup_source  release_id       main_artist_name                                                  release_title  title_sim  \
0  108936  paul baumgartner                      piano    Paul Baumgartner  Paul Baumgartner  canonical_name     1458913  Johann Sebastian Bach  Pablo Casals interpreta Bach II: 2 sonatas para cello y piano      100.0   
1  103932        zã ramalho           grandes sucessos          Zé Ramalho        Zé Ramalho  canonical_name     1781355             Zé Ramalho                                            21 grandes sucessos      100.0   
2  105165              gang         le radici e le ali            The Gang              Gang      alias_name      284112        

In [360]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build a better alias-assisted automatic subset
# ------------------------------------------------------------
# Comments:
# - Do NOT exclude titles just because they contain words like
#   "live", "en vivo", or "greatest hits".
# - Some such titles are actually highly specific.
# - Instead, exclude only titles that are too short or purely numeric.
print("Creating cds_alias_enrichment_auto_v2_safe...")

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_auto_v2_safe AS
    SELECT *
    FROM cds_alias_enrichment_auto_v1
    WHERE
        -- title must not be too short
        length(trim(coalesce(title, ''))) >= 4

        -- title must not be purely numeric
        AND NOT regexp_matches(trim(coalesce(title, '')), '^[0-9]+$')

        -- title must not be just one very short token
        AND NOT (
            regexp_matches(trim(coalesce(title, '')), '^[A-Za-z0-9]+$')
            AND length(trim(coalesce(title, ''))) <= 5
        )
""")

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_auto_v2_review AS
    SELECT *
    FROM cds_alias_enrichment_auto_v1
    WHERE cds_id NOT IN (
        SELECT cds_id
        FROM cds_alias_enrichment_auto_v2_safe
    )
""")

print("✅ Tables created: cds_alias_enrichment_auto_v2_safe, cds_alias_enrichment_auto_v2_review")

print("\n--- Safe alias auto rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_alias_enrichment_auto_v2_safe
""").fetchdf())

print("\n--- Moved from auto to review ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_alias_enrichment_auto_v2_review
""").fetchdf())

print("\n--- Preview: safe alias auto rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        release_id,
        main_artist_name,
        release_title,
        title_sim,
        alias_final_score,
        score_gap
    FROM cds_alias_enrichment_auto_v2_safe
    ORDER BY alias_final_score DESC, score_gap DESC NULLS LAST
""").fetchdf())

Connected ✅
Creating cds_alias_enrichment_auto_v2_safe...
✅ Tables created: cds_alias_enrichment_auto_v2_safe, cds_alias_enrichment_auto_v2_review

--- Safe alias auto rows ---
   n
0  7

--- Moved from auto to review ---
   n
0  1

--- Preview: safe alias auto rows ---
   cds_id           artist                      title artist_display_name  release_id main_artist_name             release_title  title_sim  alias_final_score  score_gap
0  103932       zã ramalho           grandes sucessos          Zé Ramalho     1781355       Zé Ramalho       21 grandes sucessos      100.0               92.0  28.333333
1  105165             gang         le radici e le ali            The Gang      284112         The Gang        Le radici e le ali      100.0               90.5  42.500000
2  108945          vivaldi           the four seasons     Antonio Vivaldi     2951286  Antonio Vivaldi  Vivaldi the four seasons      100.0               90.5  15.937500
3  104297      luis kalaff                   mere

In [361]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Final expert split for alias-assisted high-confidence rows
# ------------------------------------------------------------
# Comments:
# - Even after strong filtering, some high-scoring rows still have
#   generic titles ("en vivo", "merengue", year-only style cases).
# - For the thesis, we keep only the most defensible rows as fully
#   automatic alias enrichments.
# - The rest remain as review candidates.
print("Creating cds_alias_enrichment_auto_v3_safe...")

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_auto_v3_safe AS
    SELECT *
    FROM cds_alias_enrichment_auto_v2_safe
    WHERE cds_id IN (103932, 105165, 108945, 105465)
""")

con.execute("""
    CREATE OR REPLACE TABLE cds_alias_enrichment_auto_v3_review AS
    SELECT *
    FROM cds_alias_enrichment_auto_v2_safe
    WHERE cds_id NOT IN (103932, 105165, 108945, 105465)
""")

print("✅ Tables created: cds_alias_enrichment_auto_v3_safe, cds_alias_enrichment_auto_v3_review")

print("\n--- Safe alias auto rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        release_id,
        main_artist_name,
        release_title,
        title_sim,
        alias_final_score,
        score_gap
    FROM cds_alias_enrichment_auto_v3_safe
    ORDER BY alias_final_score DESC, score_gap DESC NULLS LAST
""").fetchdf())

print("\n--- Alias rows moved to review ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        artist_display_name,
        release_id,
        main_artist_name,
        release_title,
        title_sim,
        alias_final_score,
        score_gap
    FROM cds_alias_enrichment_auto_v3_review
    ORDER BY alias_final_score DESC, score_gap DESC NULLS LAST
""").fetchdf())

Connected ✅
Creating cds_alias_enrichment_auto_v3_safe...
✅ Tables created: cds_alias_enrichment_auto_v3_safe, cds_alias_enrichment_auto_v3_review

--- Safe alias auto rows ---
   cds_id      artist                      title artist_display_name  release_id main_artist_name             release_title  title_sim  alias_final_score  score_gap
0  103932  zã ramalho           grandes sucessos          Zé Ramalho     1781355       Zé Ramalho       21 grandes sucessos      100.0               92.0  28.333333
1  105165        gang         le radici e le ali            The Gang      284112         The Gang        Le radici e le ali      100.0               90.5  42.500000
2  108945     vivaldi           the four seasons     Antonio Vivaldi     2951286  Antonio Vivaldi  Vivaldi the four seasons      100.0               90.5  15.937500
3  105465   the clash  cardiff live 22 july 1977           The Clash     2151661        The Clash                      1977      100.0               89.0  13.07692

In [393]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Safe merge into final mapping v9
# ------------------------------------------------------------
# Comments:
# - The previous UNION ALL failed because v8 and the alias table
#   did not have the same number of columns.
# - Here we:
#     1) read the exact schema of v8
#     2) map alias-safe columns into that schema
#     3) fill any missing v8 columns with NULL
# - This guarantees column-count compatibility.
print("Reading schemas...")

v8_schema = con.execute("PRAGMA table_info('cds_to_mb_final_mapping_v8')").fetchdf()
alias_schema = con.execute("PRAGMA table_info('cds_alias_enrichment_auto_v3_safe')").fetchdf()

v8_cols = v8_schema["name"].tolist()
alias_cols = set(alias_schema["name"].tolist())

print("\n--- v8 columns ---")
print(v8_cols)

print("\n--- alias-safe columns ---")
print(sorted(alias_cols))

# ------------------------------------------------------------
# Mapping from v8 columns -> alias-safe source expressions
# ------------------------------------------------------------
# Adjusted to your pipeline tables
column_mapping = {
    "cds_id": "a.cds_id",
    "artist": "a.artist",
    "title": "a.title",
    "cds_year": "a.cds_year",
    "release_id": "a.release_id",
    "release_title": "a.release_title",
    "main_artist_name": "a.main_artist_name",
    "artist_credit_full": "a.artist_credit_full",
    "total_tracks": "a.total_tracks",
    "barcode": "a.barcode",
    "mb_year": "a.mb_year",
    "match_type": "'alias_enrichment_safe'",
    "score": "a.alias_final_score",
}

# ------------------------------------------------------------
# Build the alias SELECT so it matches v8 exactly
# ------------------------------------------------------------
alias_select_parts = []

for col in v8_cols:
    if col in column_mapping:
        alias_select_parts.append(f"{column_mapping[col]} AS {col}")
    else:
        alias_select_parts.append(f"NULL AS {col}")

alias_select_sql = ",\n        ".join(alias_select_parts)

print("\nCreating cds_to_mb_final_mapping_v9...")

sql = f"""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v9 AS

    SELECT
        {", ".join(v8_cols)}
    FROM cds_to_mb_final_mapping_v8

    UNION ALL

    SELECT
        {alias_select_sql}
    FROM cds_alias_enrichment_auto_v3_safe a
    LEFT JOIN cds_to_mb_final_mapping_v8 x
      ON a.cds_id = x.cds_id
    WHERE x.cds_id IS NULL
"""

con.execute(sql)

print("✅ Table created: cds_to_mb_final_mapping_v9")

print("\n--- Coverage after alias-safe merge ---")
print(con.execute("""
    SELECT
        (SELECT COUNT(*) FROM cds) AS cds_total,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v8) AS matched_before,
        (SELECT COUNT(*) FROM cds_alias_enrichment_auto_v3_safe) AS alias_added_raw,
        (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v9) AS matched_after,
        (SELECT COUNT(*) FROM cds) - (SELECT COUNT(*) FROM cds_to_mb_final_mapping_v9) AS still_unmatched
""").fetchdf())

print("\n--- Match type breakdown in v9 ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v9
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview of newly added alias-safe rows ---")
print(con.execute("""
    SELECT *
    FROM cds_to_mb_final_mapping_v9
    WHERE match_type = 'alias_enrichment_safe'
    LIMIT 20
""").fetchdf()) 

Connected ✅
Reading schemas...

--- v8 columns ---
['cds_id', 'artist', 'title', 'cds_year', 'release_id', 'release_title', 'main_artist_name', 'artist_credit_full', 'total_tracks', 'barcode', 'mb_year', 'match_type', 'confidence_label', 'score', 'title_sim', 'artist_sim']

--- alias-safe columns ---
['alias_final_score', 'alias_type_id', 'artist', 'artist_credit_full', 'artist_display_name', 'artist_id', 'artist_key_clean', 'artist_sort_name', 'barcode', 'barcode_bonus', 'best_score', 'candidate_rank', 'cds_id', 'cds_year', 'confidence_class', 'curation_route', 'enrichment_priority_v6', 'locale', 'lookup_bonus', 'lookup_name', 'lookup_source', 'main_artist_name', 'mb_year', 'primary_for_locale', 'release_id', 'release_title', 'score_gap', 'second_best_score', 'title', 'title_key_clean', 'title_sim', 'total_tracks', 'year_bonus']

Creating cds_to_mb_final_mapping_v9...
✅ Table created: cds_to_mb_final_mapping_v9

--- Coverage after alias-safe merge ---
   cds_total  matched_before  ali

In [394]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build CDS-only enrichment records from still-unmatched rows
# ------------------------------------------------------------
# Comments:
# - This branch keeps clean CDS rows even if no confident MB link exists.
# - Since barcode is not available in your cds tables, we rely on:
#     * artist
#     * title
#     * cds_year
#     * tracks
#     * curation metadata
# - Goal:
#     produce a clean CDS-derived enrichment candidate table for thesis use.
# ------------------------------------------------------------

print("Creating cds_only_enrichment_candidates_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_candidates_v1 AS
    WITH base AS (
        SELECT
            s.cds_id,
            s.artist,
            s.title,
            s.pipeline_status,

            c.curation_route,
            c.enrichment_priority_v6,
            c.artist_key_clean,
            c.title_key_clean,
            c.cds_year,
            c.tracks,
            c.is_mojibake,
            c.is_various_artists,
            c.has_disc_cd_vol,
            c.is_generic_title,

            length(coalesce(c.artist_key_clean, '')) AS artist_key_len,
            length(coalesce(c.title_key_clean, '')) AS title_key_len
        FROM cds_enrichment_master_status s
        JOIN cds_unmatched_curated_v1 c
          ON s.cds_id = c.cds_id
        WHERE s.pipeline_status = 'still_unmatched'
    )
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        curation_route,
        enrichment_priority_v6,
        artist_key_clean,
        title_key_clean,

        CASE
            WHEN cds_year IS NOT NULL AND tracks IS NOT NULL AND trim(tracks) <> '' THEN 'metadata_plus_tracks'
            WHEN cds_year IS NOT NULL THEN 'metadata_only'
            ELSE 'title_artist_only'
        END AS cds_enrichment_type,

        'cds_only_enrichment' AS enrichment_source
    FROM base
    WHERE
        coalesce(is_mojibake, 0) = 0
        AND artist IS NOT NULL
        AND trim(artist) <> ''
        AND title IS NOT NULL
        AND trim(title) <> ''
        AND NOT (
            coalesce(artist_key_len, 0) < 3
            AND coalesce(title_key_len, 0) < 3
        )
""")

print("✅ Table created: cds_only_enrichment_candidates_v1")

print("\n--- CDS-only enrichment count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_candidates_v1
""").fetchdf())

print("\n--- Breakdown by enrichment type ---")
print(con.execute("""
    SELECT cds_enrichment_type, COUNT(*) AS n
    FROM cds_only_enrichment_candidates_v1
    GROUP BY cds_enrichment_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by curation route ---")
print(con.execute("""
    SELECT curation_route, COUNT(*) AS n
    FROM cds_only_enrichment_candidates_v1
    GROUP BY curation_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: CDS-only enrichment candidates ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_candidates_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_only_enrichment_candidates_v1...
✅ Table created: cds_only_enrichment_candidates_v1

--- CDS-only enrichment count ---
      n
0  4916

--- Breakdown by enrichment type ---
    cds_enrichment_type     n
0  metadata_plus_tracks  2488
1     title_artist_only  2428

--- Breakdown by curation route ---
              curation_route     n
0        standard_enrichment  3267
1      special_various_route   896
2  special_disc_cd_vol_route   563
3                      other   166
4         careful_enrichment    24

--- Preview: CDS-only enrichment candidates ---
    cds_id                        artist                                                                         title  cds_year  \
0   100004                       emanuel                                                                    felicidade      1998   
1   100007               various artists                                                                     nop pop 2      2002   
2   100009          

In [395]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Rebuild CDS-only enrichment scoring table with all needed fields
# ------------------------------------------------------------
# Comments:
# - The previous error happened because is_generic_title was not
#   carried into cds_only_enrichment_scored_v1.
# - We rebuild the scoring table from cds_only_enrichment_candidates_v1
#   and join back to cds_unmatched_curated_v1 to recover the missing
#   curation flags.
# - Then we create a professional 3-way split:
#     1) safe
#     2) review
#     3) weak
print("Rebuilding CDS-only enrichment scoring table...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_scored_v1 AS
    SELECT
        e.*,

        -- Recover curation flags from the curated unmatched table
        c.is_generic_title,
        c.is_various_artists,
        c.has_disc_cd_vol,
        c.is_mojibake,

        -- Basic key lengths
        length(coalesce(e.artist_key_clean, '')) AS artist_key_len,
        length(coalesce(e.title_key_clean, '')) AS title_key_len,

        -- ----------------------------------------------------
        -- Structural usability:
        -- keep only records with minimally informative artist/title
        -- ----------------------------------------------------
        CASE
            WHEN coalesce(c.is_mojibake, 0) = 1 THEN 0
            WHEN coalesce(e.artist_key_clean, '') = '' THEN 0
            WHEN coalesce(e.title_key_clean, '') = '' THEN 0
            WHEN length(coalesce(e.artist_key_clean, '')) < 3 THEN 0
            WHEN length(coalesce(e.title_key_clean, '')) < 4 THEN 0
            WHEN regexp_matches(trim(coalesce(e.title, '')), '^[0-9]+$') THEN 0
            ELSE 1
        END AS structurally_usable,

        -- ----------------------------------------------------
        -- Metadata strength:
        -- stronger if year and track-like info are present
        -- ----------------------------------------------------
        CASE
            WHEN e.cds_year IS NOT NULL
                 AND e.tracks IS NOT NULL
                 AND trim(coalesce(e.tracks, '')) <> ''
            THEN 2
            WHEN e.cds_year IS NOT NULL
            THEN 1
            ELSE 0
        END AS metadata_strength,

        -- ----------------------------------------------------
        -- Generic-title flag at scoring stage:
        -- catches generic titles even if source flag was not enough
        -- ----------------------------------------------------
        CASE
            WHEN lower(trim(coalesce(e.title, ''))) IN (
                'live', 'dance', 'piano', 'pod',
                'vol 1', 'vol 2', 'vol 3', 'vol 4',
                'cd1', 'cd2', 'cd3', 'cd4'
            ) THEN 1
            WHEN regexp_matches(lower(trim(coalesce(e.title, ''))), '^vol[ ]*[0-9]+$') THEN 1
            WHEN regexp_matches(lower(trim(coalesce(e.title, ''))), '^cd[ ]*[0-9]+$') THEN 1
            ELSE 0
        END AS generic_title_rule
    FROM cds_only_enrichment_candidates_v1 e
    LEFT JOIN cds_unmatched_curated_v1 c
      ON e.cds_id = c.cds_id
""")

print("✅ Table rebuilt: cds_only_enrichment_scored_v1")

# ------------------------------------------------------------
# Safe CDS-only enrichment
# ------------------------------------------------------------
# Comments:
# - High precision subset
# - Standard route only
# - High priority only
# - Strong metadata
# - No generic-title evidence
print("Creating cds_only_enrichment_safe_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_safe_v1 AS
    SELECT *
    FROM cds_only_enrichment_scored_v1
    WHERE
        structurally_usable = 1
        AND metadata_strength >= 2
        AND lower(coalesce(curation_route, '')) = 'standard_enrichment'
        AND lower(coalesce(enrichment_priority_v6, '')) = 'high'
        AND coalesce(is_generic_title, 0) = 0
        AND coalesce(generic_title_rule, 0) = 0
        AND coalesce(is_various_artists, 0) = 0
""")

# ------------------------------------------------------------
# Review CDS-only enrichment
# ------------------------------------------------------------
# Comments:
# - Still usable
# - But weaker metadata, special route, or more ambiguity
print("Creating cds_only_enrichment_review_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_review_v1 AS
    SELECT *
    FROM cds_only_enrichment_scored_v1
    WHERE
        structurally_usable = 1
        AND cds_id NOT IN (
            SELECT cds_id FROM cds_only_enrichment_safe_v1
        )
""")

# ------------------------------------------------------------
# Weak CDS-only enrichment
# ------------------------------------------------------------
# Comments:
# - Remaining rows not reliable enough for active enrichment use
print("Creating cds_only_enrichment_weak_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_weak_v1 AS
    SELECT *
    FROM cds_only_enrichment_scored_v1
    WHERE
        cds_id NOT IN (
            SELECT cds_id FROM cds_only_enrichment_safe_v1
        )
        AND cds_id NOT IN (
            SELECT cds_id FROM cds_only_enrichment_review_v1
        )
""")

print("✅ Tables created:")
print("   - cds_only_enrichment_safe_v1")
print("   - cds_only_enrichment_review_v1")
print("   - cds_only_enrichment_weak_v1")

print("\n--- CDS-only split summary ---")
print(con.execute("""
    SELECT 'safe' AS bucket, COUNT(*) AS n FROM cds_only_enrichment_safe_v1
    UNION ALL
    SELECT 'review' AS bucket, COUNT(*) AS n FROM cds_only_enrichment_review_v1
    UNION ALL
    SELECT 'weak' AS bucket, COUNT(*) AS n FROM cds_only_enrichment_weak_v1
""").fetchdf())

print("\n--- Breakdown of SAFE by enrichment type ---")
print(con.execute("""
    SELECT cds_enrichment_type, COUNT(*) AS n
    FROM cds_only_enrichment_safe_v1
    GROUP BY cds_enrichment_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: safe CDS-only enrichment rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_safe_v1
    LIMIT 30
""").fetchdf())

print("\n--- Preview: review CDS-only enrichment rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_review_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Rebuilding CDS-only enrichment scoring table...
✅ Table rebuilt: cds_only_enrichment_scored_v1
Creating cds_only_enrichment_safe_v1...
Creating cds_only_enrichment_review_v1...
Creating cds_only_enrichment_weak_v1...
✅ Tables created:
   - cds_only_enrichment_safe_v1
   - cds_only_enrichment_review_v1
   - cds_only_enrichment_weak_v1

--- CDS-only split summary ---
   bucket     n
0    safe  1652
1  review  3068
2    weak   196

--- Breakdown of SAFE by enrichment type ---
    cds_enrichment_type     n
0  metadata_plus_tracks  1652

--- Preview: safe CDS-only enrichment rows ---
    cds_id                                        artist                                     title  cds_year  \
0   100004                                       emanuel                                felicidade      1998   
1   100009                                        4 p m                                 sweet soul      2001   
2   100028                               jazz signatures          

In [396]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Final master participation/status table for thesis reporting
# ------------------------------------------------------------
# Comments:
# - This table summarizes the final outcome of the full pipeline.
# - Each CDS row is assigned to one final bucket:
#     1) matched_automatic
#     2) matched_review_candidate
#     3) cds_only_safe
#     4) cds_only_review
#     5) weak_or_excluded
#     6) unresolved
print("Creating cds_final_master_status_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_final_master_status_v1 AS
    WITH base AS (
        SELECT
            cds_id,
            artist,
            title
        FROM cds
    )
    SELECT
        b.cds_id,
        b.artist,
        b.title,

        CASE
            WHEN m.cds_id IS NOT NULL THEN 'matched_automatic'
            WHEN r.cds_id IS NOT NULL THEN 'matched_review_candidate'
            WHEN s.cds_id IS NOT NULL THEN 'cds_only_safe'
            WHEN rv.cds_id IS NOT NULL THEN 'cds_only_review'
            WHEN w.cds_id IS NOT NULL THEN 'weak_or_excluded'
            ELSE 'unresolved'
        END AS final_status

    FROM base b
    LEFT JOIN cds_to_mb_final_mapping_v9 m
      ON b.cds_id = m.cds_id
    LEFT JOIN cds_to_mb_review_candidates_v6_round2 r
      ON b.cds_id = r.cds_id
    LEFT JOIN cds_only_enrichment_safe_v1 s
      ON b.cds_id = s.cds_id
    LEFT JOIN cds_only_enrichment_review_v1 rv
      ON b.cds_id = rv.cds_id
    LEFT JOIN cds_only_enrichment_weak_v1 w
      ON b.cds_id = w.cds_id
""")

print("✅ Table created: cds_final_master_status_v1")

print("\n--- Final status summary ---")
print(con.execute("""
    SELECT
        final_status,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM cds), 2) AS pct
    FROM cds_final_master_status_v1
    GROUP BY final_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview: final master status table ---")
print(con.execute("""
    SELECT *
    FROM cds_final_master_status_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_final_master_status_v1...
✅ Table created: cds_final_master_status_v1

--- Final status summary ---
               final_status     n    pct
0         matched_automatic  4214  43.16
1           cds_only_review  3065  31.39
2             cds_only_safe  1652  16.92
3                unresolved   593   6.07
4          weak_or_excluded   196   2.01
5  matched_review_candidate    43   0.44

--- Preview: final master status table ---
    cds_id                artist                            title       final_status
0    10000       backstreet boys                       millennium  matched_automatic
1   100001               various      frankfurt trance vol 04 cd1  matched_automatic
2   100002             no return                  self mutilation  matched_automatic
3   100006      luv lite massive                         one love  matched_automatic
4   100008                  dmbq                             dmbq  matched_automatic
5   100010               various  

In [397]:
import duckdb
from pathlib import Path

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

export_dir = Path("../data_processed/exports/final_thesis_outputs")
export_dir.mkdir(parents=True, exist_ok=True)

tables_to_export = [
    "cds_to_mb_final_mapping_v9",
    "cds_only_enrichment_safe_v1",
    "cds_only_enrichment_review_v1",
    "cds_final_master_status_v1",
    "cds_only_enrichment_weak_v1"
]

for table in tables_to_export:
    out_path = export_dir / f"{table}.csv"
    con.execute(f"""
        COPY {table}
        TO '{out_path.as_posix()}'
        (HEADER, DELIMITER ',')
    """)
    print(f"✅ Exported: {out_path}")

print("\nDone.")

Connected ✅
✅ Exported: ..\data_processed\exports\final_thesis_outputs\cds_to_mb_final_mapping_v9.csv
✅ Exported: ..\data_processed\exports\final_thesis_outputs\cds_only_enrichment_safe_v1.csv
✅ Exported: ..\data_processed\exports\final_thesis_outputs\cds_only_enrichment_review_v1.csv
✅ Exported: ..\data_processed\exports\final_thesis_outputs\cds_final_master_status_v1.csv
✅ Exported: ..\data_processed\exports\final_thesis_outputs\cds_only_enrichment_weak_v1.csv

Done.


In [36]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build a thesis-friendly enriched version of the final mapping
# ------------------------------------------------------------
# Comments:
# - cds_to_mb_final_mapping_v9 is the core linkage table.
# - Here we join it back to the original CDS source table so the final
#   export also contains the CDS-side metadata:
#       cds_id, artist, title, cds_year, category, genre, tracks
# - This creates a denormalized output that is easier to read, export,
#   and use in the thesis.
print("Creating cds_to_mb_final_mapping_v9_enriched...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v9_enriched AS
    SELECT
        -- ----------------------------------------------------
        -- CDS-side source metadata
        -- ----------------------------------------------------
        c.cds_id,
        c.artist AS cds_artist,
        c.title AS cds_title,
        c.cds_year,
        c.category,
        c.genre,

        -- ----------------------------------------------------
        -- Final mapping fields
        -- ----------------------------------------------------
        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        m.match_type,
        m.score,
        c.tracks AS cds_tracklist,


    FROM cds_to_mb_final_mapping_v9 m
    LEFT JOIN cds c
      ON m.cds_id = c.cds_id
""")

print("✅ Table created: cds_to_mb_final_mapping_v9_enriched")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v9_enriched
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM cds_to_mb_final_mapping_v9_enriched
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_to_mb_final_mapping_v9_enriched...
✅ Table created: cds_to_mb_final_mapping_v9_enriched

--- Row count ---
      n
0  4214

--- Preview ---
    cds_id            cds_artist                      cds_title  cds_year  \
0    10000       backstreet boys                     millennium      <NA>   
1   100001               various    frankfurt trance vol 04 cd1      <NA>   
2   100002             no return                self mutilation      <NA>   
3   100006      luv lite massive                       one love      <NA>   
4   100008                  dmbq                           dmbq      2002   
5   100010               various    frankfurt trance vol 05 cd1      <NA>   
6   100012    rumania montevideo               mo better tracks      <NA>   
7   100015                 keedy               chase the clouds      <NA>   
8   100019            bill evans                          alone      1968   
9   100020      nature destroyed                         shades  

In [37]:
import duckdb
import os

# Connect to DuckDB
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# Make sure exports folder exists
os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_to_mb_final_mapping_v9_enriched.csv"

# ------------------------------------------------------------
# Export thesis-friendly enriched final mapping
# ------------------------------------------------------------
con.execute(f"""
    COPY cds_to_mb_final_mapping_v9_enriched
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

Connected ✅
✅ Exported: ../data_processed/exports/cds_to_mb_final_mapping_v9_enriched.csv


In [40]:
import duckdb
import pandas as pd
import re

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Heuristic mojibake detector
# ------------------------------------------------------------
MOJIBAKE_PATTERN = re.compile(r"Ã|Â|â€™|â€œ|â€|ï¿½")

def looks_mojibake(text):
    if text is None:
        return False
    text = str(text)
    return bool(MOJIBAKE_PATTERN.search(text))

# ------------------------------------------------------------
# Try to repair common UTF-8/Latin-1 mojibake
# ------------------------------------------------------------
def try_fix_mojibake(text):
    if text is None:
        return None

    s = str(text)

    if not looks_mojibake(s):
        return s

    candidates = []

    # Method 1: latin1 -> utf8
    try:
        candidates.append(s.encode("latin1").decode("utf-8"))
    except Exception:
        pass

    # Method 2: cp1252 -> utf8
    try:
        candidates.append(s.encode("cp1252").decode("utf-8"))
    except Exception:
        pass

    # Method 3: repeated fix (sometimes double-broken)
    try:
        c = s.encode("latin1").decode("utf-8")
        candidates.append(c.encode("latin1").decode("utf-8"))
    except Exception:
        pass

    # Keep only plausible improvements
    def score_candidate(orig, cand):
        score = 0

        # Fewer mojibake markers is better
        score += orig.count("Ã") - cand.count("Ã")
        score += orig.count("Â") - cand.count("Â")
        score += orig.count("ï¿½") - cand.count("ï¿½")

        # Presence of real accented / non-ASCII chars can be good
        if any(ord(ch) > 127 for ch in cand):
            score += 1

        # Empty or weirdly tiny outputs are bad
        if len(cand.strip()) < max(1, len(orig.strip()) // 3):
            score -= 5

        return score

    best = s
    best_score = 0

    for cand in candidates:
        if cand and cand != s:
            sc = score_candidate(s, cand)
            if sc > best_score:
                best = cand
                best_score = sc

    return best if best_score > 0 else s

# ------------------------------------------------------------
# Load CDS into pandas
# ------------------------------------------------------------
df = con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks
    FROM cds
""").fetchdf()

print(f"Loaded rows: {len(df):,}")

# ------------------------------------------------------------
# Apply safe repair
# ------------------------------------------------------------
for col in ["artist", "title", "tracks"]:
    df[f"{col}_fixed"] = df[col].apply(try_fix_mojibake)
    df[f"{col}_was_fixed"] = (
        df[col].fillna("").astype(str) != df[f"{col}_fixed"].fillna("").astype(str)
    ).astype(int)

# ------------------------------------------------------------
# Register repaired dataframe and create repaired table
# ------------------------------------------------------------
con.register("cds_mojibake_fix_df", df)

con.execute("""
    CREATE OR REPLACE TABLE cds_repaired_text AS
    SELECT
        cds_id,

        artist,
        artist_fixed,
        artist_was_fixed,

        title,
        title_fixed,
        title_was_fixed,

        tracks,
        tracks_fixed,
        tracks_was_fixed
    FROM cds_mojibake_fix_df
""")

print("✅ Table created: cds_repaired_text")

print("\n--- Repair summary ---")
print(con.execute("""
    SELECT
        SUM(artist_was_fixed) AS artist_fixed_rows,
        SUM(title_was_fixed) AS title_fixed_rows,
        SUM(tracks_was_fixed) AS tracks_fixed_rows
    FROM cds_repaired_text
""").fetchdf())

print("\n--- Sample repaired rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        artist_fixed,
        title,
        title_fixed
    FROM cds_repaired_text
    WHERE artist_was_fixed = 1
       OR title_was_fixed = 1
       OR tracks_was_fixed = 1
    LIMIT 30
""").fetchdf())

Connected ✅
Loaded rows: 9,763
✅ Table created: cds_repaired_text

--- Repair summary ---
   artist_fixed_rows  title_fixed_rows  tracks_fixed_rows
0                0.0               0.0                0.0

--- Sample repaired rows ---
Empty DataFrame
Columns: [cds_id, artist, artist_fixed, title, title_fixed]
Index: []


In [41]:
import duckdb
import pandas as pd
import re
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# broader mojibake detector
MOJIBAKE_PATTERN = re.compile(
    r"Ã.|Â.|â.|ð.|œ|ž|›|‹|€™|€œ|€|™|ï¿½"
)

def looks_mojibake(text):
    if text is None:
        return False
    s = str(text)
    return bool(MOJIBAKE_PATTERN.search(s))

def try_fix_mojibake(text):
    if text is None:
        return None

    s = str(text)

    if not looks_mojibake(s):
        return s

    candidates = []

    # most common fix
    for enc in ("latin1", "cp1252"):
        try:
            candidates.append(s.encode(enc).decode("utf-8"))
        except Exception:
            pass

    # second pass for doubly-broken text
    for enc in ("latin1", "cp1252"):
        try:
            c1 = s.encode(enc).decode("utf-8")
            candidates.append(c1.encode(enc).decode("utf-8"))
        except Exception:
            pass

    def badness(x):
        if x is None:
            return 10**9
        x = str(x)
        penalties = 0
        penalties += x.count("Ã") * 3
        penalties += x.count("Â") * 3
        penalties += x.count("â") * 3
        penalties += x.count("ï¿½") * 5
        return penalties

    best = s
    best_badness = badness(s)

    for cand in candidates:
        if cand and cand != s:
            b = badness(cand)
            if b < best_badness:
                best = cand
                best_badness = b

    return best

df = con.execute("""
    SELECT cds_id, artist, title, tracks
    FROM cds
""").fetchdf()

for col in ["artist", "title", "tracks"]:
    df[f"{col}_fixed"] = df[col].apply(try_fix_mojibake)
    df[f"{col}_was_fixed"] = (
        df[col].fillna("").astype(str) != df[f"{col}_fixed"].fillna("").astype(str)
    ).astype(int)

con.register("cds_mojibake_fix_df", df)

con.execute("""
    CREATE OR REPLACE TABLE cds_repaired_text AS
    SELECT
        cds_id,
        artist,
        artist_fixed,
        artist_was_fixed,
        title,
        title_fixed,
        title_was_fixed,
        tracks,
        tracks_fixed,
        tracks_was_fixed
    FROM cds_mojibake_fix_df
""")

print("✅ Table recreated: cds_repaired_text")

print("\n--- Repair summary ---")
print(con.execute("""
    SELECT
        SUM(artist_was_fixed) AS artist_fixed_rows,
        SUM(title_was_fixed) AS title_fixed_rows,
        SUM(tracks_was_fixed) AS tracks_fixed_rows
    FROM cds_repaired_text
""").fetchdf())

print("\n--- Sample repaired rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        artist_fixed,
        title,
        title_fixed
    FROM cds_repaired_text
    WHERE artist_was_fixed = 1
       OR title_was_fixed = 1
       OR tracks_was_fixed = 1
    LIMIT 30
""").fetchdf())



Connected ✅
✅ Table recreated: cds_repaired_text

--- Repair summary ---
   artist_fixed_rows  title_fixed_rows  tracks_fixed_rows
0                0.0               0.0                0.0

--- Sample repaired rows ---
Empty DataFrame
Columns: [cds_id, artist, artist_fixed, title, title_fixed]
Index: []


In [42]:
print(con.execute("""
    SELECT cds_id, artist, title
    FROM cds
    WHERE artist LIKE '%Ã%'
       OR artist LIKE '%Â%'
       OR title  LIKE '%Ã%'
       OR title  LIKE '%Â%'
    LIMIT 20
""").fetchdf())

Empty DataFrame
Columns: [cds_id, artist, title]
Index: []


In [43]:
broken = con.execute("""
    SELECT cds_id, artist, title, tracks
    FROM cds
    WHERE artist LIKE '%Ã%'
       OR artist LIKE '%Â%'
       OR title  LIKE '%Ã%'
       OR title  LIKE '%Â%'
       OR tracks LIKE '%Ã%'
       OR tracks LIKE '%Â%'
    LIMIT 20
""").fetchdf()

broken

,cds_id,artist,title,tracks


In [44]:
broken = con.execute(r"""
    SELECT cds_id, artist, title, tracks
    FROM cds
    WHERE regexp_matches(coalesce(artist, ''), 'Ã|Â|â|ï¿½')
       OR regexp_matches(coalesce(title,  ''), 'Ã|Â|â|ï¿½')
       OR regexp_matches(coalesce(tracks, ''), 'Ã|Â|â|ï¿½')
    LIMIT 20
""").fetchdf()

broken

,cds_id,artist,title,tracks
0,100003,ã â â ã â ã â ºâ ã â å,ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ,ã â ºâ ã â â ã â šå ã â žã â ã â ¹ã â â ã â ã ...
1,100005,ã å â ã â ºâ ã â ºâ ºã â â,catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½...,ã æ â ã æ â ã æ âªã æ â¼ ã â šâªã æ â¼ã æ â ã ...
2,100011,â¾ã â â ã ãœ worship,child praise vol 1,ã ã â ã â³âªâ â â ã â â â½ãƒâ ã â ã ã ãš â³âªâ...
3,100014,modern talking,univers,tv makes the superstar iâ m no rockefeller mys...
4,100024,175r,songs,inst ã â â ã âºâº ã æ â œã æ â ã æ â¼ã æ â ã â...
5,100026,day after tomorrow,elements,high spirit time of destiny ã â šã â šå ã âµã ...
6,100027,v a,ãƒã âºâ¹ cd 09,â ã â ã ã ã ã ãœ â¼â â â³â ãœ ã ã â ã â³âªâ â ...
7,100030,ã â â ã â ã â â¾ã âµ,disc13 ã å â ã â ã â ¹ã â šå,ã å â ã â â ã â â ã â â ã â â½ã â ã â šã å ã â...
8,100034,ã â šã å â ã å å ã â â¾,lucy,lucy ã æ å¾ã æ â ã â šâ ã æ ã â šâ ã â šâ¹ã æ ...
9,100042,deux,forever,intor deux forever â³âªâ â âµâ¹â¾ã âºã â¾ã â ã...


In [45]:
print(con.execute(r"""
    SELECT artist
    FROM cds
    WHERE regexp_matches(coalesce(artist, ''), 'Ã|Â|â|ï¿½')
    LIMIT 20
""").fetchdf())

                                             artist
0                           ã â â ã â ã â ºâ ã â å 
1                       ã å â ã â ºâ ã â ºâ ºã â â 
2                              â¾ã â â ã ãœ worship
3                              ã â â ã â ã â â¾ã âµ
4                           ã â šã å â ã å å ã â â¾
5                   ã â â ã â å½ã â¾ã â ã â šë ã â 
6                             ã ë â ã å â ã â šã â 
7                            ã å â ã â²â ã â â ã â 
8                              â â ã ã â½âºã ã ã â 
9                                  â ã ã ãÿâ ã ãˆâ¹
10         â¼â¼â¹ãœã ã â¼ã â â â â¹â¾ã â¼â â â³â ãœ
11                             ã â ã â ã â¾ã â ã â 
12                                      â ã â ã â j
13                             â â¹â¼ã ã ã¼âµâµâ ãœ
14                                          â½â²âµ 
15  ãƒâ â dâ â ã ã ã â¼â â ã ã ã âªâ â ã ãžâºâ ã â 
16                                       â³â â ã â 
17                     ã â â ã â â¾ã ë ã â â ã â â¹
18          

In [46]:
print(con.execute(r"""
    SELECT title
    FROM cds
    WHERE regexp_matches(coalesce(title, ''), 'Ã|Â|â|ï¿½')
    LIMIT 20
""").fetchdf())

                                                title
0               ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ 
1   catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½...
2                                     ãƒã âºâ¹ cd 09 
3                       disc13 ã å â ã â ã â ¹ã â šå 
4      super live ã â â ã å â ã â â ã â œã â â disc 1
5           â ãµâºâ â ã âµâµâ â ã â ã ã â ã ã â½â¾ã 6
6                                â¼â¼â¹ãœã ã â¼ã â â 
7                ã â â ã âºâºã â â¾ã â å ã â â ã âºå 
8             ã â â ã â³â ã â ã â šâ ã song library 1
9                                     â sâ â â ã â ãº
10                                           ã ã â ã 
11  euro darkness 1 ã â å the early days of sebã â å 
12            â½â²âµ âªã â ã¹â lâ â³ã¼â â ã â²pâºqâ k
13                               ã ã â âºâµã ã ã â ã 
14            ich mã chtâ so gern dein teddybã r sein
15                                            â ã¼â v
16                                     ã å ë ã â ã â 
17                          

In [47]:
print(con.execute("""
    SELECT cds_id, artist, title, tracks
    FROM cds
    WHERE cds_id = 100257
""").fetchdf())

   cds_id  artist title                                             tracks
0  100257  yarkin   ten  bulmaca rã ngala ã ster 1 kayip kã mlã k vakas...


In [48]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

test_df = con.execute("""
    SELECT cds_id, artist, title, tracks
    FROM cds
    WHERE cds_id IN (100003, 100005, 100011, 100257)
""").fetchdf()

test_df

Connected ✅


,cds_id,artist,title,tracks
0,100003,ã â â ã â ã â ºâ ã â å,ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ,ã â ºâ ã â â ã â šå ã â žã â ã â ¹ã â â ã â ã ...
1,100005,ã å â ã â ºâ ã â ºâ ºã â â,catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½...,ã æ â ã æ â ã æ âªã æ â¼ ã â šâªã æ â¼ã æ â ã ...
2,100011,â¾ã â â ã ãœ worship,child praise vol 1,ã ã â ã â³âªâ â â ã â â â½ãƒâ ã â ã ã ãš â³âªâ...
3,100257,yarkin,ten,bulmaca rã ngala ã ster 1 kayip kã mlã k vakas...


In [49]:
def try_fix_chain(s):
    if s is None:
        return None

    s = str(s)
    out = {"original": s}

    # attempt 1
    try:
        out["latin1_to_utf8"] = s.encode("latin1").decode("utf-8")
    except Exception:
        out["latin1_to_utf8"] = None

    # attempt 2
    try:
        out["cp1252_to_utf8"] = s.encode("cp1252").decode("utf-8")
    except Exception:
        out["cp1252_to_utf8"] = None

    # attempt 3
    try:
        x = s.encode("latin1").decode("utf-8")
        out["double_latin1_to_utf8"] = x.encode("latin1").decode("utf-8")
    except Exception:
        out["double_latin1_to_utf8"] = None

    # attempt 4
    try:
        x = s.encode("cp1252").decode("utf-8")
        out["double_cp1252_to_utf8"] = x.encode("cp1252").decode("utf-8")
    except Exception:
        out["double_cp1252_to_utf8"] = None

    return out

for _, row in test_df.iterrows():
    print("\n" + "="*80)
    print("cds_id:", row["cds_id"])

    for col in ["artist", "title", "tracks"]:
        print(f"\n--- {col} ---")
        fixes = try_fix_chain(row[col])
        for k, v in fixes.items():
            print(f"{k}: {v}")


cds_id: 100003

--- artist ---
original: ã â â ã â ã â ºâ ã â å 
latin1_to_utf8: None
cp1252_to_utf8: None
double_latin1_to_utf8: None
double_cp1252_to_utf8: None

--- title ---
original: ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ 
latin1_to_utf8: None
cp1252_to_utf8: None
double_latin1_to_utf8: None
double_cp1252_to_utf8: None

--- tracks ---
original: ã â ºâ ã â â ã â šå ã â žã â ã â ¹ã â â ã â ã â¼å ã â ã å ã âªã â šâ ã â â ã â ã âªã â žã â â ã å ã â ã å ã â ã å ã â žã â ž ã âµã â šå ã â šã â ž ã â šã â š ã â ã ëœâ ã â âºã å ã â ã â ã â â ã â â½ã â žã â â ã å å¾ã â â ã â ã â ã â šâ ã â žã â â šã â â ã â å ã â å ã ë â ã â å ã â šã âªã å ã â šâ ã â žâ ºã â ã â šâ ¹ã â ã â ã ëœâ ã â²â ã â 
latin1_to_utf8: None
cp1252_to_utf8: None
double_latin1_to_utf8: None
double_cp1252_to_utf8: None

cds_id: 100005

--- artist ---
original: ã å â ã â ºâ ã â ºâ ºã â â 
latin1_to_utf8: None
cp1252_to_utf8: None
double_latin1_to_utf8: None
double_cp1252_to_utf8: None

--- title ---
original: catsã â â ã â š

In [50]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

con.execute("""
    CREATE OR REPLACE TABLE cds_tracks_cleaned_v1 AS
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        category,
        genre,
        tracks AS tracks_raw,

        trim(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            coalesce(tracks, ''),
                            '[\\r\\n\\t]+', ' ', 'g'
                        ),
                        '\\s+', ' ', 'g'
                    ),
                    '[,;|/]{2,}', ' ', 'g'
                ),
                '\\s*([,;|/])\\s*', ' \\1 ', 'g'
            )
        ) AS tracks_clean

    FROM cds
""")

print("✅ Table created: cds_tracks_cleaned_v1")
print(con.execute("""
    SELECT cds_id, artist, title, tracks_raw, tracks_clean
    FROM cds_tracks_cleaned_v1
    LIMIT 10
""").fetchdf())

Connected ✅
✅ Table created: cds_tracks_cleaned_v1
   cds_id                       artist  \
0   10000              backstreet boys   
1  100001                      various   
2  100002                    no return   
3  100003      ã â â ã â ã â ºâ ã â å    
4  100004                      emanuel   
5  100005  ã å â ã â ºâ ã â ºâ ºã â â    
6  100006             luv lite massive   
7  100007              various artists   
8  100008                         dmbq   
9  100009                       4 p m    

                                               title  \
0                                         millennium   
1                        frankfurt trance vol 04 cd1   
2                                    self mutilation   
3              ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ    
4                                         felicidade   
5  catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½...   
6                                           one love   
7                                     

In [52]:
import duckdb
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_only_enrichment_candidates_v1.csv"

con.execute(f"""
    COPY cds_only_enrichment_candidates_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_candidates_v1
""").fetchdf())

Connected ✅
✅ Exported: ../data_processed/exports/cds_only_enrichment_candidates_v1.csv

--- Row count ---
      n
0  4916


In [53]:
import duckdb
import os

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_unmatched_curated_v1.csv"

con.execute(f"""
    COPY cds_unmatched_curated_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_unmatched_curated_v1
""").fetchdf())

Connected ✅
✅ Exported: ../data_processed/exports/cds_unmatched_curated_v1.csv

--- Row count ---
      n
0  5015


In [54]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks_raw,
        tracks_clean
    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
    LIMIT 50
""").fetchdf())

Connected ✅
    cds_id                           artist  \
0    10000                  backstreet boys   
1   100002                        no return   
2   100003          ã â â ã â ã â ºâ ã â å    
3   100004                          emanuel   
4   100005      ã å â ã â ºâ ã â ºâ ºã â â    
5   100006                 luv lite massive   
6   100007                  various artists   
7   100008                             dmbq   
8   100009                           4 p m    
9   100010                          various   
10  100011             â¾ã â â ã ãœ worship   
11  100012               rumania montevideo   
12  100014                   modern talking   
13  100015                            keedy   
14  100016                                    
15  100017                          various   
16  100019                       bill evans   
17  100020                 nature destroyed   
18  100021                        pearl jam   
19  100022                dingdong avanzado   
2

In [55]:
print(con.execute("""
    SELECT COUNT(*) AS changed_rows
    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
""").fetchdf())

   changed_rows
0          9757


In [56]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks_raw,
        tracks_clean,

        CASE
            WHEN tracks_raw IS NULL AND tracks_clean IS NULL THEN 'no change'
            WHEN coalesce(tracks_raw, '') = coalesce(tracks_clean, '') THEN 'no change'
            WHEN regexp_replace(coalesce(tracks_raw, ''), '[\\r\\n\\t]+', ' ', 'g') <> coalesce(tracks_raw, '')
                 THEN 'newline/tab cleanup'
            WHEN regexp_replace(coalesce(tracks_raw, ''), '\\s+', ' ', 'g') <> coalesce(tracks_raw, '')
                 THEN 'extra whitespace cleanup'
            WHEN regexp_replace(coalesce(tracks_raw, ''), '[,;|/]{2,}', ' ', 'g') <> coalesce(tracks_raw, '')
                 THEN 'repeated separator cleanup'
            ELSE 'general normalization'
        END AS likely_change_type,

        length(coalesce(tracks_raw, '')) AS raw_len,
        length(coalesce(tracks_clean, '')) AS clean_len

    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
    LIMIT 50
""").fetchdf())

Connected ✅
    cds_id                           artist  \
0    10000                  backstreet boys   
1   100002                        no return   
2   100003          ã â â ã â ã â ºâ ã â å    
3   100004                          emanuel   
4   100005      ã å â ã â ºâ ã â ºâ ºã â â    
5   100006                 luv lite massive   
6   100007                  various artists   
7   100008                             dmbq   
8   100009                           4 p m    
9   100010                          various   
10  100011             â¾ã â â ã ãœ worship   
11  100012               rumania montevideo   
12  100014                   modern talking   
13  100015                            keedy   
14  100016                                    
15  100017                          various   
16  100019                       bill evans   
17  100020                 nature destroyed   
18  100021                        pearl jam   
19  100022                dingdong avanzado   
2

In [57]:
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks_raw,
        tracks_clean
    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
    LIMIT 20
""").fetchdf())

    cds_id                       artist  \
0    10000              backstreet boys   
1   100002                    no return   
2   100003      ã â â ã â ã â ºâ ã â å    
3   100004                      emanuel   
4   100005  ã å â ã â ºâ ã â ºâ ºã â â    
5   100006             luv lite massive   
6   100007              various artists   
7   100008                         dmbq   
8   100009                       4 p m    
9   100010                      various   
10  100011         â¾ã â â ã ãœ worship   
11  100012           rumania montevideo   
12  100014               modern talking   
13  100015                        keedy   
14  100016                                
15  100017                      various   
16  100019                   bill evans   
17  100020             nature destroyed   
18  100021                    pearl jam   
19  100022            dingdong avanzado   

                                                title  \
0                                      

In [58]:
import duckdb
import pandas as pd
import difflib

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

df = con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks_raw,
        tracks_clean
    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
    LIMIT 20
""").fetchdf()

def word_diff(a, b):
    a_words = str(a).split()
    b_words = str(b).split()
    diff = difflib.ndiff(a_words, b_words)
    return " ".join(diff)

df["diff"] = df.apply(lambda r: word_diff(r["tracks_raw"], r["tracks_clean"]), axis=1)

pd.set_option("display.max_colwidth", 1000)
df[["cds_id", "artist", "title", "diff"]]

Connected ✅


,cds_id,artist,title,diff
0,10000,backstreet boys,millennium,larger than life i want it that way show me the meaning of being lonely it s gotta be you i need you tonight don t want you back don t wanna lose you now the one back to your heart spanish eyes no one else comes close the perfect fan i ll be there for you
1,100002,no return,self mutilation,do or die truth and reality lost soul extractor sadistic desire the true way fanatic mind individualistic ideal one life trail of blood sect
2,100003,ã â â ã â ã â ºâ ã â å,ã æ â³ã â žã â âºã â ã â ¹ã â ã â šâ,ã â ºâ ã â â ã â šå ã â žã â ã â ¹ã â â ã â ã â¼å ã â ã å ã âªã â šâ ã â â ã â ã âªã â žã â â ã å ã â ã å ã â ã å ã â žã â ž ã âµã â šå ã â šã â ž ã â šã â š ã â ã ëœâ ã â âºã å ã â ã â ã â â ã â â½ã â žã â â ã å å¾ã â â ã â ã â ã â šâ ã â žã â â šã â â ã â å ã â å ã ë â ã â å ã â šã âªã å ã â šâ ã â žâ ºã â ã â šâ ¹ã â ã â ã ëœâ ã â²â ã â
3,100004,emanuel,felicidade,felicidade quando o telefone toca vem bailar o tic tic quero que sejas minha e de mais ninguem eu sei que me amas o melhor que hã minha vizinha deixa me a bater mal s joã o ã foliã o sã³ quero o teu carinho tudo farei para ter a tua paixã o serã s sempre minha vem bailar o tic tic verã ã o dance mix
4,100005,ã å â ã â ºâ ã â ºâ ºã â â,catsã â â ã â šâªã æ âªã â šâ ã æ å ã æ â ã â½â ã â šâ ã æ â ã â šâ¹ã æ ë,ã æ â ã æ â ã æ âªã æ â¼ ã â šâªã æ â¼ã æ â ã â šâ ã æ ã æ â ã â šâ ã â šâ ã â šâ ã æ âªã â šâ ã æ â ã â šâ½ã æ â³ã â šâ ã æ ã æ â¼ã æ å ã æ â³ã â šâ ã æ â ã â šâªã æ â ã æ â ã â šâ ã æ â ã æ æ ã æ â ž ã ë â ã å â ã â ã ã â šâ ã â šâ ã æ â ¹ã â šâ ã æ â ¹ã æ â ã æ æ ã æ â ž ã ë â ã å ã â ã â ã â šâ œã å â ã æ â ã æ â ã æ â ã â šâ ã æ â ã æ â ã â šâ ã â šâ ã æ â¼ ã ë â ã â ã â ã â ã â šå ã å â ã â šâ ã æ âªã â šâ ã æ â ã æ â ã ë â ã â â¼ã â â ã å â ã æ ã â šâ¹ã æ ë ã æ â ã â šâ ã æ â¼ã â šâ ã æ â ã æ â¼ã æ â³ã â šâº ã ë â ã...
5,100006,luv lite massive,one love,temple of dehli from naw till a mornin shock afghani lick you bad freestyle fiki fuko i m here yeah ww5 one love
6,100007,various artists,nop pop 2,golden boy miss kittin rippin kittin norther lite treat me better peaches lovertits mã rtini brã s flash kristall das leben im jetzt slam lifetimes autotune capital killers e man animal skin marc verbos special friend lasse lundmark coca bass flash und gordon frauen in autos dan maxem vs polygen tiefoderhoch green velvet la la land frank mã¼ller vs takkyu ishino
7,100008,dmbq,dmbq,small hours rock a cola ohh baby teenage limbo little leaf i movement little leaf ii swell little leaf iii etude sayonara no season altered rock can peace migrate to midnight e means eternal data track
8,100009,4 p m,sweet soul,miagetegoran yoruno hoshio woman cherish tears in heaven so much in love christmas eve too much heaven i left my heart in san francisco the lady in my life christmas medley the star spangled banner
9,100010,various,frankfurt trance vol 05 cd1,dj tom stevens violet steam single cut liquid child diving faces original mix vimana we came edit lac terra after dark fridge vs vectrex remix steve baltes thoughts original mix star party i m in love ferry corsten robert smit rmx gate iv give all you got radio edit dj stigma baby sunshine stig s original club mix synchronic minds all i need club mix astral inc remember green court edit x trance friends 2000 tom stevens rmx ravelab send me an angel club mix


In [59]:
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks_raw,
        tracks_clean
    FROM cds_tracks_cleaned_v1
    WHERE coalesce(tracks_raw, '') <> coalesce(tracks_clean, '')
    LIMIT 20
""").fetchdf())

    cds_id                       artist  \
0    10000              backstreet boys   
1   100002                    no return   
2   100003      ã â â ã â ã â ºâ ã â å    
3   100004                      emanuel   
4   100005  ã å â ã â ºâ ã â ºâ ºã â â    
5   100006             luv lite massive   
6   100007              various artists   
7   100008                         dmbq   
8   100009                       4 p m    
9   100010                      various   
10  100011         â¾ã â â ã ãœ worship   
11  100012           rumania montevideo   
12  100014               modern talking   
13  100015                        keedy   
14  100016                                
15  100017                      various   
16  100019                   bill evans   
17  100020             nature destroyed   
18  100021                    pearl jam   
19  100022            dingdong avanzado   

                                                                         title  \
0             

In [60]:
#missing tracks 
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT COUNT(*) AS missing_tracks
    FROM cds
    WHERE tracks IS NULL
       OR trim(tracks) = ''
""").fetchdf())

Connected ✅
   missing_tracks
0               1


In [61]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT COUNT(*) AS missing_artist
    FROM cds
    WHERE artist IS NULL
       OR trim(artist) = ''
""").fetchdf())

Connected ✅
   missing_artist
0               7


In [62]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT COUNT(*) AS missing_title
    FROM cds
    WHERE title IS NULL
       OR trim(title) = ''
""").fetchdf())

Connected ✅
   missing_title
0             11


In [63]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT COUNT(*) AS missing_year
    FROM cds
    WHERE cds_year IS NULL
       OR trim(title) = ''
""").fetchdf())

Connected ✅
   missing_year
0          4561


In [64]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        COUNT(*) AS total_weak_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows,
        SUM(CASE WHEN is_mojibake = 0 THEN 1 ELSE 0 END) AS non_mojibake_rows,
        ROUND(
            100.0 * SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS mojibake_pct
    FROM cds_only_enrichment_weak_v1
""").fetchdf())

Connected ✅
   total_weak_rows  mojibake_rows  non_mojibake_rows  mojibake_pct
0              196            0.0              196.0           0.0


In [67]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_unmatched_working_v6
""").fetchdf())

Connected ✅
   total_rows  mojibake_rows
0        5555          509.0


In [68]:
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        is_mojibake
    FROM cds_unmatched_working_v6
    WHERE is_mojibake = 1
    LIMIT 30
""").fetchdf())

    cds_id                                    artist  \
0   100003                   ã â â ã â ã â ºâ ã â å    
1   100005               ã å â ã â ºâ ã â ºâ ºã â â    
2   100011                      â¾ã â â ã ãœ worship   
3   100014                            modern talking   
4   100027                                       v a   
5   100030                      ã â â ã â ã â â¾ã âµ   
6   100034                   ã â šã å â ã å å ã â â¾   
7   100042                                      deux   
8   100043                                      v a    
9   100051           ã â â ã â å½ã â¾ã â ã â šë ã â    
10  100057                     ã ë â ã å â ã â šã â    
11  100059                    ã å â ã â²â ã â â ã â    
12  100067                      â â ã ã â½âºã ã ã â    
13  100071                          â ã ã ãÿâ ã ãˆâ¹   
14  100074                            ã ã ã ã ã ã ãª   
15  100083  â¼â¼â¹ãœã ã â¼ã â â â â¹â¾ã â¼â â â³â ãœ   
16  100096                                      

In [69]:
#trace mojibake stage by stage
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

tables = [
    "cds_unmatched_after_v6",
    "cds_unmatched_profiled_v6",
    "cds_unmatched_working_v6",
    "cds_unmatched_curated_v1",
    "cds_only_enrichment_candidates_v1",
    "cds_only_enrichment_scored_v1",
    "cds_only_enrichment_safe_v1",
    "cds_only_enrichment_review_v1",
    "cds_only_enrichment_weak_v1"
]

rows = []
for t in tables:
    try:
        q = f"""
            SELECT
                '{t}' AS table_name,
                COUNT(*) AS total_rows,
                SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows,
                ROUND(
                    100.0 * SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) / COUNT(*),
                    2
                ) AS mojibake_pct
            FROM {t}
        """
        rows.append(con.execute(q).fetchdf())
    except Exception as e:
        print(f"Skipped {t}: {e}")

result = rows[0]
for r in rows[1:]:
    result = result._append(r, ignore_index=True)

print(result)

Connected ✅
Skipped cds_unmatched_after_v6: Binder Error: Referenced column "is_mojibake" not found in FROM clause!
Candidate bindings: "artist_key", "title_key", "artist_clean", "cds_year", "artist"

LINE 5:                 SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows,
                                      ^
Skipped cds_only_enrichment_candidates_v1: Binder Error: Referenced column "is_mojibake" not found in FROM clause!
Candidate bindings: "artist_key_clean", "cds_year", "curation_route", "title_key_clean", "artist"

LINE 5:                 SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows,
                                      ^
                      table_name  total_rows  mojibake_rows  mojibake_pct
0      cds_unmatched_profiled_v6        5555          509.0          9.16
1       cds_unmatched_working_v6        5555          509.0          9.16
2       cds_unmatched_curated_v1        5015            0.0          0.00
3  cds_only_enrichment_scor

In [70]:
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        tracks,
        is_mojibake
    FROM cds_unmatched_working_v6
    WHERE is_mojibake = 1
    LIMIT 30
""").fetchdf())

    cds_id                                    artist  \
0   100003                   ã â â ã â ã â ºâ ã â å    
1   100005               ã å â ã â ºâ ã â ºâ ºã â â    
2   100011                      â¾ã â â ã ãœ worship   
3   100014                            modern talking   
4   100027                                       v a   
5   100030                      ã â â ã â ã â â¾ã âµ   
6   100034                   ã â šã å â ã å å ã â â¾   
7   100042                                      deux   
8   100043                                      v a    
9   100051           ã â â ã â å½ã â¾ã â ã â šë ã â    
10  100057                     ã ë â ã å â ã â šã â    
11  100059                    ã å â ã â²â ã â â ã â    
12  100067                      â â ã ã â½âºã ã ã â    
13  100071                          â ã ã ãÿâ ã ãˆâ¹   
14  100074                            ã ã ã ã ã ã ãª   
15  100083  â¼â¼â¹ãœã ã â¼ã â â â â¹â¾ã â¼â â â³â ãœ   
16  100096                                      

In [71]:
import duckdb

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print(con.execute("""
    SELECT
        'cds_unmatched_profiled_v6' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_unmatched_profiled_v6

    UNION ALL

    SELECT
        'cds_unmatched_working_v6' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_unmatched_working_v6

    UNION ALL

    SELECT
        'cds_unmatched_curated_v1' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_unmatched_curated_v1

    UNION ALL

    SELECT
        'cds_only_enrichment_scored_v1' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_only_enrichment_scored_v1

    UNION ALL

    SELECT
        'cds_only_enrichment_safe_v1' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_only_enrichment_safe_v1

    UNION ALL

    SELECT
        'cds_only_enrichment_review_v1' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_only_enrichment_review_v1

    UNION ALL

    SELECT
        'cds_only_enrichment_weak_v1' AS stage,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_mojibake = 1 THEN 1 ELSE 0 END) AS mojibake_rows
    FROM cds_only_enrichment_weak_v1
""").fetchdf())

Connected ✅
                           stage  total_rows  mojibake_rows
0      cds_unmatched_profiled_v6        5555          509.0
1       cds_unmatched_working_v6        5555          509.0
2       cds_unmatched_curated_v1        5015            0.0
3  cds_only_enrichment_scored_v1        4916            0.0
4    cds_only_enrichment_safe_v1        1652            0.0
5  cds_only_enrichment_review_v1        3068            0.0
6    cds_only_enrichment_weak_v1         196            0.0


In [72]:
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        is_mojibake,
        exclusion_reason
    FROM cds_unmatched_excluded_v1
    WHERE exclusion_reason = 'unrecoverable_mojibake'
    LIMIT 30
""").fetchdf())

    cds_id                                    artist  \
0   100003                   ã â â ã â ã â ºâ ã â å    
1   100005               ã å â ã â ºâ ã â ºâ ºã â â    
2   100011                      â¾ã â â ã ãœ worship   
3   100014                            modern talking   
4   100027                                       v a   
5   100030                      ã â â ã â ã â â¾ã âµ   
6   100034                   ã â šã å â ã å å ã â â¾   
7   100042                                      deux   
8   100043                                      v a    
9   100051           ã â â ã â å½ã â¾ã â ã â šë ã â    
10  100057                     ã ë â ã å â ã â šã â    
11  100059                    ã å â ã â²â ã â â ã â    
12  100067                      â â ã ã â½âºã ã ã â    
13  100071                          â ã ã ãÿâ ã ãˆâ¹   
14  100074                            ã ã ã ã ã ã ãª   
15  100083  â¼â¼â¹ãœã ã â¼ã â â â â¹â¾ã â¼â â â³â ãœ   
16  100096                                      

In [73]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Clean mojibake only in cds_tracklist inside the enriched final table
# ------------------------------------------------------------
# Comments:
# - We do NOT delete rows.
# - We only blank the cds_tracklist field when it looks unreadable.
# - The rule below is heuristic:
#     1) obvious mojibake marker sequences like "ã", "â", "ð", "�"
#     2) repeated corrupted-looking fragments
# - If flagged, we replace cds_tracklist with NULL.
print("Creating cds_to_mb_final_mapping_v9_enriched_cleantrack_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE cds_to_mb_final_mapping_v9_enriched_cleantrack_v1 AS
    SELECT
        * EXCLUDE (cds_tracklist),

        CASE
            WHEN cds_tracklist IS NULL OR trim(cds_tracklist) = '' THEN NULL

            -- obvious mojibake markers
            WHEN regexp_matches(lower(cds_tracklist), '(ã|â|ð|�)') THEN NULL

            -- common broken UTF-8 / Latin-1 style patterns
            WHEN regexp_matches(lower(cds_tracklist),
                '(ã[a-z0-9]|â[a-z0-9]|ð[a-z0-9]|[a-z]ã[a-z]|[a-z]â[a-z])'
            ) THEN NULL

            -- suspicious density of non-alnum junk characters
            WHEN length(regexp_replace(coalesce(cds_tracklist, ''), '[A-Za-z0-9 ]', '', 'g')) > 0.20 * length(coalesce(cds_tracklist, ''))
                THEN NULL

            ELSE cds_tracklist
        END AS cds_tracklist

    FROM cds_to_mb_final_mapping_v9_enriched
""")

print("✅ Table created: cds_to_mb_final_mapping_v9_enriched_cleantrack_v1")

print("\n--- Count of rows where cds_tracklist was blanked ---")
print(con.execute(r"""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_final_mapping_v9_enriched_cleantrack_v1 t
    JOIN cds_to_mb_final_mapping_v9_enriched s
      ON t.cds_id = s.cds_id
    WHERE s.cds_tracklist IS NOT NULL
      AND trim(s.cds_tracklist) <> ''
      AND t.cds_tracklist IS NULL
""").fetchdf())

print("\n--- Preview: rows where cds_tracklist was blanked ---")
print(con.execute(r"""
    SELECT
        t.cds_id,
        t.cds_artist,
        t.cds_title,
        s.cds_tracklist AS original_cds_tracklist,
        t.cds_tracklist AS cleaned_cds_tracklist
    FROM cds_to_mb_final_mapping_v9_enriched_cleantrack_v1 t
    JOIN cds_to_mb_final_mapping_v9_enriched s
      ON t.cds_id = s.cds_id
    WHERE s.cds_tracklist IS NOT NULL
      AND trim(s.cds_tracklist) <> ''
      AND t.cds_tracklist IS NULL
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_to_mb_final_mapping_v9_enriched_cleantrack_v1...
✅ Table created: cds_to_mb_final_mapping_v9_enriched_cleantrack_v1

--- Count of rows where cds_tracklist was blanked ---
     n
0  671

--- Preview: rows where cds_tracklist was blanked ---
    cds_id             cds_artist  \
0   100001                various   
1   100024                   175r   
2   100026     day after tomorrow   
3   100031                spartak   
4   100037   sportfreunde stiller   
5   100087          tschou zã me    
6   100089        various artists   
7   100099     fabienne thibeault   
8   100101                various   
9   100114          gerardo peã a   
10  100116      stefan sundstrã m   
11  100141              mã ra per   
12  100143                clannad   
13  100180           albin berger   
14  100191                gem boy   
15  100207        poems for laila   
16  100225  paquita la del barrio   
17  100236          terje tysland   
18  100244       jari sillanpã ã

In [74]:
import os

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_to_mb_final_mapping_v9_enriched_cleantrack_v1.csv"

con.execute(f"""
    COPY cds_to_mb_final_mapping_v9_enriched_cleantrack_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/cds_to_mb_final_mapping_v9_enriched_cleantrack_v1.csv


In [75]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build thesis-friendly enriched review-candidate table
# ------------------------------------------------------------
# Comments:
# - This keeps the MB review candidates
# - and adds CDS-side metadata from the original cds table:
#     category, genre, tracks
# - It should be interpreted as:
#     plausible MB enrichment candidates requiring review
print("Creating cds_to_mb_review_candidates_v6_round2_enriched...")

con.execute("""
    CREATE OR REPLACE TABLE cds_to_mb_review_candidates_v6_round2_enriched AS
    SELECT
        -- ----------------------------------------------------
        -- CDS-side metadata
        -- ----------------------------------------------------
        r.cds_id,
        c.artist AS cds_artist,
        c.title AS cds_title,
        c.cds_year,
        c.category,
        c.genre,
        c.tracks AS cds_tracklist,

        -- ----------------------------------------------------
        -- Review-candidate linkage fields
        -- ----------------------------------------------------
        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.total_tracks,
        r.barcode,
        r.mb_year,

        -- ----------------------------------------------------
        -- Review / scoring metadata
        -- ----------------------------------------------------
        r.artist_sim,
        r.title_sim,
        r.score_gap,
        r.best_score,
        r.second_best_score,
        'matched_review_candidate' AS match_status

    FROM cds_to_mb_review_candidates_v6_round2 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
""")

print("✅ Table created: cds_to_mb_review_candidates_v6_round2_enriched")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_to_mb_review_candidates_v6_round2_enriched
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM cds_to_mb_review_candidates_v6_round2_enriched
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_to_mb_review_candidates_v6_round2_enriched...
✅ Table created: cds_to_mb_review_candidates_v6_round2_enriched

--- Row count ---
    n
0  64

--- Preview ---
    cds_id           cds_artist                               cds_title  \
0   100415       vã ctor herdia                          vã ctor herdia   
1   100504          mary bergin                           feadoga stain   
2   100837       billie holiday             1949 52 radio tv broadcasts   
3   100881                scarp                                   scarp   
4   100911       nellie lutcher       nellie lutcher her rhythm 2 of 4    
5   101041          lada kerndl                            evergreens 2   
6   101111        george benson                         jazz masters 21   
7   101143   samla mammas manna                                  maltid   
8   101220      louis armstrong  the great chicago concert 1956 disc 2    
9   101632          bill bryson     in a sunburned country disc 7 o

In [78]:
import os

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_to_mb_review_candidates_v6_round2_enriched.csv"

con.execute(f"""
    COPY cds_to_mb_review_candidates_v6_round2_enriched
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/cds_to_mb_review_candidates_v6_round2_enriched.csv


In [79]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build one combined MB enrichment table:
# automatic matches + review matches
# ------------------------------------------------------------
# Comments:
# - We use the schema of the automatic enriched table as the reference schema.
# - Then we map review-candidate columns into that schema.
# - Any missing columns on the review side are filled with typed NULLs.
# - We also prevent duplicate cds_id rows:
#   if a CDS row is already present in the automatic table,
#   we do not add it again from the review table.
print("Reading schema of cds_to_mb_final_mapping_v9_enriched_cleantrack_v1...")

auto_schema = con.execute(
    "PRAGMA table_info('cds_to_mb_final_mapping_v9_enriched_cleantrack_v1')"
).fetchdf()

auto_cols = auto_schema["name"].tolist()
auto_types = dict(zip(auto_schema["name"], auto_schema["type"]))

# ------------------------------------------------------------
# Map review columns into the automatic schema
# ------------------------------------------------------------
review_mapping = {
    "cds_id": "r.cds_id",
    "cds_artist": "r.cds_artist",
    "cds_title": "r.cds_title",
    "cds_year": "r.cds_year",
    "category": "r.category",
    "genre": "r.genre",
    "cds_tracklist": "r.cds_tracklist",
    "release_id": "r.release_id",
    "release_title": "r.release_title",
    "main_artist_name": "r.main_artist_name",
    "artist_credit_full": "r.artist_credit_full",
    "total_tracks": "r.total_tracks",
    "barcode": "r.barcode",
    "mb_year": "r.mb_year",
    "match_type": "'matched_review_candidate'",
    "score": "r.best_score",
}

review_select_parts = []
for col in auto_cols:
    if col in review_mapping:
        review_select_parts.append(f"{review_mapping[col]} AS {col}")
    else:
        review_select_parts.append(f"NULL::{auto_types[col]} AS {col}")

review_select_sql = ",\n        ".join(review_select_parts)

print("Creating cds_mb_enrichment_combined_v1...")

con.execute(f"""
    CREATE OR REPLACE TABLE cds_mb_enrichment_combined_v1 AS

    -- --------------------------------------------------------
    -- A. Automatic enriched rows
    -- --------------------------------------------------------
    SELECT
        {", ".join(auto_cols)}
    FROM cds_to_mb_final_mapping_v9_enriched_cleantrack_v1

    UNION ALL

    -- --------------------------------------------------------
    -- B. Review enriched rows not already present automatically
    -- --------------------------------------------------------
    SELECT
        {review_select_sql}
    FROM cds_to_mb_review_candidates_v6_round2_enriched r
    WHERE NOT EXISTS (
        SELECT 1
        FROM cds_to_mb_final_mapping_v9_enriched_cleantrack_v1 a
        WHERE a.cds_id = r.cds_id
    )
""")

print("✅ Table created: cds_mb_enrichment_combined_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_enrichment_combined_v1
""").fetchdf())

print("\n--- Match type breakdown ---")
print(con.execute("""
    SELECT match_type, COUNT(*) AS n
    FROM cds_mb_enrichment_combined_v1
    GROUP BY match_type
    ORDER BY n DESC
""").fetchdf())

print("\n--- Uniqueness check by cds_id ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT cds_id) AS distinct_cds_id,
        COUNT(*) - COUNT(DISTINCT cds_id) AS duplicate_cds_id_rows
    FROM cds_mb_enrichment_combined_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM cds_mb_enrichment_combined_v1
    LIMIT 20
""").fetchdf())

Connected ✅
Reading schema of cds_to_mb_final_mapping_v9_enriched_cleantrack_v1...
Creating cds_mb_enrichment_combined_v1...
✅ Table created: cds_mb_enrichment_combined_v1

--- Row count ---
      n
0  4257

--- Match type breakdown ---
                 match_type     n
0                     exact  3482
1         fuzzy_second_pass   543
2        various_title_only   183
3  matched_review_candidate    43
4     alias_enrichment_safe     3
5    enrichment_round2_auto     3

--- Uniqueness check by cds_id ---
   total_rows  distinct_cds_id  duplicate_cds_id_rows
0        4257             4257                      0

--- Preview ---
    cds_id            cds_artist                      cds_title  cds_year  \
0    10000       backstreet boys                     millennium      <NA>   
1   100001               various    frankfurt trance vol 04 cd1      <NA>   
2   100002             no return                self mutilation      <NA>   
3   100006      luv lite massive                       o

In [80]:
import os

os.makedirs("../data_processed/exports", exist_ok=True)

out_path = "../data_processed/exports/cds_mb_enrichment_combined_v1.csv"

con.execute(f"""
    COPY cds_mb_enrichment_combined_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/cds_mb_enrichment_combined_v1.csv


In [82]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Post-hoc recovery:
# promote CDS-only safe rows that actually exist in MusicBrainz
# ------------------------------------------------------------
# Comments:
# - This checks whether rows in cds_only_enrichment_safe_v1 are actually
#   exact normalized artist/title matches to MB release data.
# - If yes, they should be moved from CDS-only enrichment to
#   automatic MB-linked enrichment.
print("Creating cds_only_safe_recovered_auto_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_safe_recovered_auto_v1 AS
    WITH cds_safe AS (
        SELECT
            s.cds_id,
            s.artist,
            s.title,
            s.cds_year,
            s.tracks,
            s.artist_key_clean,
            s.title_key_clean
        FROM cds_only_enrichment_safe_v1 s
    ),
    mb_norm AS (
        SELECT
            r.release_id,
            r.release_title,
            r.main_artist_name,
            r.artist_credit_full,
            r.total_tracks,
            r.barcode,
            r.mb_year,
            lower(regexp_replace(coalesce(r.release_title, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_title_key,
            lower(regexp_replace(coalesce(r.main_artist_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_artist_key
        FROM mb_release_features r
    )
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.tracks,
        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year,
        'recovered_exact_safe' AS match_type,
        100.0 AS score
    FROM cds_safe c
    JOIN mb_norm m
      ON c.title_key_clean = m.mb_title_key
     AND c.artist_key_clean = m.mb_artist_key
""")

print("✅ Table created: cds_only_safe_recovered_auto_v1")

print("\n--- Recovered rows count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_safe_recovered_auto_v1
""").fetchdf())

print("\n--- Distinct CDS rows recovered ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS n
    FROM cds_only_safe_recovered_auto_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM cds_only_safe_recovered_auto_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_only_safe_recovered_auto_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_only_safe_recovered_auto_v1

--- Recovered rows count ---
    n
0  46

--- Distinct CDS rows recovered ---
   n
0  6

--- Preview ---
    cds_id  artist                         title  cds_year  \
0    10119  r e m   fables of the reconstruction      1986   
1    10119  r e m   fables of the reconstruction      1986   
2    10119  r e m   fables of the reconstruction      1986   
3    10119  r e m   fables of the reconstruction      1986   
4    10119  r e m   fables of the reconstruction      1986   
5   101964    a ha         minor earth major sky      2000   
6    10119  r e m   fables of the reconstruction      1986   
7    10119  r e m   fables of the reconstruction      1986   
8    10119  r e m   fables of the reconstruction      1986   
9    10119  r e m   fables of the reconstruction      1986   
10   10119  r e m   fables of the reconstruction      1986   
11   10119  r e m   fables of the reconstruction      1986   
12   10119  r e m   fables of the recon

In [83]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("Reclassifying recovered rows with stricter auto rules...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_safe_recovered_auto_unique_v1 AS
    SELECT *
    FROM cds_only_safe_recovered_ranked_v1
    WHERE rn = 1
      AND (
            (
                candidate_count = 1
                AND year_gap IS NOT NULL
                AND year_gap <= 5
            )
            OR (
                recovery_rank_score >= 90
                AND recovery_score_gap >= 10
                AND (year_gap IS NULL OR year_gap <= 5)
            )
          )
""")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_safe_recovered_review_v1 AS
    SELECT *
    FROM cds_only_safe_recovered_ranked_v1
    WHERE rn = 1
      AND cds_id NOT IN (
          SELECT cds_id
          FROM cds_only_safe_recovered_auto_unique_v1
      )
""")

print("✅ Tables created:")
print("   - cds_only_safe_recovered_auto_unique_v1")
print("   - cds_only_safe_recovered_review_v1")

print("\n--- Recovery split summary (strict version) ---")
print(con.execute("""
    SELECT 'auto_unique' AS bucket, COUNT(*) AS n
    FROM cds_only_safe_recovered_auto_unique_v1
    UNION ALL
    SELECT 'review' AS bucket, COUNT(*) AS n
    FROM cds_only_safe_recovered_review_v1
""").fetchdf())

print("\n--- Preview: strict auto rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        release_title,
        main_artist_name,
        cds_year,
        mb_year,
        year_gap,
        candidate_count
    FROM cds_only_safe_recovered_auto_unique_v1
""").fetchdf())

Connected ✅
Reclassifying recovered rows with stricter auto rules...
✅ Tables created:
   - cds_only_safe_recovered_auto_unique_v1
   - cds_only_safe_recovered_review_v1

--- Recovery split summary (strict version) ---
        bucket  n
0  auto_unique  0
1       review  6

--- Preview: strict auto rows ---
Empty DataFrame
Columns: [cds_id, artist, title, release_id, release_title, main_artist_name, cds_year, mb_year, year_gap, candidate_count]
Index: []


In [84]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: keep only the truly CDS-only safe rows
# ------------------------------------------------------------
# Comments:
# - cds_only_enrichment_safe_v1 currently contains some rows that were
#   later recovered as already present in MusicBrainz.
# - Those recovered rows should no longer remain in the "true CDS-only"
#   safe enrichment bucket.
print("Creating cds_only_enrichment_safe_true_unmatched_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_safe_true_unmatched_v1 AS
    SELECT *
    FROM cds_only_enrichment_safe_v1 s
    WHERE s.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_only_safe_recovered_review_v1
    )
""")

print("✅ Table created: cds_only_enrichment_safe_true_unmatched_v1")

print("\n--- Count of true CDS-only safe rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_safe_true_unmatched_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 2: build master thesis enrichment table
# ------------------------------------------------------------
# Comments:
# - This is not only a mapping table.
# - It is a thesis-facing enrichment table with three outcomes:
#     1) confirmed_mb_enrichment
#     2) review_mb_enrichment
#     3) new_cds_enrichment
# - We standardize the schema so all rows can be stacked with UNION ALL.
print("\nCreating thesis_musicbrainz_enrichment_master_v1...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_musicbrainz_enrichment_master_v1 AS

    -- --------------------------------------------------------
    -- A. Confirmed or review-linked MB enrichment already in
    --    cds_mb_enrichment_combined_v1
    -- --------------------------------------------------------
    SELECT
        cds_id,
        cds_artist AS artist,
        cds_title AS title,
        cds_year,
        category,
        genre,
        cds_tracklist AS tracks,

        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,

        match_type AS enrichment_route,

        CASE
            WHEN match_type = 'matched_review_candidate' THEN 'review_mb_enrichment'
            ELSE 'confirmed_mb_enrichment'
        END AS enrichment_status

    FROM cds_mb_enrichment_combined_v1

    UNION ALL

    -- --------------------------------------------------------
    -- B. Recovered MB-present rows that remain review-only
    -- --------------------------------------------------------
    SELECT
        r.cds_id,
        c.artist AS artist,
        c.title AS title,
        c.cds_year,
        c.category,
        c.genre,
        c.tracks AS tracks,

        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.total_tracks,
        r.barcode,
        r.mb_year,

        'recovered_mb_presence_review' AS enrichment_route,
        'review_mb_enrichment' AS enrichment_status

    FROM cds_only_safe_recovered_review_v1 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id

    UNION ALL

    -- --------------------------------------------------------
    -- C. True CDS-only enrichment rows not found in MB
    -- --------------------------------------------------------
    SELECT
        s.cds_id,
        s.artist,
        s.title,
        s.cds_year,
        c.category,
        c.genre,
        c.tracks,

        NULL AS release_id,
        NULL AS release_title,
        NULL AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS total_tracks,
        NULL AS barcode,
        NULL AS mb_year,

        'cds_only_safe' AS enrichment_route,
        'new_cds_enrichment' AS enrichment_status

    FROM cds_only_enrichment_safe_true_unmatched_v1 s
    LEFT JOIN cds c
      ON s.cds_id = c.cds_id
""")

print("✅ Table created: thesis_musicbrainz_enrichment_master_v1")

print("\n--- Master enrichment summary ---")
print(con.execute("""
    SELECT
        enrichment_status,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM thesis_musicbrainz_enrichment_master_v1), 2) AS pct
    FROM thesis_musicbrainz_enrichment_master_v1
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by route ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v1
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM thesis_musicbrainz_enrichment_master_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_only_enrichment_safe_true_unmatched_v1...
✅ Table created: cds_only_enrichment_safe_true_unmatched_v1

--- Count of true CDS-only safe rows ---
      n
0  1646

Creating thesis_musicbrainz_enrichment_master_v1...
✅ Table created: thesis_musicbrainz_enrichment_master_v1

--- Master enrichment summary ---
         enrichment_status     n    pct
0  confirmed_mb_enrichment  4214  71.31
1       new_cds_enrichment  1646  27.86
2     review_mb_enrichment    49   0.83

--- Breakdown by route ---
               enrichment_route     n
0                         exact  3482
1                 cds_only_safe  1646
2             fuzzy_second_pass   543
3            various_title_only   183
4      matched_review_candidate    43
5  recovered_mb_presence_review     6
6         alias_enrichment_safe     3
7        enrichment_round2_auto     3

--- Preview ---
    cds_id                artist                            title  cds_year  \
0    10000       backstreet boys            

In [85]:
import os

os.makedirs("../data_processed/exports/final_thesis_outputs", exist_ok=True)

out_path = "../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v1.csv"

con.execute(f"""
    COPY thesis_musicbrainz_enrichment_master_v1
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v1.csv


In [86]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Recover MB-present rows from cds_only_enrichment_review_v1
# ------------------------------------------------------------
# Comments:
# - We check whether rows currently in the CDS-only review table
#   are actually present in MusicBrainz by exact normalized
#   artist-title agreement.
# - This is the same recovery idea we used earlier on the safe table.
print("Creating cds_only_review_recovered_mb_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_review_recovered_mb_v1 AS
    WITH cds_review AS (
        SELECT
            r.cds_id,
            r.artist,
            r.title,
            r.cds_year,
            r.tracks,
            r.artist_key_clean,
            r.title_key_clean
        FROM cds_only_enrichment_review_v1 r
    ),
    mb_norm AS (
        SELECT
            m.release_id,
            m.release_title,
            m.main_artist_name,
            m.artist_credit_full,
            m.total_tracks,
            m.barcode,
            m.mb_year,
            lower(regexp_replace(coalesce(m.main_artist_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_artist_key,
            lower(regexp_replace(coalesce(m.release_title, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_title_key
        FROM mb_release_features m
    )
    SELECT
        c.cds_id,
        c.artist,
        c.title,
        c.cds_year,
        c.tracks,
        c.artist_key_clean,
        c.title_key_clean,
        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year
    FROM cds_review c
    JOIN mb_norm m
      ON c.artist_key_clean = m.mb_artist_key
     AND c.title_key_clean = m.mb_title_key
""")

print("✅ Table created: cds_only_review_recovered_mb_v1")

print("\n--- Recovered candidate rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_review_recovered_mb_v1
""").fetchdf())

print("\n--- Distinct CDS rows recovered ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS n
    FROM cds_only_review_recovered_mb_v1
""").fetchdf())

print("\n--- Preview recovered rows ---")
print(con.execute("""
    SELECT *
    FROM cds_only_review_recovered_mb_v1
    LIMIT 20
""").fetchdf())

# ------------------------------------------------------------
# Step 2: Rank recovered rows using year evidence only
# ------------------------------------------------------------
# Comments:
# - We do NOT use track-gap because CDS tracks are text, not numeric count.
# - We rank by:
#     * year agreement
#     * barcode presence
#     * stable tie-break by release_id
print("\nCreating cds_only_review_recovered_ranked_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_review_recovered_ranked_v1 AS
    WITH scored AS (
        SELECT
            r.*,

            CASE
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                    THEN abs(r.cds_year - r.mb_year)
                ELSE NULL
            END AS year_gap,

            CASE
                WHEN r.barcode IS NOT NULL AND trim(r.barcode) <> '' THEN 1
                ELSE 0
            END AS has_barcode,

            CASE
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) = 0 THEN 100
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) = 1 THEN 90
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) <= 3 THEN 75
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) <= 5 THEN 60
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NULL THEN 40
                WHEN r.cds_year IS NULL AND r.mb_year IS NOT NULL THEN 30
                ELSE 20
            END
            +
            CASE
                WHEN r.barcode IS NOT NULL AND trim(r.barcode) <> '' THEN 5
                ELSE 0
            END AS recovery_rank_score
        FROM cds_only_review_recovered_mb_v1 r
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cds_id
                ORDER BY
                    recovery_rank_score DESC,
                    year_gap ASC NULLS LAST,
                    has_barcode DESC,
                    mb_year ASC NULLS LAST,
                    release_id
            ) AS rn,

            COUNT(*) OVER (
                PARTITION BY cds_id
            ) AS candidate_count,

            LEAD(recovery_rank_score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    recovery_rank_score DESC,
                    year_gap ASC NULLS LAST,
                    has_barcode DESC,
                    mb_year ASC NULLS LAST,
                    release_id
            ) AS second_score
        FROM scored
    )
    SELECT
        *,
        recovery_rank_score - coalesce(second_score, 0) AS recovery_score_gap
    FROM ranked
""")

print("✅ Table created: cds_only_review_recovered_ranked_v1")

print("\n--- Candidate count by CDS row ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        candidate_count
    FROM cds_only_review_recovered_ranked_v1
    WHERE rn = 1
    ORDER BY candidate_count DESC, cds_id
""").fetchdf())

# ------------------------------------------------------------
# Step 3: classify recovered review rows into auto vs review
# ------------------------------------------------------------
# Comments:
# - We are strict:
#   auto only if unique and year-plausible, or clearly dominant
print("\nCreating cds_only_review_recovered_auto_v1 and cds_only_review_recovered_review_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_review_recovered_auto_v1 AS
    SELECT *
    FROM cds_only_review_recovered_ranked_v1
    WHERE rn = 1
      AND (
            (
                candidate_count = 1
                AND year_gap IS NOT NULL
                AND year_gap <= 5
            )
            OR (
                recovery_rank_score >= 90
                AND recovery_score_gap >= 10
                AND (year_gap IS NULL OR year_gap <= 5)
            )
          )
""")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_review_recovered_review_v1 AS
    SELECT *
    FROM cds_only_review_recovered_ranked_v1
    WHERE rn = 1
      AND cds_id NOT IN (
          SELECT cds_id
          FROM cds_only_review_recovered_auto_v1
      )
""")

print("✅ Tables created:")
print("   - cds_only_review_recovered_auto_v1")
print("   - cds_only_review_recovered_review_v1")

print("\n--- Recovery split summary ---")
print(con.execute("""
    SELECT 'auto' AS bucket, COUNT(*) AS n
    FROM cds_only_review_recovered_auto_v1
    UNION ALL
    SELECT 'review' AS bucket, COUNT(*) AS n
    FROM cds_only_review_recovered_review_v1
""").fetchdf())

print("\n--- Preview: recovered auto rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        release_title,
        main_artist_name,
        cds_year,
        mb_year,
        year_gap,
        recovery_rank_score,
        recovery_score_gap,
        candidate_count
    FROM cds_only_review_recovered_auto_v1
    ORDER BY cds_id
    LIMIT 20
""").fetchdf())

print("\n--- Preview: recovered review rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        release_id,
        release_title,
        main_artist_name,
        cds_year,
        mb_year,
        year_gap,
        recovery_rank_score,
        recovery_score_gap,
        candidate_count
    FROM cds_only_review_recovered_review_v1
    ORDER BY cds_id
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_only_review_recovered_mb_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_only_review_recovered_mb_v1

--- Recovered candidate rows ---
     n
0  344

--- Distinct CDS rows recovered ---
    n
0  88

--- Preview recovered rows ---
    cds_id             artist                               title  cds_year  \
0   106128               2pac   r u still down remember me disc 2      1997   
1   101723               2pac  r u still down remember me disc 1       1997   
2   104191                afi                     sing the sorrow      2003   
3   107319                dig           defenders of the universe      <NA>   
4   109049  leonard bernstein    bernstein conducts bernstein cd5      <NA>   
5   106128               2pac   r u still down remember me disc 2      1997   
6   101723               2pac  r u still down remember me disc 1       1997   
7   108252    various artists      winter solstice on ice disc 1       1999   
8     7898                ash                     free all angels      2001   
9   107469    various artists   

In [87]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 1: Reclassify recovered CDS-only review rows
# ------------------------------------------------------------
# Comments:
# - cds_only_review_recovered_review_v1 contains CDS rows that were
#   originally in the CDS-only review bucket, but were later found
#   to already exist in MusicBrainz.
# - These rows should therefore move to MB review enrichment.
# - The remaining rows in cds_only_enrichment_review_v1 stay as
#   true CDS-only review enrichment candidates.
print("Creating cds_mb_enrichment_review_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_mb_enrichment_review_v1 AS
    SELECT
        r.cds_id,
        c.artist AS cds_artist,
        c.title AS cds_title,
        c.cds_year,
        c.category,
        c.genre,
        c.tracks AS cds_tracklist,

        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.total_tracks,
        r.barcode,
        r.mb_year,

        r.year_gap,
        r.recovery_rank_score,
        r.recovery_score_gap,
        r.candidate_count,

        'recovered_mb_presence_review' AS match_type,
        'review_mb_enrichment' AS enrichment_status
    FROM cds_only_review_recovered_review_v1 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
""")

print("✅ Table created: cds_mb_enrichment_review_v1")

print("\nCreating cds_only_enrichment_review_true_unmatched_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_review_true_unmatched_v1 AS
    SELECT *
    FROM cds_only_enrichment_review_v1 r
    WHERE r.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_mb_enrichment_review_v1
    )
""")

print("✅ Table created: cds_only_enrichment_review_true_unmatched_v1")

print("\n--- Reclassification summary ---")
print(con.execute("""
    SELECT 'mb_review_enrichment' AS bucket, COUNT(*) AS n
    FROM cds_mb_enrichment_review_v1
    UNION ALL
    SELECT 'true_cds_only_review' AS bucket, COUNT(*) AS n
    FROM cds_only_enrichment_review_true_unmatched_v1
""").fetchdf())

print("\n--- Preview: MB review enrichment rows ---")
print(con.execute("""
    SELECT *
    FROM cds_mb_enrichment_review_v1
    LIMIT 20
""").fetchdf())

print("\n--- Preview: true CDS-only review rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        tracks,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_review_true_unmatched_v1
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_mb_enrichment_review_v1...
✅ Table created: cds_mb_enrichment_review_v1

Creating cds_only_enrichment_review_true_unmatched_v1...
✅ Table created: cds_only_enrichment_review_true_unmatched_v1

--- Reclassification summary ---
                 bucket     n
0  mb_review_enrichment    88
1  true_cds_only_review  2980

--- Preview: MB review enrichment rows ---
    cds_id       cds_artist                                 cds_title  \
0   100196              eko                                   celtica   
1   100293              bzn                             crystal gazer   
2   100314         trovante                     uma noite sã³ disc 2    
3   100810              him                        razorblade romance   
4   100850  various artists               ska the third wave volume 5   
5   100863  various artists               hed kandi back to love cd2    
6   101229  various artists                           jazz moods cd 2   
7   101470  various artists    

In [88]:
#delete this cell

In [89]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Step 2: Build final thesis enrichment master table
# ------------------------------------------------------------
# Comments:
# - This table now integrates all usable enrichment outputs:
#   1) confirmed_mb_enrichment
#   2) review_mb_enrichment
#   3) new_cds_enrichment_safe
#   4) new_cds_enrichment_review
# - Recovered MB-present rows from the CDS-only review branch are now
#   included as review-level MB enrichment.
print("Creating thesis_musicbrainz_enrichment_master_v3...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_musicbrainz_enrichment_master_v3 AS

    -- --------------------------------------------------------
    -- A. Confirmed MB enrichment + already included MB review
    -- --------------------------------------------------------
    SELECT
        cds_id,
        cds_artist AS artist,
        cds_title AS title,
        cds_year,
        category,
        genre,
        cds_tracklist AS tracks,

        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,

        match_type AS enrichment_route,
        CASE
            WHEN match_type = 'matched_review_candidate' THEN 'review_mb_enrichment'
            ELSE 'confirmed_mb_enrichment'
        END AS enrichment_status
    FROM cds_mb_enrichment_combined_v1

    UNION ALL

    -- --------------------------------------------------------
    -- B. Additional MB review enrichment recovered from
    --    CDS-only review rows
    -- --------------------------------------------------------
    SELECT
        cds_id,
        cds_artist AS artist,
        cds_title AS title,
        cds_year,
        category,
        genre,
        cds_tracklist AS tracks,

        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,

        match_type AS enrichment_route,
        enrichment_status
    FROM cds_mb_enrichment_review_v1
    WHERE cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_mb_enrichment_combined_v1
    )

    UNION ALL

    -- --------------------------------------------------------
    -- C. True CDS-only safe enrichment
    -- --------------------------------------------------------
    SELECT
        s.cds_id,
        s.artist,
        s.title,
        s.cds_year,
        c.category,
        c.genre,
        c.tracks,

        NULL AS release_id,
        NULL AS release_title,
        NULL AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS total_tracks,
        NULL AS barcode,
        NULL AS mb_year,

        'cds_only_safe' AS enrichment_route,
        'new_cds_enrichment_safe' AS enrichment_status
    FROM cds_only_enrichment_safe_true_unmatched_v1 s
    LEFT JOIN cds c
      ON s.cds_id = c.cds_id

    UNION ALL

    -- --------------------------------------------------------
    -- D. True CDS-only review enrichment
    -- --------------------------------------------------------
    SELECT
        r.cds_id,
        r.artist,
        r.title,
        r.cds_year,
        c.category,
        c.genre,
        c.tracks,

        NULL AS release_id,
        NULL AS release_title,
        NULL AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS total_tracks,
        NULL AS barcode,
        NULL AS mb_year,

        'cds_only_review' AS enrichment_route,
        'new_cds_enrichment_review' AS enrichment_status
    FROM cds_only_enrichment_review_true_unmatched_v1 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
""")

print("✅ Table created: thesis_musicbrainz_enrichment_master_v3")

print("\n--- Final enrichment summary ---")
print(con.execute("""
    SELECT
        enrichment_status,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM thesis_musicbrainz_enrichment_master_v3), 2) AS pct
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by route ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM thesis_musicbrainz_enrichment_master_v3
    LIMIT 30
""").fetchdf())

Connected ✅
Creating thesis_musicbrainz_enrichment_master_v3...
✅ Table created: thesis_musicbrainz_enrichment_master_v3

--- Final enrichment summary ---
           enrichment_status     n    pct
0    confirmed_mb_enrichment  4214  46.97
1  new_cds_enrichment_review  2980  33.22
2    new_cds_enrichment_safe  1646  18.35
3       review_mb_enrichment   131   1.46

--- Breakdown by route ---
               enrichment_route     n
0                         exact  3482
1               cds_only_review  2980
2                 cds_only_safe  1646
3             fuzzy_second_pass   543
4            various_title_only   183
5  recovered_mb_presence_review    88
6      matched_review_candidate    43
7         alias_enrichment_safe     3
8        enrichment_round2_auto     3

--- Preview ---
    cds_id                artist                            title  cds_year  \
0    10000       backstreet boys                       millennium      <NA>   
1   100001               various      frankfurt tran

In [90]:
#delete this cell

In [91]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Rename the recovered review route label inside the final thesis table
# ------------------------------------------------------------
# Comments:
# - We keep the table structure the same.
# - We only replace the route name with a cleaner thesis-facing label.
print("Updating enrichment_route label in thesis_musicbrainz_enrichment_master_v3...")

con.execute("""
    UPDATE thesis_musicbrainz_enrichment_master_v3
    SET enrichment_route = 'review_mb_enrichment_recovered'
    WHERE enrichment_route = 'recovered_mb_presence_review'
""")

print("✅ Route label updated")

print("\n--- Updated route breakdown ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

Connected ✅
Updating enrichment_route label in thesis_musicbrainz_enrichment_master_v3...
✅ Route label updated

--- Updated route breakdown ---
                 enrichment_route     n
0                           exact  3482
1                 cds_only_review  2980
2                   cds_only_safe  1646
3               fuzzy_second_pass   543
4              various_title_only   183
5  review_mb_enrichment_recovered    88
6        matched_review_candidate    43
7           alias_enrichment_safe     3
8          enrichment_round2_auto     3


In [92]:
import os

os.makedirs("../data_processed/exports/final_thesis_outputs", exist_ok=True)

out_path = "../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v3.csv"

con.execute(f"""
    COPY thesis_musicbrainz_enrichment_master_v3
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v3.csv


In [93]:
import duckdb

# Connect to your database
con = duckdb.connect("../data_processed/thesis.duckdb")

out_path = "../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v3.csv"
con.execute(f"""
    COPY thesis_musicbrainz_enrichment_master_v3
    TO '{out_path}'
    WITH (
        HEADER,
        DELIMITER ',',
        QUOTE '"',
        ESCAPE '"'
    )
""")

print(f"✅ Exported clean CSV: {out_path}")

✅ Exported clean CSV: ../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v3.csv


In [94]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build corrected thesis_musicbrainz_enrichment_master_v3
# ------------------------------------------------------------
# Comments:
# - Start from thesis_musicbrainz_enrichment_master_v1, because it already
#   contains the original recovered_mb_presence_review = 5 rows.
# - Rename that old route label to the cleaner thesis-facing name:
#     review_mb_enrichment_recovered
# - Then append only NEW recovered MB-review rows from
#   cds_mb_enrichment_review_v1 that are not already in v1.
# - Then append only the remaining true CDS-only review rows that are not
#   already present in v1.
print("Creating corrected thesis_musicbrainz_enrichment_master_v3...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_musicbrainz_enrichment_master_v3 AS

    -- --------------------------------------------------------
    -- A. Start from v1 and rename the old recovered route label
    -- --------------------------------------------------------
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        category,
        genre,
        tracks,
        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        total_tracks,
        barcode,
        mb_year,

        CASE
            WHEN enrichment_route = 'recovered_mb_presence_review'
                THEN 'review_mb_enrichment_recovered'
            ELSE enrichment_route
        END AS enrichment_route,

        enrichment_status
    FROM thesis_musicbrainz_enrichment_master_v1

    UNION ALL

    -- --------------------------------------------------------
    -- B. Add NEW recovered MB review rows not already in v1
    -- --------------------------------------------------------
    SELECT
        r.cds_id,
        r.cds_artist AS artist,
        r.cds_title AS title,
        r.cds_year,
        r.category,
        r.genre,
        r.cds_tracklist AS tracks,
        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.total_tracks,
        r.barcode,
        r.mb_year,
        'review_mb_enrichment_recovered' AS enrichment_route,
        'review_mb_enrichment' AS enrichment_status
    FROM cds_mb_enrichment_review_v1 r
    WHERE r.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM thesis_musicbrainz_enrichment_master_v1
    )

    UNION ALL

    -- --------------------------------------------------------
    -- C. Add remaining true CDS-only review rows not already in v1
    -- --------------------------------------------------------
    SELECT
        r.cds_id,
        r.artist,
        r.title,
        r.cds_year,
        c.category,
        c.genre,
        c.tracks,
        NULL AS release_id,
        NULL AS release_title,
        NULL AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS total_tracks,
        NULL AS barcode,
        NULL AS mb_year,
        'cds_only_review' AS enrichment_route,
        'new_cds_enrichment_review' AS enrichment_status
    FROM cds_only_enrichment_review_true_unmatched_v1 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
    WHERE r.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM thesis_musicbrainz_enrichment_master_v1
    )
""")

print("✅ Table created: thesis_musicbrainz_enrichment_master_v3")

print("\n--- Final enrichment summary ---")
print(con.execute("""
    SELECT
        enrichment_status,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM thesis_musicbrainz_enrichment_master_v3), 2) AS pct
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by route ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Check recovered-review rows specifically ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    WHERE enrichment_route = 'review_mb_enrichment_recovered'
    GROUP BY enrichment_route
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM thesis_musicbrainz_enrichment_master_v3
    LIMIT 30
""").fetchdf())

Connected ✅
Creating corrected thesis_musicbrainz_enrichment_master_v3...
✅ Table created: thesis_musicbrainz_enrichment_master_v3

--- Final enrichment summary ---
           enrichment_status     n    pct
0    confirmed_mb_enrichment  4214  46.96
1  new_cds_enrichment_review  2977  33.17
2         new_cds_enrichment  1646  18.34
3       review_mb_enrichment   137   1.53

--- Breakdown by route ---
                 enrichment_route     n
0                           exact  3482
1                 cds_only_review  2977
2                   cds_only_safe  1646
3               fuzzy_second_pass   543
4              various_title_only   183
5  review_mb_enrichment_recovered    94
6        matched_review_candidate    43
7           alias_enrichment_safe     3
8          enrichment_round2_auto     3

--- Check recovered-review rows specifically ---
                 enrichment_route   n
0  review_mb_enrichment_recovered  94

--- Preview ---
    cds_id                artist                      

In [95]:
print(con.execute("""
    SELECT enrichment_status, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print(con.execute("""
    SELECT enrichment_route, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v3
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

           enrichment_status     n
0    confirmed_mb_enrichment  4214
1  new_cds_enrichment_review  2977
2         new_cds_enrichment  1646
3       review_mb_enrichment   137
                 enrichment_route     n
0                           exact  3482
1                 cds_only_review  2977
2                   cds_only_safe  1646
3               fuzzy_second_pass   543
4              various_title_only   183
5  review_mb_enrichment_recovered    94
6        matched_review_candidate    43
7          enrichment_round2_auto     3
8           alias_enrichment_safe     3


In [96]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Verify exactly which cds_id values were excluded
# ------------------------------------------------------------
# Logic:
# - Naive expectation for v3 was:
#     rows in v1
#   + rows in cds_mb_enrichment_review_v1
#   + rows in cds_only_enrichment_review_true_unmatched_v1
# - But v3 excludes cds_id values already present in v1.
# - This cell shows exactly which overlapping cds_id values caused
#   the final total to be 4 rows lower than the naive sum.
print("--- Overlapping cds_id values excluded from later additions ---")

excluded_ids = con.execute("""
    WITH v1_ids AS (
        SELECT DISTINCT cds_id
        FROM thesis_musicbrainz_enrichment_master_v1
    ),
    later_additions AS (
        SELECT DISTINCT cds_id, 'cds_mb_enrichment_review_v1' AS source
        FROM cds_mb_enrichment_review_v1

        UNION ALL

        SELECT DISTINCT cds_id, 'cds_only_enrichment_review_true_unmatched_v1' AS source
        FROM cds_only_enrichment_review_true_unmatched_v1
    )
    SELECT
        l.cds_id,
        string_agg(l.source, ', ' ORDER BY l.source) AS overlapping_source
    FROM later_additions l
    JOIN v1_ids v
      ON l.cds_id = v.cds_id
    GROUP BY l.cds_id
    ORDER BY l.cds_id
""").fetchdf()

print(excluded_ids)

print(f"\nNumber of excluded overlapping cds_id values: {len(excluded_ids)}")

Connected ✅
--- Overlapping cds_id values excluded from later additions ---
   cds_id                            overlapping_source
0  103932  cds_only_enrichment_review_true_unmatched_v1
1  105165  cds_only_enrichment_review_true_unmatched_v1
2  108945  cds_only_enrichment_review_true_unmatched_v1

Number of excluded overlapping cds_id values: 3


In [97]:
print("--- Excluded cds_id values with CDS context ---")

print(con.execute("""
    WITH excluded_ids AS (
        WITH v1_ids AS (
            SELECT DISTINCT cds_id
            FROM thesis_musicbrainz_enrichment_master_v1
        ),
        later_additions AS (
            SELECT DISTINCT cds_id
            FROM cds_mb_enrichment_review_v1

            UNION

            SELECT DISTINCT cds_id
            FROM cds_only_enrichment_review_true_unmatched_v1
        )
        SELECT l.cds_id
        FROM later_additions l
        JOIN v1_ids v
          ON l.cds_id = v.cds_id
    )
    SELECT
        e.cds_id,
        c.artist,
        c.title,
        c.cds_year
    FROM excluded_ids e
    LEFT JOIN cds c
      ON e.cds_id = c.cds_id
    ORDER BY e.cds_id
""").fetchdf())

--- Excluded cds_id values with CDS context ---
   cds_id      artist               title  cds_year
0  103932  zã ramalho    grandes sucessos      <NA>
1  105165        gang  le radici e le ali      <NA>
2  108945     vivaldi    the four seasons      <NA>


In [98]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_weak_v1
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT *
    FROM cds_only_enrichment_weak_v1
    LIMIT 30
""").fetchdf())

print("\n--- Basic quality profile ---")
print(con.execute("""
    SELECT
        SUM(CASE WHEN artist IS NULL OR trim(artist) = '' THEN 1 ELSE 0 END) AS missing_artist,
        SUM(CASE WHEN title IS NULL OR trim(title) = '' THEN 1 ELSE 0 END) AS missing_title,
        SUM(CASE WHEN lower(coalesce(artist, '')) LIKE '%ã%'
               OR lower(coalesce(title, '')) LIKE '%ã%'
               OR lower(coalesce(tracks, '')) LIKE '%ã%'
            THEN 1 ELSE 0 END) AS obvious_mojibake_like,
        SUM(CASE WHEN length(coalesce(artist_key_clean, '')) < 3 THEN 1 ELSE 0 END) AS short_artist_key,
        SUM(CASE WHEN length(coalesce(title_key_clean, '')) < 3 THEN 1 ELSE 0 END) AS short_title_key
    FROM cds_only_enrichment_weak_v1
""").fetchdf())

print("\n--- Breakdown by route / type ---")
print(con.execute("""
    SELECT
        curation_route,
        cds_enrichment_type,
        COUNT(*) AS n
    FROM cds_only_enrichment_weak_v1
    GROUP BY curation_route, cds_enrichment_type
    ORDER BY n DESC
""").fetchdf())

Connected ✅
--- Row count ---
     n
0  196

--- Preview ---
    cds_id               artist                                    title  \
0   100048         the breeders                                      pod   
1   100049      johnny hallyday                                    vol 4   
2   100070          perseguidos                                      iii   
3   100136            joan baez                                        5   
4   100144             doe maar                                      4us   
5   100224           edith piaf                                      n 3   
6   100248         wilbury clan                                     live   
7   100257               yarkin                                      ten   
8   100270        dara bubamara                                     2001   
9   100333  minstrels of mayhem                                      now   
10  100371        pierre perret                                 ã ui lã    
11  100390     mannik grigo

In [99]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Salvage readable rows from the weak bucket
# ------------------------------------------------------------
# Comments:
# - Do NOT trust the whole weak table.
# - Keep only rows that are still readable and structurally usable enough
#   for manual review-level enrichment.
print("Creating cds_only_enrichment_weak_salvaged_v1...")

con.execute(r"""
    CREATE OR REPLACE TABLE cds_only_enrichment_weak_salvaged_v1 AS
    SELECT *
    FROM cds_only_enrichment_weak_v1
    WHERE
        artist IS NOT NULL
        AND trim(artist) <> ''
        AND title IS NOT NULL
        AND trim(title) <> ''
        AND length(coalesce(artist_key_clean, '')) >= 3
        AND length(coalesce(title_key_clean, '')) >= 3

        -- exclude obvious mojibake-like corruption
        AND NOT regexp_matches(lower(coalesce(artist, '')), '(ã|â|ð|�)')
        AND NOT regexp_matches(lower(coalesce(title, '')), '(ã|â|ð|�)')
        AND NOT regexp_matches(lower(coalesce(tracks, '')), '(ã|â|ð|�)')

        -- exclude ultra-generic weak titles
        AND lower(trim(coalesce(title, ''))) NOT IN (
            'vol 1', 'vol 2', 'vol 3', 'disc 1', 'disc 2', 'cd 1', 'cd 2'
        )
""")

print("✅ Table created: cds_only_enrichment_weak_salvaged_v1")

print("\n--- Salvaged weak rows ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_weak_salvaged_v1
""").fetchdf())

print("\n--- Preview salvaged weak rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_weak_salvaged_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating cds_only_enrichment_weak_salvaged_v1...
✅ Table created: cds_only_enrichment_weak_salvaged_v1

--- Salvaged weak rows ---
    n
0  47

--- Preview salvaged weak rows ---
    cds_id                artist     title  cds_year  \
0   100048          the breeders       pod      1990   
1   100070           perseguidos       iii      2004   
2   100144              doe maar       4us      <NA>   
3   100270         dara bubamara      2001      2001   
4   100333   minstrels of mayhem       now      <NA>   
5   100390      mannik grigorian       van      <NA>   
6   100923             soulciety     s t p      <NA>   
7   101096            boz scaggs       dig      <NA>   
8   101957       antonio palombo       lui      <NA>   
9   102155                  hint      dys       <NA>   
10  102374          wave in head       you      2002   
11  102551             jiri korn       tep      2002   
12  102565     harry connick jr        she      <NA>   
13  102697             se

In [100]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Weak-bucket handling:
# 1) try to recover MB presence from salvaged weak rows
# 2) keep recovered rows only as MB REVIEW enrichment
# 3) keep non-recovered salvaged rows as CDS-only REVIEW enrichment
# 4) put the rest back into excluded weak
# ------------------------------------------------------------
# Comments:
# - We do NOT create automatic enrichment from the weak bucket.
# - Even if a weak row is found in MusicBrainz, it stays review-only.
# - This is the conservative thesis-safe treatment.

# ------------------------------------------------------------
# Step 1: Recover MB-present rows from salvaged weak rows
# ------------------------------------------------------------
print("Creating cds_only_weak_recovered_mb_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_weak_recovered_mb_v1 AS
    WITH weak_salvaged AS (
        SELECT
            w.cds_id,
            w.artist,
            w.title,
            w.cds_year,
            w.tracks,
            w.artist_key_clean,
            w.title_key_clean,
            w.curation_route,
            w.enrichment_priority_v6,
            w.cds_enrichment_type
        FROM cds_only_enrichment_weak_salvaged_v1 w
    ),
    mb_norm AS (
        SELECT
            m.release_id,
            m.release_title,
            m.main_artist_name,
            m.artist_credit_full,
            m.total_tracks,
            m.barcode,
            m.mb_year,
            lower(regexp_replace(coalesce(m.main_artist_name, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_artist_key,
            lower(regexp_replace(coalesce(m.release_title, ''), '[^a-zA-Z0-9]+', '', 'g')) AS mb_title_key
        FROM mb_release_features m
    )
    SELECT
        w.cds_id,
        w.artist,
        w.title,
        w.cds_year,
        w.tracks,
        w.artist_key_clean,
        w.title_key_clean,
        w.curation_route,
        w.enrichment_priority_v6,
        w.cds_enrichment_type,

        m.release_id,
        m.release_title,
        m.main_artist_name,
        m.artist_credit_full,
        m.total_tracks,
        m.barcode,
        m.mb_year
    FROM weak_salvaged w
    JOIN mb_norm m
      ON w.artist_key_clean = m.mb_artist_key
     AND w.title_key_clean = m.mb_title_key
""")

print("✅ Table created: cds_only_weak_recovered_mb_v1")

print("\n--- Recovered candidate rows from salvaged weak ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_weak_recovered_mb_v1
""").fetchdf())

print("\n--- Distinct CDS rows recovered from salvaged weak ---")
print(con.execute("""
    SELECT COUNT(DISTINCT cds_id) AS n
    FROM cds_only_weak_recovered_mb_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 2: Rank recovered weak rows
# ------------------------------------------------------------
# Comments:
# - No track-gap is used because CDS tracks are text, not numeric count.
# - Rank by year plausibility + barcode presence.
print("\nCreating cds_only_weak_recovered_ranked_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_weak_recovered_ranked_v1 AS
    WITH scored AS (
        SELECT
            r.*,

            CASE
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                    THEN abs(r.cds_year - r.mb_year)
                ELSE NULL
            END AS year_gap,

            CASE
                WHEN r.barcode IS NOT NULL AND trim(r.barcode) <> '' THEN 1
                ELSE 0
            END AS has_barcode,

            CASE
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) = 0 THEN 100
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) = 1 THEN 90
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) <= 3 THEN 75
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NOT NULL
                     AND abs(r.cds_year - r.mb_year) <= 5 THEN 60
                WHEN r.cds_year IS NOT NULL AND r.mb_year IS NULL THEN 40
                WHEN r.cds_year IS NULL AND r.mb_year IS NOT NULL THEN 30
                ELSE 20
            END
            +
            CASE
                WHEN r.barcode IS NOT NULL AND trim(r.barcode) <> '' THEN 5
                ELSE 0
            END AS recovery_rank_score
        FROM cds_only_weak_recovered_mb_v1 r
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY cds_id
                ORDER BY
                    recovery_rank_score DESC,
                    year_gap ASC NULLS LAST,
                    has_barcode DESC,
                    mb_year ASC NULLS LAST,
                    release_id
            ) AS rn,

            COUNT(*) OVER (
                PARTITION BY cds_id
            ) AS candidate_count,

            LEAD(recovery_rank_score) OVER (
                PARTITION BY cds_id
                ORDER BY
                    recovery_rank_score DESC,
                    year_gap ASC NULLS LAST,
                    has_barcode DESC,
                    mb_year ASC NULLS LAST,
                    release_id
            ) AS second_score
        FROM scored
    )
    SELECT
        *,
        recovery_rank_score - coalesce(second_score, 0) AS recovery_score_gap
    FROM ranked
""")

print("✅ Table created: cds_only_weak_recovered_ranked_v1")

print("\n--- Candidate count by CDS row ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        candidate_count
    FROM cds_only_weak_recovered_ranked_v1
    WHERE rn = 1
    ORDER BY candidate_count DESC, cds_id
""").fetchdf())

# ------------------------------------------------------------
# Step 3: Weak rows found in MB -> REVIEW only
# ------------------------------------------------------------
print("\nCreating cds_mb_enrichment_weak_review_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_mb_enrichment_weak_review_v1 AS
    SELECT
        r.cds_id,
        c.artist AS cds_artist,
        c.title AS cds_title,
        c.cds_year,
        c.category,
        c.genre,
        c.tracks AS cds_tracklist,

        r.release_id,
        r.release_title,
        r.main_artist_name,
        r.artist_credit_full,
        r.total_tracks,
        r.barcode,
        r.mb_year,

        r.year_gap,
        r.recovery_rank_score,
        r.recovery_score_gap,
        r.candidate_count,

        'weak_mb_review_recovered' AS match_type,
        'review_mb_enrichment' AS enrichment_status
    FROM cds_only_weak_recovered_ranked_v1 r
    LEFT JOIN cds c
      ON r.cds_id = c.cds_id
    WHERE r.rn = 1
""")

print("✅ Table created: cds_mb_enrichment_weak_review_v1")

print("\n--- Weak rows reclassified to MB review enrichment ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_mb_enrichment_weak_review_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 4: Salvaged weak rows not found in MB -> CDS-only REVIEW
# ------------------------------------------------------------
print("\nCreating cds_only_enrichment_weak_review_v1...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_weak_review_v1 AS
    SELECT *
    FROM cds_only_enrichment_weak_salvaged_v1 s
    WHERE s.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_mb_enrichment_weak_review_v1
    )
""")

print("✅ Table created: cds_only_enrichment_weak_review_v1")

print("\n--- Salvaged weak rows kept as CDS-only review enrichment ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM cds_only_enrichment_weak_review_v1
""").fetchdf())

# ------------------------------------------------------------
# Step 5: Remaining weak rows stay excluded
# ------------------------------------------------------------
print("\nCreating cds_only_enrichment_weak_excluded_v2...")

con.execute("""
    CREATE OR REPLACE TABLE cds_only_enrichment_weak_excluded_v2 AS
    SELECT *
    FROM cds_only_enrichment_weak_v1 w
    WHERE w.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_mb_enrichment_weak_review_v1
    )
    AND w.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM cds_only_enrichment_weak_review_v1
    )
""")

print("✅ Table created: cds_only_enrichment_weak_excluded_v2")

print("\n--- Final weak split summary ---")
print(con.execute("""
    SELECT 'mb_review_enrichment' AS bucket, COUNT(*) AS n
    FROM cds_mb_enrichment_weak_review_v1
    UNION ALL
    SELECT 'cds_only_review' AS bucket, COUNT(*) AS n
    FROM cds_only_enrichment_weak_review_v1
    UNION ALL
    SELECT 'excluded' AS bucket, COUNT(*) AS n
    FROM cds_only_enrichment_weak_excluded_v2
""").fetchdf())

print("\n--- Preview: weak MB review enrichment ---")
print(con.execute("""
    SELECT
        cds_id,
        cds_artist,
        cds_title,
        release_id,
        release_title,
        main_artist_name,
        cds_year,
        mb_year,
        year_gap,
        candidate_count,
        recovery_rank_score
    FROM cds_mb_enrichment_weak_review_v1
    ORDER BY cds_id
    LIMIT 20
""").fetchdf())

print("\n--- Preview: weak CDS-only review enrichment ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_weak_review_v1
    ORDER BY cds_id
    LIMIT 20
""").fetchdf())

print("\n--- Preview: weak excluded rows ---")
print(con.execute("""
    SELECT
        cds_id,
        artist,
        title,
        cds_year,
        curation_route,
        enrichment_priority_v6,
        cds_enrichment_type
    FROM cds_only_enrichment_weak_excluded_v2
    ORDER BY cds_id
    LIMIT 20
""").fetchdf())

Connected ✅
Creating cds_only_weak_recovered_mb_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: cds_only_weak_recovered_mb_v1

--- Recovered candidate rows from salvaged weak ---
    n
0  87

--- Distinct CDS rows recovered from salvaged weak ---
    n
0  23

Creating cds_only_weak_recovered_ranked_v1...
✅ Table created: cds_only_weak_recovered_ranked_v1

--- Candidate count by CDS row ---
    cds_id                artist  title  candidate_count
0   104203            faith hill    cry               12
1   104917              bee gees  e s p               12
2   100048          the breeders    pod               10
3   104795                mansun    six               10
4   106992             testament    low                7
5     6374               maxwell    now                6
6   100144              doe maar    4us                5
7   104603         peter gabriel    ovo                4
8   101096            boz scaggs    dig                3
9   102565     harry connick jr     she                3
10  105990            fur patrol    pet                2
11

In [101]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Extend the thesis master enrichment table with weak-review buckets
# ------------------------------------------------------------
# Comments:
# - We keep excluded weak rows OUTSIDE the final enrichment table.
# - We add only:
#     1) weak MB review enrichment
#     2) weak CDS-only review enrichment
# - This gives a fuller thesis-facing enrichment output without
#   pretending weak rows are safe or automatic.
print("Creating thesis_musicbrainz_enrichment_master_v4...")

con.execute("""
    CREATE OR REPLACE TABLE thesis_musicbrainz_enrichment_master_v4 AS

    -- --------------------------------------------------------
    -- A. Keep everything already present in v3
    -- --------------------------------------------------------
    SELECT *
    FROM thesis_musicbrainz_enrichment_master_v3

    UNION ALL

    -- --------------------------------------------------------
    -- B. Add weak rows that are present in MusicBrainz
    --    -> review-level MB enrichment only
    -- --------------------------------------------------------
    SELECT
        w.cds_id,
        w.cds_artist AS artist,
        w.cds_title AS title,
        w.cds_year,
        w.category,
        w.genre,
        w.cds_tracklist AS tracks,

        w.release_id,
        w.release_title,
        w.main_artist_name,
        w.artist_credit_full,
        w.total_tracks,
        w.barcode,
        w.mb_year,

        'review_mb_enrichment_weak' AS enrichment_route,
        'review_mb_enrichment_weak' AS enrichment_status
    FROM cds_mb_enrichment_weak_review_v1 w
    WHERE w.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM thesis_musicbrainz_enrichment_master_v3
    )

    UNION ALL

    -- --------------------------------------------------------
    -- C. Add weak rows still not found in MusicBrainz
    --    -> review-level CDS-only enrichment
    -- --------------------------------------------------------
    SELECT
        w.cds_id,
        w.artist,
        w.title,
        w.cds_year,
        c.category,
        c.genre,
        c.tracks,

        NULL AS release_id,
        NULL AS release_title,
        NULL AS main_artist_name,
        NULL AS artist_credit_full,
        NULL AS total_tracks,
        NULL AS barcode,
        NULL AS mb_year,

        'cds_only_review_weak' AS enrichment_route,
        'new_cds_enrichment_review_weak' AS enrichment_status
    FROM cds_only_enrichment_weak_review_v1 w
    LEFT JOIN cds c
      ON w.cds_id = c.cds_id
    WHERE w.cds_id NOT IN (
        SELECT DISTINCT cds_id
        FROM thesis_musicbrainz_enrichment_master_v3
    )
""")

print("✅ Table created: thesis_musicbrainz_enrichment_master_v4")

print("\n--- Final enrichment summary ---")
print(con.execute("""
    SELECT
        enrichment_status,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM thesis_musicbrainz_enrichment_master_v4), 2) AS pct
    FROM thesis_musicbrainz_enrichment_master_v4
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by route ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Total rows in v4 ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
""").fetchdf())

print("\n--- Weak additions actually included ---")
print(con.execute("""
    SELECT 'mb_weak_review_added' AS bucket, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE enrichment_route = 'review_mb_enrichment_weak'

    UNION ALL

    SELECT 'cds_weak_review_added' AS bucket, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE enrichment_route = 'cds_only_review_weak'
""").fetchdf())

Connected ✅
Creating thesis_musicbrainz_enrichment_master_v4...
✅ Table created: thesis_musicbrainz_enrichment_master_v4

--- Final enrichment summary ---
                enrichment_status     n    pct
0         confirmed_mb_enrichment  4214  46.71
1       new_cds_enrichment_review  2977  33.00
2              new_cds_enrichment  1646  18.25
3            review_mb_enrichment   137   1.52
4  new_cds_enrichment_review_weak    24   0.27
5       review_mb_enrichment_weak    23   0.25

--- Breakdown by route ---
                  enrichment_route     n
0                            exact  3482
1                  cds_only_review  2977
2                    cds_only_safe  1646
3                fuzzy_second_pass   543
4               various_title_only   183
5   review_mb_enrichment_recovered    94
6         matched_review_candidate    43
7             cds_only_review_weak    24
8        review_mb_enrichment_weak    23
9           enrichment_round2_auto     3
10           alias_enrichment_safe   

In [102]:
import os

os.makedirs("../data_processed/exports/final_thesis_outputs", exist_ok=True)

out_path = "../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v4.csv"

con.execute(f"""
    COPY thesis_musicbrainz_enrichment_master_v4
    TO '{out_path}'
    WITH (HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"')
""")

print(f"✅ Exported: {out_path}")

✅ Exported: ../data_processed/exports/final_thesis_outputs/thesis_musicbrainz_enrichment_master_v4.csv


In [103]:
import duckdb
from pathlib import Path

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Export all final thesis tables
# ------------------------------------------------------------
# Comments:
# - These are the final tables supporting the thesis results.
# - The exports are written as CSV files to a dedicated folder.
# - QUOTE/ESCAPE are set explicitly for safer CSV output.
export_dir = Path("../data_processed/exports/final_thesis_outputs")
export_dir.mkdir(parents=True, exist_ok=True)

final_tables = [
    
    "cds_mb_enrichment_review_v1",
    "cds_only_enrichment_safe_true_unmatched_v1",
    "cds_only_enrichment_review_true_unmatched_v1",
    "cds_mb_enrichment_weak_review_v1",
    "cds_only_enrichment_weak_review_v1",
    "cds_only_enrichment_weak_excluded_v2"
]

print("Exporting final thesis tables...\n")

for table in final_tables:
    out_path = export_dir / f"{table}.csv"
    con.execute(f"""
        COPY {table}
        TO '{out_path.as_posix()}'
        WITH (
            HEADER,
            DELIMITER ',',
            QUOTE '"',
            ESCAPE '"'
        )
    """)
    print(f"✅ Exported: {out_path.name}")

print("\nDone ✅")
print(f"Folder: {export_dir}")

Connected ✅
Exporting final thesis tables...

✅ Exported: cds_mb_enrichment_review_v1.csv
✅ Exported: cds_only_enrichment_safe_true_unmatched_v1.csv
✅ Exported: cds_only_enrichment_review_true_unmatched_v1.csv
✅ Exported: cds_mb_enrichment_weak_review_v1.csv
✅ Exported: cds_only_enrichment_weak_review_v1.csv
✅ Exported: cds_only_enrichment_weak_excluded_v2.csv

Done ✅
Folder: ..\data_processed\exports\final_thesis_outputs


In [104]:
import duckdb
from pathlib import Path

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Export all final thesis tables
# ------------------------------------------------------------
# Comments:
# - These are the final tables supporting the thesis results.
# - The exports are written as CSV files to a dedicated folder.
# - QUOTE/ESCAPE are set explicitly for safer CSV output.
export_dir = Path("../data_processed/exports")
export_dir.mkdir(parents=True, exist_ok=True)

final_tables = [
    
    "cds_unmatched_excluded_v1",
  
]


for table in final_tables:
    out_path = export_dir / f"{table}.csv"
    con.execute(f"""
        COPY {table}
        TO '{out_path.as_posix()}'
        WITH (
            HEADER,
            DELIMITER ',',
            QUOTE '"',
            ESCAPE '"'
        )
    """)
    print(f"✅ Exported: {out_path.name}")

print("\nDone ✅")
print(f"Folder: {export_dir}")

Connected ✅
✅ Exported: cds_unmatched_excluded_v1.csv

Done ✅
Folder: ..\data_processed\exports


In [132]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build a MusicBrainz-centered release table enriched from CDS
# ------------------------------------------------------------
# Comments:
# - Base table: mb_release_features
# - Enrichment evidence: thesis_musicbrainz_enrichment_master_v4
# - We only use rows already linked to a MusicBrainz release
#   (release_id IS NOT NULL).
# - Since multiple CDS-derived enrichment rows may point to the same
#   MusicBrainz release, we choose exactly one "best" enrichment row
#   per release_id using a conservative ranking policy:
#       1) confirmed MB enrichment before review MB enrichment
#       2) non-weak routes before weak review routes
#       3) richer metadata preferred (genre/category/tracks/year)
#       4) stable tie-break by cds_id
# - Important correction:
#   enrichment_source and is_enriched_from_cds must only be set when
#   a CDS enrichment row actually exists for that release.
print("Creating corrected musicbrainz_release_enriched_from_cds_v1...")

con.execute("""
    CREATE OR REPLACE TABLE musicbrainz_release_enriched_from_cds_v1 AS
    WITH mb_evidence AS (
        SELECT
            e.*,

            -- ----------------------------------------------------
            -- Route priority:
            -- confirmed enrichment is strongest,
            -- then normal review,
            -- then weak review
            -- ----------------------------------------------------
            CASE
                WHEN e.enrichment_status = 'confirmed_mb_enrichment' THEN 1
                WHEN e.enrichment_status = 'review_mb_enrichment' THEN 2
                WHEN e.enrichment_status = 'review_mb_enrichment_weak' THEN 3
                ELSE 9
            END AS status_priority,

            -- ----------------------------------------------------
            -- Metadata richness:
            -- prefer rows that actually contribute more information
            -- ----------------------------------------------------
            (
                CASE WHEN e.category IS NOT NULL AND trim(e.category) <> '' THEN 1 ELSE 0 END +
                CASE WHEN e.genre IS NOT NULL AND trim(e.genre) <> '' THEN 1 ELSE 0 END +
                CASE WHEN e.tracks IS NOT NULL AND trim(e.tracks) <> '' THEN 1 ELSE 0 END +
                CASE WHEN e.cds_year IS NOT NULL THEN 1 ELSE 0 END
            ) AS metadata_richness

        FROM thesis_musicbrainz_enrichment_master_v4 e
        WHERE e.release_id IS NOT NULL
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY release_id
                ORDER BY
                    status_priority ASC,
                    metadata_richness DESC,
                    cds_year ASC NULLS LAST,
                    cds_id ASC
            ) AS rn,

            COUNT(*) OVER (
                PARTITION BY release_id
            ) AS source_record_count

        FROM mb_evidence
    )
    SELECT
        -- --------------------------------------------------------
        -- Original MusicBrainz release fields
        -- --------------------------------------------------------
        mb.release_id,
        mb.release_title,
        mb.barcode,
        mb.artist_credit_id,
        mb.artist_credit_full,
        mb.main_artist_name,
        mb.mb_year,
        mb.total_tracks,

        -- --------------------------------------------------------
        -- CDS-derived enrichment fields
        -- --------------------------------------------------------
        r.cds_id AS enrichment_cds_id,
        r.artist AS source_artist,
        r.title AS source_title,
        r.cds_year AS source_year,
        r.category AS enriched_category,
        r.genre AS enriched_genre,
        r.tracks AS external_tracklist,

        -- --------------------------------------------------------
        -- Provenance / enrichment metadata
        -- Only set if enrichment really exists
        -- --------------------------------------------------------
        r.enrichment_route,
        r.enrichment_status,

        CASE
            WHEN r.cds_id IS NOT NULL THEN 'CDS'
            ELSE NULL
        END AS enrichment_source,

        CASE
            WHEN r.cds_id IS NOT NULL THEN r.source_record_count
            ELSE NULL
        END AS source_record_count,

        CASE
            WHEN r.cds_id IS NOT NULL THEN TRUE
            ELSE FALSE
        END AS is_enriched_from_cds

    FROM mb_release_features mb
    LEFT JOIN ranked r
      ON mb.release_id = r.release_id
     AND r.rn = 1
""")

print("✅ Table created: musicbrainz_release_enriched_from_cds_v1")

print("\n--- Row count / uniqueness check ---")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT release_id) AS distinct_release_id,
        COUNT(*) - COUNT(DISTINCT release_id) AS duplicate_release_rows
    FROM musicbrainz_release_enriched_from_cds_v1
""").fetchdf())

print("\n--- Enrichment coverage report ---")
print(con.execute("""
    SELECT
        COUNT(*) AS mb_total_releases,
        COUNT(*) FILTER (WHERE is_enriched_from_cds = TRUE) AS mb_releases_with_cds_enrichment,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE is_enriched_from_cds = TRUE) / COUNT(*),
            4
        ) AS pct_enriched_from_cds
    FROM musicbrainz_release_enriched_from_cds_v1
""").fetchdf())

print("\n--- Preview: non-enriched rows should now have NULL source and FALSE flag ---")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        enrichment_cds_id,
        enrichment_source,
        source_record_count,
        is_enriched_from_cds
    FROM musicbrainz_release_enriched_from_cds_v1
    WHERE enrichment_cds_id IS NULL
    LIMIT 20
""").fetchdf())

print("\n--- Preview: enriched rows ---")
print(con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        enriched_category,
        enriched_genre,
        enrichment_source,
        source_record_count,
        is_enriched_from_cds
    FROM musicbrainz_release_enriched_from_cds_v1
    WHERE is_enriched_from_cds = TRUE
    LIMIT 20
""").fetchdf())

Connected ✅
Creating corrected musicbrainz_release_enriched_from_cds_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: musicbrainz_release_enriched_from_cds_v1

--- Row count / uniqueness check ---
   total_rows  distinct_release_id  duplicate_release_rows
0     5309878              5309878                       0

--- Enrichment coverage report ---
   mb_total_releases  mb_releases_with_cds_enrichment  pct_enriched_from_cds
0            5309878                             4200                 0.0791

--- Preview: non-enriched rows should now have NULL source and FALSE flag ---
    release_id  \
0      2608361   
1      2794510   
2      2608364   
3      2608363   
4      2608362   
5      1644640   
6       463456   
7      2608365   
8      2782819   
9      2787252   
10     2788717   
11     2787253   
12     2777624   
13     2750291   
14     4965994   
15     2766859   
16     2785878   
17     2608366   
18     2912117   
19     3237600   

                                                                                       release_title  \
0                                 

In [133]:
import duckdb
con = duckdb.connect("../data_processed/thesis.duckdb")

# True enrichment count (no indentation)
result = con.execute("""
    SELECT 
        COUNT(*) AS total_releases,
        COUNT(*) FILTER (WHERE enrichment_status IS NOT NULL) AS actually_enriched,
        ROUND(100.0 * COUNT(*) FILTER (WHERE enrichment_status IS NOT NULL) / COUNT(*), 4) AS pct
    FROM musicbrainz_release_enriched_from_cds_v1
""").fetchdf()

print(result)

   total_releases  actually_enriched     pct
0         5309878               4200  0.0791


In [134]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build a residual external enrichment table for CDS-only rows
# ------------------------------------------------------------
# Comments:
# - This table keeps enrichment evidence from CDS that could not be
#   safely linked to an existing MusicBrainz release.
# - These rows are still useful for the thesis because they represent
#   potential new external music-enrichment evidence.
# - We only use rows from the thesis master enrichment table where
#   release_id IS NULL, i.e. no confirmed MusicBrainz release link.
# - This table is separate from the MusicBrainz-centered enriched
#   release table because these records are not safe to merge directly
#   into mb_release_features.
print("Creating musicbrainz_external_residual_enrichment_v1...")

con.execute("""
    CREATE OR REPLACE TABLE musicbrainz_external_residual_enrichment_v1 AS
    SELECT
        -- --------------------------------------------------------
        -- External source identifiers / metadata
        -- --------------------------------------------------------
        cds_id AS external_record_id,
        artist AS external_artist,
        title AS external_title,
        cds_year AS external_year,
        category AS external_category,
        genre AS external_genre,
        tracks AS external_tracklist,

        -- --------------------------------------------------------
        -- Enrichment status / provenance
        -- --------------------------------------------------------
        enrichment_route,
        enrichment_status,
        'CDS' AS enrichment_source,

        -- --------------------------------------------------------
        -- Target-side placeholders:
        -- no confirmed MusicBrainz release was found
        -- --------------------------------------------------------
        NULL::BIGINT AS release_id,
        NULL::VARCHAR AS release_title,
        NULL::VARCHAR AS main_artist_name,
        NULL::VARCHAR AS artist_credit_full,
        NULL::BIGINT AS total_tracks,
        NULL::VARCHAR AS barcode,
        NULL::INTEGER AS mb_year,

        FALSE AS is_linked_to_musicbrainz

    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE release_id IS NULL
""")

print("✅ Table created: musicbrainz_external_residual_enrichment_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM musicbrainz_external_residual_enrichment_v1
""").fetchdf())

print("\n--- Breakdown by enrichment status ---")
print(con.execute("""
    SELECT
        enrichment_status,
        COUNT(*) AS n
    FROM musicbrainz_external_residual_enrichment_v1
    GROUP BY enrichment_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Breakdown by enrichment route ---")
print(con.execute("""
    SELECT
        enrichment_route,
        COUNT(*) AS n
    FROM musicbrainz_external_residual_enrichment_v1
    GROUP BY enrichment_route
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        external_record_id,
        external_artist,
        external_title,
        external_year,
        external_category,
        external_genre,
        enrichment_route,
        enrichment_status,
        enrichment_source,
        is_linked_to_musicbrainz
    FROM musicbrainz_external_residual_enrichment_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating musicbrainz_external_residual_enrichment_v1...
✅ Table created: musicbrainz_external_residual_enrichment_v1

--- Row count ---
      n
0  4647

--- Breakdown by enrichment status ---
                enrichment_status     n
0       new_cds_enrichment_review  2977
1              new_cds_enrichment  1646
2  new_cds_enrichment_review_weak    24

--- Breakdown by enrichment route ---
       enrichment_route     n
0       cds_only_review  2977
1         cds_only_safe  1646
2  cds_only_review_weak    24

--- Preview ---
    external_record_id                               external_artist  \
0               100004                                       emanuel   
1               100028                               jazz signatures   
2               100045                               matsuka phonics   
3               100046                                    kool savas   
4               100050                                       phallus   
5               100062      

In [135]:
import duckdb
import pandas as pd

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Build one unified base table for the next WebTable enrichment step
# ------------------------------------------------------------
# Comments:
# - This table combines:
#     1) MusicBrainz releases already enriched from CDS
#     2) MusicBrainz releases not enriched from CDS
#     3) residual external enrichment rows not yet linked to MusicBrainz
# - Goal:
#   create a single input table for the next WebTable enrichment phase.
# - Important correction:
#   rows coming from musicbrainz_release_enriched_from_cds_v1 are NOT all
#   "linked_mb_release". Some are enriched from CDS, others are simply
#   untouched MusicBrainz releases.
print("Creating corrected music_enrichment_base_for_webtables_v1...")

con.execute("""
    CREATE OR REPLACE TABLE music_enrichment_base_for_webtables_v1 AS

    -- --------------------------------------------------------
    -- A. All MusicBrainz release rows
    --    distinguish enriched vs not enriched
    -- --------------------------------------------------------
    SELECT
        CAST(release_id AS BIGINT) AS base_record_id,

        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        barcode,
        mb_year,
        total_tracks,

        enrichment_cds_id AS external_record_id,
        source_artist AS external_artist,
        source_title AS external_title,
        source_year AS external_year,
        enriched_category AS external_category,
        enriched_genre AS external_genre,
        external_tracklist,

        enrichment_route,
        enrichment_status,
        enrichment_source,
        is_enriched_from_cds AS is_linked_to_musicbrainz,

        CASE
            WHEN is_enriched_from_cds = TRUE THEN 'mb_release_enriched_from_cds'
            ELSE 'mb_release_not_enriched'
        END AS base_record_status

    FROM musicbrainz_release_enriched_from_cds_v1

    UNION ALL

    -- --------------------------------------------------------
    -- B. External residual rows not linked to MusicBrainz
    -- --------------------------------------------------------
    SELECT
        CAST(external_record_id AS BIGINT) AS base_record_id,

        release_id,
        release_title,
        main_artist_name,
        artist_credit_full,
        barcode,
        mb_year,
        total_tracks,

        external_record_id,
        external_artist,
        external_title,
        external_year,
        external_category,
        external_genre,
        external_tracklist,

        enrichment_route,
        enrichment_status,
        enrichment_source,
        is_linked_to_musicbrainz,

        'external_residual' AS base_record_status

    FROM musicbrainz_external_residual_enrichment_v1
""")

print("✅ Table created: music_enrichment_base_for_webtables_v1")

print("\n--- Row count ---")
print(con.execute("""
    SELECT COUNT(*) AS n
    FROM music_enrichment_base_for_webtables_v1
""").fetchdf())

print("\n--- Breakdown by base record status ---")
print(con.execute("""
    SELECT
        base_record_status,
        COUNT(*) AS n
    FROM music_enrichment_base_for_webtables_v1
    GROUP BY base_record_status
    ORDER BY n DESC
""").fetchdf())

print("\n--- Linked vs not linked to MusicBrainz ---")
print(con.execute("""
    SELECT
        is_linked_to_musicbrainz,
        COUNT(*) AS n
    FROM music_enrichment_base_for_webtables_v1
    GROUP BY is_linked_to_musicbrainz
    ORDER BY n DESC
""").fetchdf())

print("\n--- Preview ---")
print(con.execute("""
    SELECT
        base_record_id,
        base_record_status,
        release_id,
        release_title,
        main_artist_name,
        external_record_id,
        external_artist,
        external_title,
        enrichment_status,
        enrichment_source,
        is_linked_to_musicbrainz
    FROM music_enrichment_base_for_webtables_v1
    LIMIT 30
""").fetchdf())

Connected ✅
Creating corrected music_enrichment_base_for_webtables_v1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Table created: music_enrichment_base_for_webtables_v1

--- Row count ---
         n
0  5314525

--- Breakdown by base record status ---
             base_record_status        n
0       mb_release_not_enriched  5305678
1             external_residual     4647
2  mb_release_enriched_from_cds     4200

--- Linked vs not linked to MusicBrainz ---
   is_linked_to_musicbrainz        n
0                     False  5310325
1                      True     4200

--- Preview ---
    base_record_id       base_record_status  release_id  \
0          2608361  mb_release_not_enriched     2608361   
1          2794510  mb_release_not_enriched     2794510   
2          2608364  mb_release_not_enriched     2608364   
3          2608363  mb_release_not_enriched     2608363   
4          2608362  mb_release_not_enriched     2608362   
5          1644640  mb_release_not_enriched     1644640   
6           463456  mb_release_not_enriched      463456   
7          2608365  mb_release_not_enriched     26083

In [136]:
import duckdb
from pathlib import Path

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ------------------------------------------------------------
# Export all final thesis tables
# ------------------------------------------------------------
# Comments:
# - These are the final tables supporting the thesis results.
# - The exports are written as CSV files to a dedicated folder.
# - QUOTE/ESCAPE are set explicitly for safer CSV output.
export_dir = Path("../data_processed/exports")
export_dir.mkdir(parents=True, exist_ok=True)

final_tables = [
    
    "musicbrainz_external_residual_enrichment_v1",
    "musicbrainz_release_enriched_from_cds_v1",
    "music_enrichment_base_for_webtables_v1"

  
]


for table in final_tables:
    out_path = export_dir / f"{table}.csv"
    con.execute(f"""
        COPY {table}
        TO '{out_path.as_posix()}'
        WITH (
            HEADER,
            DELIMITER ',',
            QUOTE '"',
            ESCAPE '"'
        )
    """)
    print(f"✅ Exported: {out_path.name}")

print("\nDone ✅")
print(f"Folder: {export_dir}")

Connected ✅
✅ Exported: musicbrainz_external_residual_enrichment_v1.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: musicbrainz_release_enriched_from_cds_v1.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: music_enrichment_base_for_webtables_v1.csv

Done ✅
Folder: ..\data_processed\exports


In [137]:
print(con.execute("""
    SELECT
        COUNT(*) AS overlap_n
    FROM music_enrichment_base_for_webtables_v1 a
    JOIN music_enrichment_base_for_webtables_v1 b
      ON a.base_record_id = b.base_record_id
    WHERE a.base_record_status = 'mb_release_enriched_from_cds'
      AND b.base_record_status = 'external_residual'
""").fetchdf())

   overlap_n
0         24


In [138]:
print(con.execute("""
    SELECT
        base_record_status,
        COUNT(*) AS n,
        SUM(CASE WHEN release_id IS NULL THEN 1 ELSE 0 END) AS rows_with_null_release_id,
        SUM(CASE WHEN release_id IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_real_release_id
    FROM music_enrichment_base_for_webtables_v1
    GROUP BY base_record_status
    ORDER BY n DESC
""").fetchdf())

             base_record_status        n  rows_with_null_release_id  \
0       mb_release_not_enriched  5305678                        0.0   
1             external_residual     4647                     4647.0   
2  mb_release_enriched_from_cds     4200                        0.0   

   rows_with_real_release_id  
0                  5305678.0  
1                        0.0  
2                     4200.0  


In [139]:
print(con.execute("""
    SELECT 'mb_linked_in_master_v4' AS bucket, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE release_id IS NOT NULL

    UNION ALL

    SELECT 'external_residual_in_master_v4' AS bucket, COUNT(*) AS n
    FROM thesis_musicbrainz_enrichment_master_v4
    WHERE release_id IS NULL
""").fetchdf())

                           bucket     n
0          mb_linked_in_master_v4  4374
1  external_residual_in_master_v4  4647


In [140]:
print(con.execute("""
SELECT 
    COUNT(*) AS enriched_count,
FROM music_enrichment_base_for_webtables_v1
WHERE base_record_status = 'mb_release_enriched_from_cds';
""").fetchdf())

   enriched_count
0            4200


In [141]:
print(con.execute("""
    SELECT *
    FROM musicbrainz_release_enriched_from_cds_v1
    WHERE release_id = 1726393
""").fetchdf())

   release_id release_title barcode  artist_credit_id    artist_credit_full  \
0     1726393      Die Hard    None           1127198  The Wolves of Avalon   

       main_artist_name  mb_year  total_tracks  enrichment_cds_id  \
0  The Wolves of Avalon     2016           2.0               <NA>   

  source_artist source_title  source_year enriched_category enriched_genre  \
0          None         None         <NA>              None           None   

  external_tracklist enrichment_route enrichment_status enrichment_source  \
0               None             None              None              None   

   source_record_count  is_enriched_from_cds  
0                 <NA>                 False  


In [142]:
print(con.execute("""
    SELECT *
    FROM music_enrichment_base_for_webtables_v1
    WHERE release_id = 1726393
""").fetchdf())

   base_record_id  release_id release_title      main_artist_name  \
0         1726393     1726393      Die Hard  The Wolves of Avalon   

     artist_credit_full barcode  mb_year  total_tracks  external_record_id  \
0  The Wolves of Avalon    None     2016           2.0                <NA>   

  external_artist external_title  external_year external_category  \
0            None           None           <NA>              None   

  external_genre external_tracklist enrichment_route enrichment_status  \
0           None               None             None              None   

  enrichment_source  is_linked_to_musicbrainz       base_record_status  
0              None                     False  mb_release_not_enriched  


In [143]:
print(con.execute("""
    SELECT release_id, release_title, main_artist_name, mb_year, barcode, total_tracks
    FROM mb_release_features
    WHERE release_id IN (1726393, 726393)
""").fetchdf())

   release_id         release_title      main_artist_name  mb_year  \
0     1726393              Die Hard  The Wolves of Avalon     2016   
1      726393  Ghost in the Machine            The Police     2026   

        barcode  total_tracks  
0          None           2.0  
1  606949365523          12.0  


In [144]:
print(con.execute("""
    SELECT
        release_id,
        release_title,
        main_artist_name,
        barcode,
        mb_year,
        total_tracks,
        enriched_category,
        enriched_genre,
        is_enriched_from_cds
    FROM musicbrainz_release_enriched_from_cds_v1
    WHERE lower(release_title) = 'ghost in the machine'
      AND lower(main_artist_name) = 'the police'
    ORDER BY release_id
""").fetchdf())

    release_id         release_title main_artist_name         barcode  \
0         7907  Ghost in the Machine       The Police     07502137302   
1       397051  Ghost in the Machine       The Police    606949359829   
2       577748  Ghost in the Machine       The Police    075021373020   
3       577749  Ghost in the Machine       The Police    075021373020   
4       726393  Ghost in the Machine       The Police    606949365523   
5      1099303  Ghost in the Machine       The Police   4988005501851   
6      1156586  Ghost in the Machine       The Police                   
7      1268744  Ghost in the Machine       The Police    606949364625   
8      1271585  Ghost in the Machine       The Police    082839373029   
9      1282532  Ghost in the Machine       The Police                   
10     1726073  Ghost in the Machine       The Police   4988005329875   
11     1885651  Ghost in the Machine       The Police                   
12     2215274  Ghost in the Machine       The Poli

In [145]:
con.close()

In [5]:
import duckdb

# Connect to your database
con = duckdb.connect("../data_processed/thesis.duckdb")

# Show all tables
tables_df = con.execute("SHOW TABLES;").fetchdf()
print(tables_df)

con.close()

                                  name
0                   best_fuzzy_per_cds
1                                  cds
2           cds_artist_alias_bridge_v1
3                      cds_enriched_v6
4                      cds_enriched_v7
..                                 ...
98                            mb_norm3
99                          mb_release
100                mb_release_features
101              mb_title_only_cleaned
102  mb_various_title_integration_view

[103 rows x 1 columns]


In [3]:
import duckdb

# Connect to your thesis database
con = duckdb.connect("../data_processed/thesis.duckdb")

# Query DuckDB's internal system to show all tables and their estimated row counts
query = """
    SELECT 
        table_name, 
        estimated_size AS row_count
    FROM duckdb_tables()
    ORDER BY estimated_size DESC;
"""

# Fetch the results as a pandas DataFrame and print it
tables_df = con.execute(query).fetchdf()
print(tables_df)

# Always good practice to close the connection!
con.close()

                                  table_name  row_count
0                         cds_mb_fuzzy_pairs   11904344
1                    cds_mb_enrichment_pairs   11873845
2          cds_mb_enrichment_pairs_v6_round2    9898596
3               cds_mb_curated_main_pairs_v1    8796214
4                      mb_artist_credit_name    6642750
..                                       ...        ...
82             cds_generic_scored_candidates         14
83                 cds_generic_mb_candidates         14
84  cds_to_mb_best_enrichment_v6_round2_auto          5
85           cds_to_mb_generic_title_safe_v1          4
86                    cds_generic_best_match          3

[87 rows x 2 columns]
